Simulamos nulos

In [3]:
# =============================================================================
# 01_simular_nulos_corregido_v2.py
#
# Paso 1: Simulación de Datos Faltantes
# CORRECCIÓN: Convierte las fechas de UTC a la hora local de cada ciudad.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Creando DataFrame de ejemplo...");
    # ... (código de ejemplo) ...
    # (El código de ejemplo no se modificará, ya que es solo un fallback)
    df_air_quality_raw = pd.DataFrame() # Simplificado para el ejemplo

if not df_air_quality_raw.empty:
    # Asegurar que la columna 'Date' es de tipo datetime y está localizada en UTC
    df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)

    if 'CO2' in df_air_quality_raw.columns:
        df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])
        print("Columna 'CO2' descartada.")

    # --- 1. Conversión de Zona Horaria ---
    print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad...")

    city_timezones = {
        'Brasilia': 'America/Sao_Paulo', # GMT-3
        'Cairo': 'Africa/Cairo',         # GMT+2
        'Dubai': 'Asia/Dubai',           # GMT+4
        'London': 'Europe/London',       # GMT+0/GMT+1 (Pandas maneja DST, pero aquí será consistente)
        'New York': 'America/New_York',    # GMT-5 (EST)
        'Sydney': 'Australia/Sydney'     # GMT+10 (AEST)
    }

    # Reemplazar "New_York" por "New York" si es necesario
    df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

    df_parts = []
    for city, tz in city_timezones.items():
        df_city = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
        if not df_city.empty:
            df_city['Date'] = df_city['Date'].dt.tz_convert(tz)
            df_parts.append(df_city)
        else:
            print(f"  Advertencia: No se encontraron datos para la ciudad '{city}' en el mapeo de zonas horarias.")
    
    # Concatenar de nuevo y ordenar por ciudad y fecha
    df_air_quality = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)

    print("Conversión de zona horaria completada.")
    # Verificación visual
    print("\nEjemplo de fechas convertidas (Brasilia):")
    print(df_air_quality[df_air_quality['City'] == 'Brasilia'].head(2)[['Date', 'City']])
    print("\nEjemplo de fechas convertidas (Sydney):")
    print(df_air_quality[df_air_quality['City'] == 'Sydney'].head(2)[['Date', 'City']])
else:
    # Si usamos el fallback de datos de ejemplo
    df_air_quality = df_air_quality_raw


# --- Parámetros de Simulación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL
N_REPETITIONS = 5

NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados') # Nueva carpeta de salida
os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Simulación de NaNs (sin cambios)---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input); random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans); row_indices_to_nan_positions = []
    scenario_type, config = scenario_config_input['type'], scenario_config_input['config']
    if scenario_type == 'row_percentage':
        num_rows_to_nan = int(n_rows * config['percentage'])
        if num_rows_to_nan > 0 and n_rows > 0: row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length, percentage = config['length'], config.get('percentage', 0.20); total_rows_to_nan = int(n_rows * percentage)
        num_blocks = max(1, total_rows_to_nan // block_length) if block_length > 0 else 0
        used_indices, possible_starts = set(), list(range(n_rows - block_length + 1)); random.shuffle(possible_starts)
        blocks_created = 0
        for start_pos in possible_starts:
            if blocks_created >= num_blocks: break
            block_range = set(range(start_pos, start_pos + block_length))
            if not block_range.intersection(used_indices):
                row_indices_to_nan_positions.extend(list(block_range)); used_indices.update(block_range); blocks_created += 1
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan: df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
    return df_with_nans

# --- Script Principal de Simulación ---
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Saliendo."); exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)

    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty: print(f"  No hay datos para la ciudad {city_name_original}."); continue
    if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill.");
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_original_for_sim.columns: df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs en {city_name_original}. Omitiendo."); continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent")
        print(f"  Generando escenario: {scenario_name_safe}")
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)
        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time() * 1000 + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)) % (2**32 - 1)
            df_city_with_nans = introduce_nans_by_row(df_city_original_for_sim, scenario, random_seed_input=current_seed)
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
    print(f"  Simulación de escenarios para {city_name_original} completada.")

end_time_script = time.time()
print(f"\nProceso de simulación de NaNs completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Convirtiendo fechas de UTC a la hora local de cada ciudad...
Conversión de zona horaria completada.

Ejemplo de fechas convertidas (Brasilia):
                        Date      City
0  2023-12-31 21:00:00-03:00  Brasilia
1  2023-12-31 22:00:00-03:00  Brasilia

Ejemplo de fechas convertidas (Sydney):
                            Date    City
43920  2024-01-01 11:00:00+11:00  Sydney
43921  2024-01-01 12:00:00+11:00  Sydney

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Generando escenario: RandomRows_20percent
  Generando escenario: BlockRows_12h_20percent
  Generando escenario: BlockRows_24h_20percent
  Generando escenario: BlockRows_36h_20percent
  Generando escenario: BlockRows_48h_20percent
  Generando escenario: BlockRows_60h_20percent
  Simulación de escenarios para Brasilia completada.

Procesando Ciudad: Cairo (Directorio seguro: Cairo)
  Generando escenario: RandomRows_20percent
  Generando escena

Interpolación Lineal

In [4]:
# =============================================================================
# 02_imputacion_lineal.py
#
# Paso 2: Imputación con Interpolación Lineal y Evaluación
#
# Esta celda lee los archivos con NaNs generados, les aplica interpolación 
# lineal y evalúa su rendimiento.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob

# --- 0. Carga de Datos ORIGINALES (Ground Truth) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load)
    print(f"Datos 'Ground Truth' cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Es necesario para la evaluación.")
    exit()

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])

# --- Parámetros de Imputación y Evaluación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

# --- Lista de escenarios a procesar (DEBE COINCIDIR CON LA CELDA 1) ---
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Métricas ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            
            nrmae_range = np.nan
            nrmse_range = np.nan
            if original_col_range > 1e-9:
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range

            results[col]['MAE'] = mae; results[col]['RMSE'] = rmse
            results[col]['NRMAE_range'] = nrmae_range; results[col]['NRMSE_range'] = nrmse_range
        except Exception as e: pass
    return results

# --- Script Principal de Imputación y Evaluación ---
all_detailed_metrics_results = []
start_time_script = time.time()

for city_name_original in df_air_quality_ground_truth['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original}")
    
    df_city_gt = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == city_name_original].copy()
    if df_city_gt[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
            if col_fill in df_city_gt.columns: df_city_gt[col_fill] = df_city_gt[col_fill].ffill().bfill()
    if df_city_gt.isnull().values.any():
        print(f"  No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo ciudad.")
        continue
    df_city_original_ground_truth_reset = df_city_gt.reset_index(drop=True)

    sim_data_city_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    if not os.path.exists(sim_data_city_dir):
        print(f"  Directorio de datos simulados no encontrado para {city_name_original}. Omitiendo.")
        continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Evaluando escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(sim_data_city_dir, scenario_name_safe)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"    Directorio de escenario no encontrado: {sim_data_scenario_dir}. Omitiendo.")
            continue
            
        imputed_csv_scenario_dir = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe, scenario_name_safe)
        os.makedirs(imputed_csv_scenario_dir, exist_ok=True)

        repetition_files = glob.glob(os.path.join(sim_data_scenario_dir, "*_with_nans.csv"))

        for sim_file_path in repetition_files:
            file_name = os.path.basename(sim_file_path)
            try:
                df_city_with_nans_reset = pd.read_csv(sim_file_path)
                rep_str = file_name.split('_rep')[-1].split('_')[0]
                seed_str = file_name.split('_seed')[-1].split('_')[0]
                rep = int(rep_str) if rep_str.isdigit() else 0
                current_seed = int(seed_str) if seed_str.isdigit() else 0
            except Exception as e:
                print(f"      Error cargando {file_name}: {e}")
                continue
            
            # 1. Realizar imputación
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None)
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()

            # 2. Guardar archivo imputado
            imputed_filename = file_name.replace('_with_nans.csv', f'_imputed_{INTERPOLATION_METHOD_NAME}.csv')
            df_city_imputed_linear_reset.to_csv(os.path.join(imputed_csv_scenario_dir, imputed_filename), index=False)

            # 3. Calcular métricas
            nan_mask = df_city_with_nans_reset[ALL_TARGET_COLS].isnull() & ~df_city_original_ground_truth_reset[ALL_TARGET_COLS].isnull()
            metrics_scores = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset, df_city_imputed_linear_reset, nan_mask)

            # 4. Almacenar resultados detallados
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 
                    'MAE': metrics_scores[var_name]['MAE'], 
                    'RMSE': metrics_scores[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores[var_name]['NRMAE_range'],
                    'NRMSE_range': metrics_scores[var_name]['NRMSE_range']
                })
    print(f"  Procesamiento de escenarios para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original in grouped_by_city_var['City'].unique():
        city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_original]
        variables_summary_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_original} guardado en: {variables_summary_path}")
        if city_name_original == df_air_quality_ground_truth['City'].unique()[0]: print(city_metrics_df)

    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
    print(f"  Resumen combinado de contaminantes guardado en: {contaminants_summary_path}")
    print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False)
    print(f"  Resumen combinado de AQI guardado en: {aqi_summary_path}")
    print(aqi_combined_summary)
else:
    print("No se generaron resultados de métricas detalladas.")

end_time_script = time.time()
print(f"\nProceso de imputación y evaluación completado.")
print(f"Archivos de datos simulados leídos desde: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Archivos de métricas resumidas guardados en: {METRICS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos 'Ground Truth' cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad: Brasilia
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Brasilia completado.

Procesando Ciudad: Cairo
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Cairo completado.

Procesando Ciudad: Dubai
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluan

SARIMA ORDERS

In [ ]:
# CELDA 1: BÚSQUEDA DE ÓRDENES ÓPTIMOS ARIMA/SARIMA (EJECUTAR SOLO UNA VEZ O CUANDO SEA NECESARIO)
import pandas as pd
import numpy as np
import os
import pmdarima as pm
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- 0. Carga de Datos (Asegúrate de que df_air_quality esté cargado) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality = pd.read_csv(file_path_load)
    df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
    if 'CO2' in df_air_quality.columns: df_air_quality = df_air_quality.drop(columns=['CO2'])
    print(f"Datos cargados desde: {file_path_load} para Celda 1")
except FileNotFoundError:
    print("Error: df_air_quality no encontrado en Celda 1.")
    _example_data = [] # Datos de ejemplo mínimos
    _cities_example = ['Brasilia', 'Cairo']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(24*7*4): # 4 semanas de datos horarios
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 40+np.random.rand()*20 ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'AQI'])
    print("DataFrame de ejemplo creado para Celda 1.")


# --- Parámetros ---
COLUMNS_FOR_ARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
SEASONAL_PERIOD = 24

PROJECT_ROOT_DIR_ARIMA_SEARCH = os.getcwd()
# Directorio donde se guardarán los JSONs directamente
ORDERS_OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR_ARIMA_SEARCH, 'imputaciones', 'SARIMA_auto', 'model_orders_v2')
os.makedirs(ORDERS_OUTPUT_DIR, exist_ok=True) # Solo se crea la carpeta base 'model_orders'

print(f"--- INICIO DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")
print(f"Los órdenes se guardarán directamente en: {ORDERS_OUTPUT_DIR}")

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está definido o está vacío en Celda 1.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando ciudad: {city_name_original}")

    df_city = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city.empty:
        print(f"  No hay datos para {city_name_original}, omitiendo.")
        continue

    for col_name in COLUMNS_FOR_ARIMA:
        if col_name not in df_city.columns:
            print(f"  Variable {col_name} no encontrada en los datos de {city_name_original}, omitiendo.")
            continue

        print(f"  Buscando orden para variable: {col_name}...")
        series_data = df_city[col_name].dropna()

        if len(series_data) < 2 * SEASONAL_PERIOD:
            print(f"    Datos insuficientes para SARIMA ({len(series_data)} puntos). Intentando ARIMA no estacional.")
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, d=None, seasonal=False, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)
                order = auto_model.order; seasonal_order = (0,0,0,0)
            except Exception as e:
                print(f"    Error en auto_arima no estacional para {col_name}: {e}. Usando (1,0,0).")
                order = (1,0,0); seasonal_order = (0,0,0,0)
        elif len(series_data) < 10:
             print(f"    Datos muy insuficientes (<10 puntos). Usando (1,0,0).")
             order = (1,0,0); seasonal_order = (0,0,0,0)
        else:
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, m=SEASONAL_PERIOD, start_P=0, D=None, max_P=2, max_Q=2, seasonal=True, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)     
                order = auto_model.order; seasonal_order = auto_model.seasonal_order
                print(f"    Orden encontrado para {col_name}: {order}, Estacional: {seasonal_order}")
            except Exception as e:
                print(f"    Error en auto_arima para {col_name}: {e}. Usando (1,0,1)(1,0,1,{SEASONAL_PERIOD}).")
                order = (1,0,1); seasonal_order = (1,0,1,SEASONAL_PERIOD) if SEASONAL_PERIOD > 0 else (0,0,0,0)
        
        order_data = {'order': list(order), 'seasonal_order': list(seasonal_order)}
        # Guardar directamente en ORDERS_OUTPUT_DIR, sin subcarpeta de ciudad para el JSON
        file_path_order = os.path.join(ORDERS_OUTPUT_DIR, f'orders_{city_name_safe}_{col_name}.json')
        with open(file_path_order, 'w') as f:
            json.dump(order_data, f, indent=4)
        print(f"    Órdenes guardados en: {file_path_order}")

print(f"--- FIN DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
--- Fase 1: Encontrando/Cargando Órdenes SARIMA Óptimos ---
  Procesando ciudad para encontrar órdenes: Brasilia
    Procesando variable para auto_arima: CO
      Ejecutando auto_arima para Brasilia-CO (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-CO: Order=(3, 0, 0), Seasonal=(1, 0, 1, 24)
    Procesando variable para auto_arima: NO2
      Ejecutando auto_arima para Brasilia-NO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-NO2: Order=(1, 1, 0), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: SO2
      Ejecutando auto_arima para Brasilia-SO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-SO2: Order=(3, 1, 1), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: O3


KeyboardInterrupt: 

Imputación SARIMA

In [5]:
# =============================================================================
# 02_imputacion_sarima.py (Versión con Target City Opcional)
# =============================================================================

import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Parámetros de Imputación y Evaluación ---
# CORRECCIÓN: Añadir parámetro de ciudad objetivo
TARGET_CITY_SARIMA = "Brasilia"  # Poner a None para ejecutar para TODAS las ciudades

COLUMNS_TO_IMPUTE_SARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_REPETITIONS_SARIMA = 1

NAN_SIMULATION_SCENARIOS_SARIMA = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

# --- Rutas de Entrada y Salida ---
PROJECT_ROOT_DIR_SARIMA_IMPUTE = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'CSVs', 'simulados')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', 'SARIMA', 'model_orders')
SARIMA_METHOD_NAME = 'SARIMA'
IMPUTATIONS_SARIMA_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', SARIMA_METHOD_NAME)
METRICS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'metricas')
IMPUTED_CSVS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'CSVs')
os.makedirs(METRICS_SARIMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, exist_ok=True)

# --- Función de Imputación SARIMA (sin cambios) ---
def impute_sarima_kalman_smoother(series_with_nans, order_tuple, seasonal_order_tuple, original_series_for_bounds=None):
    imputed_series = series_with_nans.copy()
    original_nan_mask = imputed_series.isnull()
    if not original_nan_mask.any(): return series_with_nans
    min_len_heuristic = max(sum(order_tuple), sum(seasonal_order_tuple[:3])) + seasonal_order_tuple[3] if seasonal_order_tuple[3] > 0 else max(sum(order_tuple), sum(seasonal_order_tuple[:3])); min_len_heuristic = max(min_len_heuristic, 10)
    if len(imputed_series.dropna()) < min_len_heuristic + 5: return series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    sarima_failed_or_unstable = False
    try:
        model = SARIMAX(imputed_series, order=order_tuple, seasonal_order=seasonal_order_tuple, enforce_stationarity=False, enforce_invertibility=False, initialization='approximate_diffuse')
        results = model.fit(disp=False, maxiter=200)
        predictions_all = results.predict(start=imputed_series.index[0], end=imputed_series.index[-1])
        imputed_series.loc[original_nan_mask] = predictions_all[original_nan_mask]
        if imputed_series.isnull().any():
            temp_linear_fill = series_with_nans.interpolate(method='linear', limit_direction='both'); imputed_series = imputed_series.fillna(temp_linear_fill).ffill().bfill()
    except Exception as e: sarima_failed_or_unstable = True
    if sarima_failed_or_unstable: imputed_series = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    if original_series_for_bounds is not None and original_nan_mask.any():
        clean_original_data = original_series_for_bounds.dropna()
        if not clean_original_data.empty:
            q01, q99 = clean_original_data.quantile(0.001), clean_original_data.quantile(0.999); iqr_val = clean_original_data.quantile(0.75) - clean_original_data.quantile(0.25)
            padding = max(1.0, 0.1 * abs(q99 - q01)) if (q99 - q01) != 0 else 1.0
            lower_bound, upper_bound = q01 - 3 * iqr_val - padding, q99 + 3 * iqr_val + padding
            extreme_values_mask = original_nan_mask & ((imputed_series < lower_bound) | (imputed_series > upper_bound))
            if extreme_values_mask.any():
                temp_linear_imputation = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill(); imputed_series.loc[extreme_values_mask] = temp_linear_imputation[extreme_values_mask]
    return imputed_series

# --- Función para calcular métricas (sin cambios) ---
def calculate_metrics_sarima_full(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in COLUMNS_TO_IMPUTE_SARIMA:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            nrmae_range = np.nan if original_col_range < 1e-9 else mae / original_col_range; nrmse_range = np.nan if original_col_range < 1e-9 else rmse / original_col_range
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal de Imputación SARIMA ---
all_detailed_metrics_sarima_results = []
start_time_script_sarima = time.time()
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent = os.path.dirname(os.getcwd()); alt_file_path_load_orig = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        file_path_load_orig = alt_file_path_load_orig if os.path.exists(alt_file_path_load_orig) else ''
    if not file_path_load_orig: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_original_gt = pd.read_csv(file_path_load_orig); df_air_quality_original_gt['Date'] = pd.to_datetime(df_air_quality_original_gt['Date'])
    if 'CO2' in df_air_quality_original_gt.columns: df_air_quality_original_gt = df_air_quality_original_gt.drop(columns=['CO2'])
    print(f"Datos originales GT cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error crítico: Archivo 'Air_Quality.csv' de ground truth no encontrado."); exit()

# CORRECCIÓN: Lógica para seleccionar la(s) ciudad(es) a procesar
if TARGET_CITY_SARIMA:
    cities_to_process = [TARGET_CITY_SARIMA]
    if TARGET_CITY_SARIMA not in df_air_quality_original_gt['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_SARIMA}' no se encuentra en el dataset. No se procesará nada.")
        cities_to_process = []
else:
    cities_to_process = df_air_quality_original_gt['City'].unique()

for city_name_original_sarima in cities_to_process:
    city_name_safe_sarima = city_name_original_sarima.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación SARIMA: {city_name_original_sarima}")

    city_sim_data_input_dir_root = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR, city_name_safe_sarima)
    city_imputed_csv_sarima_dir_root = os.path.join(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, city_name_safe_sarima)
    os.makedirs(city_imputed_csv_sarima_dir_root, exist_ok=True)

    df_city_gt_for_metrics = df_air_quality_original_gt[df_air_quality_original_gt['City'] == city_name_original_sarima].copy()
    if df_city_gt_for_metrics.empty: print(f"No hay datos para {city_name_original_sarima}. Omitiendo."); continue
    if df_city_gt_for_metrics[COLUMNS_TO_IMPUTE_SARIMA].isnull().any().any():
        for col_fill_gt in COLUMNS_TO_IMPUTE_SARIMA:
            if col_fill_gt in df_city_gt_for_metrics.columns: df_city_gt_for_metrics[col_fill_gt] = df_city_gt_for_metrics[col_fill_gt].ffill().bfill()
    df_city_gt_for_metrics_reset = df_city_gt_for_metrics.reset_index(drop=True)

    for scenario_sarima in NAN_SIMULATION_SCENARIOS_SARIMA:
        scenario_name_safe_sarima = scenario_sarima['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Escenario: {scenario_name_safe_sarima}")
        scenario_city_sim_data_input_dir = os.path.join(city_sim_data_input_dir_root, scenario_name_safe_sarima)
        scenario_city_imputed_csv_sarima_dir = os.path.join(city_imputed_csv_sarima_dir_root, scenario_name_safe_sarima); os.makedirs(scenario_city_imputed_csv_sarima_dir, exist_ok=True)

        for rep_s in range(1, N_REPETITIONS_SARIMA + 1):
            sim_files_pattern = os.path.join(scenario_city_sim_data_input_dir, f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_*_with_nans.csv")
            sim_files_in_dir = sorted(glob.glob(sim_files_pattern))
            if not sim_files_in_dir: continue
            
            sim_file_path_to_load = sim_files_in_dir[0]; current_seed_loaded = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]
            try: df_city_with_nans_loaded = pd.read_csv(sim_file_path_to_load)
            except FileNotFoundError: print(f"    Error: Archivo simulado no encontrado: {sim_file_path_to_load}"); continue
            
            df_city_imputed_sarima = df_city_with_nans_loaded.copy()
            mask_for_metrics_s = pd.DataFrame(False, index=df_city_gt_for_metrics_reset.index, columns=COLUMNS_TO_IMPUTE_SARIMA)
            for col_mask in COLUMNS_TO_IMPUTE_SARIMA:
                 if col_mask in df_city_with_nans_loaded.columns and col_mask in df_city_gt_for_metrics_reset.columns:
                    mask_for_metrics_s[col_mask] = df_city_with_nans_loaded[col_mask].isnull() & ~df_city_gt_for_metrics_reset[col_mask].isnull()

            for col_impute_s in COLUMNS_TO_IMPUTE_SARIMA:
                print(f"    Imputando columna: {col_impute_s}...")
                order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe_sarima}_{col_impute_s}.json')
                try:
                    with open(order_file_path, 'r') as f: orders = json.load(f)
                    order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
                except (FileNotFoundError, json.JSONDecodeError):
                    print(f"      Archivo de órdenes no encontrado/corrupto para {col_impute_s}. Usando (1,0,1)(0,0,0,0)."); order_tuple, seasonal_order_tuple = (1,0,1), (0,0,0,0)
                series_to_impute, original_clean_series_for_bounds = df_city_with_nans_loaded[col_impute_s].copy(), df_city_gt_for_metrics_reset[col_impute_s].copy()
                imputed_values_col = impute_sarima_kalman_smoother(series_to_impute, order_tuple, seasonal_order_tuple, original_clean_series_for_bounds)
                df_city_imputed_sarima[col_impute_s] = imputed_values_col
            
            imputed_sarima_filename = f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_seed{current_seed_loaded}_imputed_sarima.csv"
            df_city_imputed_sarima.to_csv(os.path.join(scenario_city_imputed_csv_sarima_dir, imputed_sarima_filename), index=False)
            metrics_scores = calculate_metrics_sarima_full(df_city_gt_for_metrics_reset, df_city_imputed_sarima, mask_for_metrics_s)
            
            for var_name in COLUMNS_TO_IMPUTE_SARIMA:
                all_detailed_metrics_sarima_results.append({'City': city_name_original_sarima, 'Scenario': scenario_name_safe_sarima, 'Repetition': rep_s, 'Seed': current_seed_loaded, 'Imputation_Method': SARIMA_METHOD_NAME, 'Variable': var_name, 'MAE': metrics_scores[var_name]['MAE'], 'RMSE': metrics_scores[var_name]['RMSE'], 'NRMAE_range': metrics_scores[var_name]['NRMAE_range'], 'NRMSE_range': metrics_scores[var_name]['NRMSE_range']})

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_sarima_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_sarima_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name in grouped_by_city_var['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name]
        variables_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False); print(f"  Resumen de variables para {city_name} guardado en: {variables_summary_path}")

    print("\nGenerando resumen combinado de contaminantes...")
    # CORRECCIÓN: Filtrar correctamente las columnas de contaminantes
    contaminant_cols = [c for c in COLUMNS_TO_IMPUTE_SARIMA if c != 'AQI']
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(contaminant_cols)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False); print(f"  Resumen contaminantes guardado: {contaminants_summary_path}"); print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False); print(f"  Resumen AQI guardado: {aqi_summary_path}"); print(aqi_combined_summary)

    global_detailed_s_file = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'ALL_metrics_detailed_{SARIMA_METHOD_NAME}.csv')
    df_all_detailed_metrics.to_csv(global_detailed_s_file, index=False); print(f"\nMétricas detalladas globales de SARIMA guardadas en: {global_detailed_s_file}")
else:
    print("\nNo se generaron resultados de métricas detalladas.")

end_time_script_sarima = time.time()
print(f"\nProceso SARIMA completado. CSVs: {IMPUTED_CSVS_SARIMA_OUTPUT_DIR}, Métricas: {METRICS_SARIMA_OUTPUT_DIR}")
print(f"Tiempo SARIMA: {(end_time_script_sarima - start_time_script_sarima) / 60:.2f} min.")

Datos originales GT cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación SARIMA: Brasilia
  Escenario: RandomRows_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_12h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_24h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_36h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O

RANDOM FOREST

In [6]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_orig_load = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_orig_load = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_orig_load['Date'] = pd.to_datetime(df_air_quality_orig_load['Date'])
if 'CO2' in df_air_quality_orig_load.columns:
    df_air_quality_orig_load = df_air_quality_orig_load.drop(columns=['CO2'])

# --- Parámetros ---
TARGET_CITY_RF = "Brasilia" # Poner a None para procesar todas las ciudades

CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

df_air_quality_orig_load['Hour'] = df_air_quality_orig_load['Date'].dt.hour
df_air_quality_orig_load['DayOfWeek'] = df_air_quality_orig_load['Date'].dt.dayofweek
df_air_quality_orig_load['Month'] = df_air_quality_orig_load['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

N_LAGS = [1, 2, 24, 48]
LAG_FEATURES = []
df_with_features = df_air_quality_orig_load.copy()
for col_to_lag in ALL_TARGET_COLS:
    for lag_val in N_LAGS:
        lag_col_name = f"{col_to_lag}_lag{lag_val}"
        df_with_features[lag_col_name] = df_with_features.groupby('City')[col_to_lag].shift(lag_val)
        LAG_FEATURES.append(lag_col_name)

COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM = ALL_TARGET_COLS + TIME_FEATURES + LAG_FEATURES

N_REPETITIONS = 1 # Ajustar según sea necesario
# CORRECCIÓN: Usar la nueva lista de escenarios
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

RF_IMPUTER_METHOD_NAME = 'RandomForestImputer'
IMPUTATIONS_RF_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', RF_IMPUTER_METHOD_NAME)
METRICS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'metricas')
IMPUTED_CSVS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'CSVs')

os.makedirs(METRICS_RF_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_RF_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares (sin cambios) ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range, nrmse_range = np.nan, np.nan
            if original_col_range > 1e-9: nrmae_range, nrmse_range = mae / original_col_range, rmse / original_col_range
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_rf_results = []
script_start_time_rf = time.time()

if 'df_with_features' not in locals() or df_with_features.empty:
    print("Error: df_with_features no está definido. Saliendo."); exit()

# CORRECCIÓN: Lógica para seleccionar ciudades a procesar
if TARGET_CITY_RF:
    cities_to_process = [TARGET_CITY_RF]
    if TARGET_CITY_RF not in df_with_features['City'].unique():
        print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_RF}' no se encuentra en el dataset. No se procesará nada.")
        cities_to_process = []
else:
    cities_to_process = df_with_features['City'].unique()

for city_name_original in cities_to_process:
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación RF: {city_name_original}")

    df_city_original_features = df_with_features[df_with_features['City'] == city_name_original].copy()
    if df_city_original_features.empty: continue

    df_city_gt_eval = df_air_quality_orig_load[df_air_quality_orig_load['City'] == city_name_original].copy()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_gt_eval.columns: df_city_gt_eval[col_fill] = df_city_gt_eval[col_fill].ffill().bfill()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Error: No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo."); continue
    df_city_original_ground_truth_reset = df_city_gt_eval.reset_index(drop=True)

    city_imputed_csv_rf_dir_root = os.path.join(IMPUTED_CSVS_RF_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_rf_dir_root, exist_ok=True)



    rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt')
    iterative_imputer_rf = IterativeImputer(estimator=rf_estimator, max_iter=15, tol=1e-3, random_state=42, initial_strategy='mean', verbose=1)
    scaler = StandardScaler()
    
    
    
    data_for_fitting_imputer = df_city_original_features[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
    print(f"  Ajustando Scaler e IterativeImputer para {city_name_original}...")
    start_fit_time = time.time()
    try:
        scaled_data_for_fitting = scaler.fit_transform(data_for_fitting_imputer)
        iterative_imputer_rf.fit(scaled_data_for_fitting)
        print(f"  Scaler e Imputer ajustados en {time.time() - start_fit_time:.2f} segundos.")
    except Exception as e_fit:
        print(f"  Error ajustando Scaler/Imputer: {e_fit}. Omitiendo ciudad."); continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"    Escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario_name_safe)
        imputed_rf_scenario_dir = os.path.join(city_imputed_csv_rf_dir_root, scenario_name_safe); os.makedirs(imputed_rf_scenario_dir, exist_ok=True)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"      Directorio de datos simulados no encontrado: {sim_data_scenario_dir}. Omitiendo."); continue

        for rep in range(1, N_REPETITIONS + 1):
            matching_files = sorted(glob.glob(os.path.join(sim_data_scenario_dir, f"{city_name_safe}_{scenario_name_safe}_rep{rep}_*_with_nans.csv")))
            if not matching_files: continue
            sim_file_path_to_load = matching_files[0]
            current_seed_loaded = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]

            try:
                df_city_with_nans_loaded_raw = pd.read_csv(sim_file_path_to_load)
                df_for_imputation_base = df_city_original_features.copy().reset_index(drop=True)
                df_city_with_nans_loaded_reset = df_city_with_nans_loaded_raw.reset_index(drop=True)
                for col_target in ALL_TARGET_COLS:
                    if col_target in df_city_with_nans_loaded_reset.columns:
                        df_for_imputation_base[col_target] = df_city_with_nans_loaded_reset[col_target]
            except Exception as e_load:
                print(f"      Error cargando/procesando {sim_file_path_to_load}: {e_load}."); continue
            
            temp_introduced_nan_mask_df = pd.DataFrame(False, index=df_city_original_ground_truth_reset.index, columns=ALL_TARGET_COLS)
            for col_mask_check in ALL_TARGET_COLS:
                 if col_mask_check in df_city_with_nans_loaded_reset.columns and col_mask_check in df_city_original_ground_truth_reset.columns:
                    temp_introduced_nan_mask_df[col_mask_check] = df_city_with_nans_loaded_reset[col_mask_check].isnull() & ~df_city_original_ground_truth_reset[col_mask_check].isnull()

            df_to_impute_for_rf = df_for_imputation_base[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
            
            try:
                scaled_data_to_impute = scaler.transform(df_to_impute_for_rf)
                imputed_scaled_data_rf_array = iterative_imputer_rf.transform(scaled_data_to_impute)
                imputed_data_rf_array = scaler.inverse_transform(imputed_scaled_data_rf_array)
                df_imputed_features = pd.DataFrame(imputed_data_rf_array, columns=COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM)
                df_final_imputed_rf = df_for_imputation_base[['Date', 'City']].copy()
                for col_final in ALL_TARGET_COLS + TIME_FEATURES:
                    if col_final in df_imputed_features.columns: df_final_imputed_rf[col_final] = df_imputed_features[col_final]
            except Exception as e_impute:
                print(f"      Error durante la imputación RF para rep {rep}: {e_impute}."); continue

            imputed_filename_rf = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed_loaded}_imputed_rf.csv"
            df_final_imputed_rf.to_csv(os.path.join(imputed_rf_scenario_dir, imputed_filename_rf), index=False)

            metrics_scores_rf = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset[ALL_TARGET_COLS], df_final_imputed_rf[ALL_TARGET_COLS], temp_introduced_nan_mask_df)
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_rf_results.append({'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep, 'Seed': current_seed_loaded, 'Imputation_Method': RF_IMPUTER_METHOD_NAME, 'Variable': var_name, 'MAE': metrics_scores_rf[var_name]['MAE'], 'RMSE': metrics_scores_rf[var_name]['RMSE'], 'NRMAE_range': metrics_scores_rf[var_name]['NRMAE_range'], 'NRMSE_range': metrics_scores_rf[var_name]['NRMSE_range']})
    print(f"  Procesamiento de escenarios RF para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_rf_results:
    df_all_detailed_metrics_rf = pd.DataFrame(all_detailed_metrics_rf_results)
    print("\nGenerando resúmenes de métricas RF por variable y ciudad...")
    grouped_by_city_var_rf = df_all_detailed_metrics_rf.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_rf.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original_summary in grouped_by_city_var_rf['City'].unique():
        city_name_safe_rf_summary = city_name_original_summary.replace(" ", "_").replace("-", "_"); city_metrics_df_rf = grouped_by_city_var_rf[grouped_by_city_var_rf['City'] == city_name_original_summary]
        variables_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_rf_summary}.csv')
        city_metrics_df_rf.to_csv(variables_summary_path_rf, index=False); print(f"  Resumen RF de variables para {city_name_original_summary} guardado en: {variables_summary_path_rf}")
        if city_name_original_summary == cities_to_process[0]: print(city_metrics_df_rf)

    print("\nGenerando resumen RF combinado de contaminantes...")
    df_contaminants_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary_rf = df_contaminants_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary_rf.to_csv(contaminants_summary_path_rf, index=False); print(f"  Resumen RF combinado de contaminantes guardado en: {contaminants_summary_path_rf}"); print(contaminants_combined_summary_rf)
    
    print("\nGenerando resumen RF combinado de AQI...")
    df_aqi_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'] == 'AQI']
    aqi_combined_summary_rf = df_aqi_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary_rf.to_csv(aqi_summary_path_rf, index=False); print(f"  Resumen RF combinado de AQI guardado en: {aqi_summary_path_rf}"); print(aqi_combined_summary_rf)
else:
    print("No se generaron resultados de métricas detalladas para Random Forest Imputer.")

script_end_time_rf = time.time()
print(f"\nProceso de Imputación con Random Forest completado. Tiempo total: {(script_end_time_rf - script_start_time_rf) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación RF: Brasilia
  Ajustando Scaler e IterativeImputer para Brasilia...
[IterativeImputer] Completing matrix with shape (8784, 38)
[IterativeImputer] Change: 6.442078832853756, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 3.7969897000559465, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.484989515146827, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 3.0486658070600754, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.9925322609138094, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.4537382930967846, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 2.349269439012307, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 1.6335229209468896, scaled tolerance: 0.008004094305323934 
[IterativeImputer] Change: 1.2187169412200536, scaled tolerance: 0.008004094305323934 
[IterativeImput

LSTM-VAE

TARGET_CITY_VAE = "Brasilia"
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS_VAE = CONTAMINANT_COLS + AQI_COL
N_LAGS_VAE = 60
LATENT_DIM_VAE = 16
ENCODER_NEURONS_L1_VAE = 64
DECODER_NEURONS_L1_VAE = 64
VAE_INITIAL_LR = 5e-4; KL_LOSS_WEIGHT = 0.00005
VAE_BATCH_SIZE = 32
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO = 0.1
WFV_VALIDATION_POINTS_MIN_RATIO = 0.1
WFV_STEP_POINTS_MIN_RATIO = 0.1
WFV_EPOCHS_PER_FOLD = 30
WFV_ES_PATIENCE = 5
WFV_LR_PATIENCE = 3
MAX_WFV_FOLDS = 3
N_REPETITIONS_FOR_EVAL = 1
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]



In [1]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
import keras # Keras 3 de alto nivel
from keras import layers, Model, optimizers
from keras import backend as K
import time
import glob
import warnings
import shutil
import re
from unicodedata import normalize
import joblib
import random

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing TF Names ---
def sanitize_tf_name(name_str):
    s = normalize('NFKD', str(name_str)); s = s.encode('ascii', 'ignore'); s = s.decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s)
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    s = re.sub(r'_+', '_', s)
    if s.startswith('_') and len(s) > 1: s = s[1:]
    if s.endswith('_') and len(s) > 1: s = s[:-1]
    if not s: s = "sanitized_default_name_v2"
    return s

# --- 0. Carga de Datos Originales ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos originales cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(550):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)
if 'CO2' in df_air_quality_raw.columns:
    df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])
df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

# --- 1. Conversión de Zona Horaria y Extracción de Características de Tiempo ---
print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...")
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_parts = []
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

for city, tz in city_timezones.items():
    df_city_part = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
    if not df_city_part.empty:
        df_city_part['Date'] = df_city_part['Date'].dt.tz_convert(tz)
        df_city_part['Hour'] = df_city_part['Date'].dt.hour
        df_city_part['DayOfWeek'] = df_city_part['Date'].dt.dayofweek
        df_city_part['Month'] = df_city_part['Date'].dt.month
        df_parts.append(df_city_part)

if not df_parts:
    print("Advertencia: No se procesaron datos para ninguna ciudad.")
    expected_cols = list(df_air_quality_raw.columns) + TIME_FEATURES
    df_air_quality_full_original = pd.DataFrame(columns=expected_cols)
else:
    df_air_quality_full_original = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)
print("Conversión de zona horaria y extracción de características de tiempo completada.")

# --- Parámetros ---
TARGET_CITY_VAE = "Brasilia"
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS_VAE = CONTAMINANT_COLS + AQI_COL
N_LAGS_VAE = 60
LATENT_DIM_VAE = 16
ENCODER_NEURONS_L1_VAE = 64
DECODER_NEURONS_L1_VAE = 64
VAE_INITIAL_LR = 5e-4; KL_LOSS_WEIGHT = 0.00005
VAE_BATCH_SIZE = 32
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO = 0.1
WFV_VALIDATION_POINTS_MIN_RATIO = 0.1
WFV_STEP_POINTS_MIN_RATIO = 0.1
WFV_EPOCHS_PER_FOLD = 30
WFV_ES_PATIENCE = 5
WFV_LR_PATIENCE = 3
MAX_WFV_FOLDS = 3
N_REPETITIONS_FOR_EVAL = 1
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
LSTM_VAE_METHOD_NAME = 'LSTM_VAE_imputer_v6'
IMPUTATIONS_LSTM_VAE_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', LSTM_VAE_METHOD_NAME)
METRICS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'metricas')
IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'CSVs_unified')
MODELS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'model_weights')
os.makedirs(METRICS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)

# --- Funciones VAE y Auxiliares ---
def sampling(args):
    z_mean, z_log_var = args; batch = tf.shape(z_mean)[0]; dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim), mean=0., stddev=1.0)
    return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon

class VAE(keras.Model):
    def __init__(self, encoder, decoder, kl_loss_weight_param, name="vae_model", **kwargs):
        super().__init__(name=name, **kwargs)
        self.encoder_model = encoder; self.decoder_model = decoder
        self.kl_loss_weight = kl_loss_weight_param
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.mse_loss_fn = keras.losses.MeanSquaredError(reduction="sum_over_batch_size")
    @property
    def metrics(self): return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]
    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder_model(inputs, training=training)
        return self.decoder_model(z, training=training)
    def train_step(self, data):
        x_input = data[0] if isinstance(data, tuple) else data; target_for_loss = x_input[:, :, 0:1]
        with tf.GradientTape() as tape:
            z_mean_val, z_log_var_val, z_val = self.encoder_model(x_input, training=True)
            reconstruction = self.decoder_model(z_val, training=True)
            reconstruction_loss_val = self.mse_loss_fn(target_for_loss, reconstruction)
            kl_loss_val = -0.5 * keras.ops.mean(keras.ops.sum(1 + z_log_var_val - keras.ops.square(z_mean_val) - keras.ops.exp(z_log_var_val), axis=1))
            total_loss_val = reconstruction_loss_val + self.kl_loss_weight * kl_loss_val
        grads = tape.gradient(total_loss_val, self.trainable_weights); self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss_val); self.reconstruction_loss_tracker.update_state(reconstruction_loss_val); self.kl_loss_tracker.update_state(kl_loss_val)
        return {m.name: m.result() for m in self.metrics}
    def test_step(self, data):
        x_input = data[0] if isinstance(data, tuple) else data; target_for_loss = x_input[:, :, 0:1]
        z_mean_val, z_log_var_val, z_val = self.encoder_model(x_input, training=False)
        reconstruction = self.decoder_model(z_val, training=False)
        reconstruction_loss_val = self.mse_loss_fn(target_for_loss, reconstruction)
        kl_loss_val = -0.5 * keras.ops.mean(keras.ops.sum(1 + z_log_var_val - keras.ops.square(z_mean_val) - keras.ops.exp(z_log_var_val), axis=1))
        total_loss_val = reconstruction_loss_val + self.kl_loss_weight * kl_loss_val
        self.total_loss_tracker.update_state(total_loss_val); self.reconstruction_loss_tracker.update_state(reconstruction_loss_val); self.kl_loss_tracker.update_state(kl_loss_val)
        return {m.name: m.result() for m in self.metrics}

def create_vae_model_architecture(n_lags, n_features_input, latent_dim, enc_neurons, dec_neurons, var_name_for_layer_names="var"):
    safe_var_prefix = sanitize_tf_name(var_name_for_layer_names)
    enc_in = layers.Input(shape=(n_lags, n_features_input), name=f"{safe_var_prefix}_encoder_input"); enc_lstm = layers.LSTM(enc_neurons, activation='tanh', name=f"{safe_var_prefix}_encoder_lstm")(enc_in)
    z_m = layers.Dense(latent_dim, name=f"{safe_var_prefix}_z_mean")(enc_lstm); z_lv = layers.Dense(latent_dim, name=f"{safe_var_prefix}_z_log_var")(enc_lstm)
    z = layers.Lambda(sampling, name=f"{safe_var_prefix}_z_sampling_layer")([z_m, z_lv]); encoder = Model(enc_in, [z_m, z_lv, z], name=f"{safe_var_prefix}_encoder")
    dec_in = layers.Input(shape=(latent_dim,), name=f"{safe_var_prefix}_decoder_latent_input"); dec_rep = layers.RepeatVector(n_lags, name=f"{safe_var_prefix}_decoder_repeat_vector")(dec_in)
    dec_lstm = layers.LSTM(dec_neurons, activation='tanh', return_sequences=True, name=f"{safe_var_prefix}_decoder_lstm")(dec_rep)
    dec_out = layers.TimeDistributed(layers.Dense(1), name=f"{safe_var_prefix}_decoder_reconstruction_output")(dec_lstm); decoder = Model(dec_in, dec_out, name=f"{safe_var_prefix}_decoder")
    return encoder, decoder

def create_sequences_for_vae_training(target_series_1d_scaled, exog_df_scaled_nullable, n_lags_vae):
    X_seq_list = []; num_target_features = 1; num_exog_features = exog_df_scaled_nullable.shape[1] if exog_df_scaled_nullable is not None and not exog_df_scaled_nullable.empty else 0
    for i in range(len(target_series_1d_scaled) - n_lags_vae + 1):
        target_window_reshaped = target_series_1d_scaled[i : i + n_lags_vae].reshape(-1, num_target_features)
        if num_exog_features > 0:
            exog_window = exog_df_scaled_nullable.iloc[i : i + n_lags_vae].values
            if exog_window.shape[0] != n_lags_vae: continue
            combined_window = np.concatenate((target_window_reshaped, exog_window), axis=1)
        else: combined_window = target_window_reshaped
        X_seq_list.append(combined_window)
    if not X_seq_list: return np.array([]).reshape(0, n_lags_vae, num_target_features + num_exog_features)
    return np.array(X_seq_list)

def impute_segment_with_lstm_vae_batch(target_series_1d_with_nans, exog_df_for_segment_scaled, vae_model, scaler_target, n_lags, fallback_value_unscaled):
    imputed_series_unscaled = pd.Series(target_series_1d_with_nans.copy().flatten())
    original_nan_indices = imputed_series_unscaled[imputed_series_unscaled.isnull()].index.tolist()
    if not original_nan_indices: return imputed_series_unscaled.values.reshape(-1, 1)

    temp_filled_series = imputed_series_unscaled.ffill().bfill()
    if temp_filled_series.isnull().any(): temp_filled_series.fillna(fallback_value_unscaled, inplace=True)

    input_sequences, valid_nan_indices_for_prediction = [], []
    for idx_nan in original_nan_indices:
        if idx_nan < n_lags: continue
        start_idx, end_idx = idx_nan - n_lags, idx_nan
        target_window_unscaled = temp_filled_series.iloc[start_idx:end_idx].values
        target_window_scaled = scaler_target.transform(target_window_unscaled.reshape(-1, 1))
        
        if exog_df_for_segment_scaled is not None and not exog_df_for_segment_scaled.empty:
            exog_window_scaled = exog_df_for_segment_scaled.iloc[start_idx:end_idx].values
            vae_input_sequence = np.concatenate((target_window_scaled, exog_window_scaled), axis=1)
        else: vae_input_sequence = target_window_scaled
        
        input_sequences.append(vae_input_sequence)
        valid_nan_indices_for_prediction.append(idx_nan)

    if not valid_nan_indices_for_prediction:
        return imputed_series_unscaled.ffill().bfill().fillna(fallback_value_unscaled).values.reshape(-1, 1)

    X_to_predict = np.array(input_sequences)
    reconstructed_sequences_scaled = vae_model.predict(X_to_predict, batch_size=256, verbose=0)
    imputed_values_scaled = reconstructed_sequences_scaled[:, -1, 0]
    imputed_values_unscaled = scaler_target.inverse_transform(imputed_values_scaled.reshape(-1, 1)).flatten()
    
    imputed_series_unscaled.iloc[valid_nan_indices_for_prediction] = imputed_values_unscaled
    if imputed_series_unscaled.isnull().any():
        imputed_series_unscaled = imputed_series_unscaled.ffill().bfill().fillna(fallback_value_unscaled)

    return imputed_series_unscaled.values.reshape(-1, 1)

def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {};
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values.astype(bool)
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae, rmse = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid), np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range, nrmse_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan), (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal de Imputación con LSTM-VAE ---
all_detailed_metrics_lstm_vae_results = []
script_start_time_lstm_vae = time.time()

cities_to_process_vae = [TARGET_CITY_VAE] if TARGET_CITY_VAE else (df_air_quality_full_original['City'].unique() if not df_air_quality_full_original.empty else [])
if TARGET_CITY_VAE and TARGET_CITY_VAE not in df_air_quality_full_original['City'].unique():
    print(f"Advertencia: La ciudad objetivo '{TARGET_CITY_VAE}' no se encuentra en el dataset."); cities_to_process_vae = []

# --- BUCLE PRINCIPAL POR CIUDAD ---
for current_city_name_vae in cities_to_process_vae:
    print(f"\n=== PROCESANDO CIUDAD: {current_city_name_vae} ===")
    df_city_original_clean = df_air_quality_full_original[df_air_quality_full_original['City'] == current_city_name_vae].copy()
    if df_city_original_clean.empty: print(f"No hay datos para {current_city_name_vae}. Omitiendo."); continue

    trained_vaes = {}; scalers_target = {}
    df_exog_base_city = df_city_original_clean[TIME_FEATURES].copy().ffill().bfill()
    if df_exog_base_city.isnull().any().any(): print(f"Error: NaNs en TFs globales para {current_city_name_vae}."); continue

    scaler_exog_city = MinMaxScaler(feature_range=(0,1))
    if df_exog_base_city.empty: print(f"Advertencia: df_exog_base_city está vacía para {current_city_name_vae}."); continue
    scaled_exog_global_city_df = pd.DataFrame(scaler_exog_city.fit_transform(df_exog_base_city), index=df_exog_base_city.index, columns=df_exog_base_city.columns)
    
    city_model_weights_root_dir = os.path.join(MODELS_LSTM_VAE_OUTPUT_DIR, sanitize_tf_name(current_city_name_vae))
    os.makedirs(city_model_weights_root_dir, exist_ok=True)
    joblib.dump(scaler_exog_city, os.path.join(city_model_weights_root_dir, f'scaler_exog_GLOBAL_{sanitize_tf_name(current_city_name_vae)}.joblib'))
    n_exog_features_global = scaled_exog_global_city_df.shape[1]
    n_features_input_global_vae = 1 + n_exog_features_global

    # --- Bucle de Entrenamiento (uno por variable, con WFV) ---
    for target_var_to_impute in ALL_TARGET_COLS_VAE:
        K.clear_session(); base_seed_val = int(time.time_ns() / 1000) + hash(target_var_to_impute + current_city_name_vae)
        tf.random.set_seed(abs(base_seed_val) % (2**32 - 1)); np.random.seed(abs(base_seed_val + 1) % (2**32 - 1)); random.seed(abs(base_seed_val + 2) % (2**32 - 1))
        print(f"\n--- Preparando VAE para VARIABLE: {target_var_to_impute} (Ciudad: {current_city_name_vae}) ---")
        
        series_clean_target_wfv = df_city_original_clean[target_var_to_impute].ffill().bfill()
        if series_clean_target_wfv.isnull().any(): print(f"  Serie {target_var_to_impute} con NaNs irreparables. Omitiendo."); continue
        
        current_scaler_target_var = MinMaxScaler(feature_range=(0,1)); scaled_target_wfv_full_for_var = current_scaler_target_var.fit_transform(series_clean_target_wfv.values.reshape(-1,1)); scalers_target[target_var_to_impute] = current_scaler_target_var
        var_model_weights_dir = os.path.join(city_model_weights_root_dir, sanitize_tf_name(target_var_to_impute)); os.makedirs(var_model_weights_dir, exist_ok=True)
        joblib.dump(current_scaler_target_var, os.path.join(var_model_weights_dir, f'scaler_target_{sanitize_tf_name(target_var_to_impute)}.joblib'))
        
        print(f"  Iniciando WFV para VAE de {target_var_to_impute}...")
        total_clean_points_var = len(series_clean_target_wfv)
        wfv_init_train_len = int(total_clean_points_var * WFV_INITIAL_TRAIN_POINTS_MIN_RATIO)
        wfv_val_len = int(total_clean_points_var * WFV_VALIDATION_POINTS_MIN_RATIO)
        wfv_step_len = max(1, int(total_clean_points_var * WFV_STEP_POINTS_MIN_RATIO))
        
        if total_clean_points_var < wfv_init_train_len + wfv_val_len + N_LAGS_VAE + 10: print(f"  Datos insuficientes para WFV de {target_var_to_impute}. Omitiendo entrenamiento."); trained_vaes[target_var_to_impute] = None; continue

        current_train_end_idx_wfv, fold_num_wfv, best_val_loss_overall_wfv = wfv_init_train_len, 0, float('inf')
        vae_model_weights_path_final_var = os.path.join(var_model_weights_dir, f'vae_weights_final_{sanitize_tf_name(target_var_to_impute)}.weights.h5')
        
        current_encoder_arch_vae, current_decoder_arch_vae = create_vae_model_architecture(N_LAGS_VAE, n_features_input_global_vae, LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE, var_name_for_layer_names=target_var_to_impute)
        
        if not os.path.exists(vae_model_weights_path_final_var):
            print(f"  No se encontraron pesos VAE pre-entrenados para {target_var_to_impute}. Se entrenará un nuevo modelo.")
            
        while (current_train_end_idx_wfv + wfv_val_len <= total_clean_points_var) and (fold_num_wfv < MAX_WFV_FOLDS):
            fold_num_wfv += 1
            train_target_scaled_fold = scaled_target_wfv_full_for_var[:current_train_end_idx_wfv]
            val_target_scaled_fold = scaled_target_wfv_full_for_var[current_train_end_idx_wfv : current_train_end_idx_wfv + wfv_val_len]
            train_exog_scaled_fold_df = scaled_exog_global_city_df.iloc[:current_train_end_idx_wfv]
            val_exog_scaled_fold_df = scaled_exog_global_city_df.iloc[current_train_end_idx_wfv : current_train_end_idx_wfv + wfv_val_len]
            print(f"    WFV Fold {fold_num_wfv} ({target_var_to_impute}): Train {len(train_target_scaled_fold)}, Val {len(val_target_scaled_fold)}")
            
            X_train_fold_vae = create_sequences_for_vae_training(train_target_scaled_fold.flatten(), train_exog_scaled_fold_df, N_LAGS_VAE)
            X_val_fold_vae = create_sequences_for_vae_training(val_target_scaled_fold.flatten(), val_exog_scaled_fold_df, N_LAGS_VAE)
            
            if X_train_fold_vae.shape[0] < VAE_BATCH_SIZE or X_val_fold_vae.shape[0] == 0:
                print(f"      Fold {fold_num_wfv} con secuencias insuficientes. Avanzando."); current_train_end_idx_wfv += wfv_step_len; continue
            
            fold_vae_model = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT)
            
            # SOLUTION: Add loss=None to the compile call
            fold_vae_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR, weight_decay=0.001), loss=None)
            
            if not fold_vae_model.built: fold_vae_model(tf.zeros((1, N_LAGS_VAE, n_features_input_global_vae)))
            
            if fold_num_wfv > 1 and os.path.exists(vae_model_weights_path_final_var):
                try: fold_vae_model.load_weights(vae_model_weights_path_final_var)
                except Exception as e_lw: print(f"      Error cargando pesos globales en fold {fold_num_wfv}: {e_lw}")

            callbacks_list = [
                keras.callbacks.EarlyStopping(monitor='val_total_loss', mode='min', patience=WFV_ES_PATIENCE, restore_best_weights=True),
                keras.callbacks.ReduceLROnPlateau(monitor='val_total_loss', mode='min', factor=0.5, patience=WFV_LR_PATIENCE, min_lr=1e-7)
            ]
            
            print(f"    Iniciando entrenamiento del Fold {fold_num_wfv} para {target_var_to_impute}...")
            fold_history_vae = fold_vae_model.fit(
                X_train_fold_vae, # Datos de entrenamiento
                # X_train_fold_vae, # Target (es el mismo para VAE) # <--- Keras 3 infiere esto si y=None
                epochs=WFV_EPOCHS_PER_FOLD,
                batch_size=VAE_BATCH_SIZE,
                validation_data=X_val_fold_vae, # Datos de validación # <--- Keras 3 infiere target de val si y_val=None
                callbacks=callbacks_list,
                verbose=1 # o verbose=2. verbose=1 muestra barra de progreso y métricas al final de la época.
            )
            
            # Después del entrenamiento del fold, puedes imprimir un resumen si lo deseas.
            # La historia ya está en fold_history_vae.history
            # El EarlyStopping con restore_best_weights=True ya habrá restaurado los mejores pesos.
            # `current_fold_val_loss` se refiere a la mejor `val_total_loss` obtenida durante este fold.
            current_fold_val_loss = fold_vae_model.evaluate(X_val_fold_vae, verbose=0)[0] # Evaluar el modelo restaurado para obtener su val_total_loss
            # El índice [0] asume que 'total_loss' es la primera métrica devuelta por evaluate, que lo es
            # porque es la primera en self.metrics y también es la loss compilada.
            # O, si confías en que EarlyStopping restauró la mejor y quieres ese valor exacto del historial:
            # current_fold_val_loss = min(fold_history_vae.history.get('val_total_loss', [float('inf')]))

            print(f"      Fold {fold_num_wfv} completado. Val Total Loss (del modelo restaurado/mejor): {current_fold_val_loss:.4f}")
            if current_fold_val_loss < best_val_loss_overall_wfv:
                best_val_loss_overall_wfv = current_fold_val_loss
                print(f"      NUEVO MEJOR VAE ({target_var_to_impute}, loss: {best_val_loss_overall_wfv:.4f}). Guardando pesos.")
                fold_vae_model.save_weights(vae_model_weights_path_final_var)

            current_train_end_idx_wfv += wfv_step_len; K.clear_session()

        final_vae_model_for_var = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT)
        
        # SOLUTION: Add loss=None to the compile call here as well
        final_vae_model_for_var.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR), loss=None)
        
        if not final_vae_model_for_var.built: final_vae_model_for_var(tf.zeros((1, N_LAGS_VAE, n_features_input_global_vae)))
        
        if os.path.exists(vae_model_weights_path_final_var):
            try: 
                final_vae_model_for_var.load_weights(vae_model_weights_path_final_var)
                print(f"  Modelo VAE final para {target_var_to_impute} cargado para imputación.")
            except Exception as e: print(f"  Error cargando pesos finales VAE {target_var_to_impute}: {e}."); final_vae_model_for_var = None
        else: print(f"  Pesos VAE no encontrados para {target_var_to_impute} después de WFV."); final_vae_model_for_var = None
        
        trained_vaes[target_var_to_impute] = final_vae_model_for_var

    # --- Imputation Phase for the current city ---
    print(f"\n--- Iniciando Imputación para {current_city_name_vae} ---")
    df_city_original_gt_reset_for_metrics = df_city_original_clean.reset_index(drop=True)
    city_name_safe_for_files = sanitize_tf_name(current_city_name_vae)

    for scenario_vae in NAN_SIMULATION_SCENARIOS:
        scenario_name_for_dir = scenario_vae['name'].replace("%", "percent")
        scenario_name_safe_vae = sanitize_tf_name(scenario_name_for_dir)
        print(f"  Procesando Escenario: {scenario_name_for_dir}")
        
        imputed_unified_scenario_dir = os.path.join(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, city_name_safe_for_files, scenario_name_safe_vae)
        os.makedirs(imputed_unified_scenario_dir, exist_ok=True)

        for rep_vae in range(1, N_REPETITIONS_FOR_EVAL + 1):
            sim_data_scenario_dir_for_rep = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe_for_files, scenario_name_for_dir)
            matching_files_sim = glob.glob(os.path.join(sim_data_scenario_dir_for_rep, f"{city_name_safe_for_files}_{scenario_name_safe_vae}_rep{rep_vae}_*_with_nans.csv"))
            
            if not matching_files_sim: print(f"    Archivo no encontrado para rep {rep_vae} en {sim_data_scenario_dir_for_rep}. Omitiendo."); continue
            
            sim_file_path_to_load = matching_files_sim[0]
            current_seed_loaded_vae = os.path.basename(sim_file_path_to_load).split('_seed')[-1].split('_with_nans.csv')[0]
            
            try: df_with_nans_all_vars_loaded = pd.read_csv(sim_file_path_to_load)
            except Exception as e: print(f"    Error cargando {sim_file_path_to_load}: {e}."); continue

            df_to_save_imputed_all_vars_rep = df_with_nans_all_vars_loaded.copy()
            df_with_nans_all_vars_loaded['Date'] = pd.to_datetime(df_with_nans_all_vars_loaded['Date'])
            
            current_scaled_exog_for_rep_df = None
            if n_exog_features_global > 0 and scaler_exog_city is not None:
                exog_for_this_rep_df = pd.DataFrame(index=range(len(df_with_nans_all_vars_loaded)))
                if hasattr(df_with_nans_all_vars_loaded['Date'].dt, 'hour'):
                    exog_for_this_rep_df['Hour'] = df_with_nans_all_vars_loaded['Date'].dt.hour
                    exog_for_this_rep_df['DayOfWeek'] = df_with_nans_all_vars_loaded['Date'].dt.dayofweek
                    exog_for_this_rep_df['Month'] = df_with_nans_all_vars_loaded['Date'].dt.month
                    current_scaled_exog_for_rep_df = pd.DataFrame(scaler_exog_city.transform(exog_for_this_rep_df), columns=TIME_FEATURES)

            nan_mask_for_metrics_rep = pd.DataFrame(False, index=df_city_original_gt_reset_for_metrics.index, columns=ALL_TARGET_COLS_VAE)
            df_with_nans_reset = df_with_nans_all_vars_loaded.reset_index(drop=True)
            for col_mask in ALL_TARGET_COLS_VAE:
                if col_mask in df_with_nans_reset.columns:
                    mask_len = min(len(df_with_nans_reset), len(df_city_original_gt_reset_for_metrics))
                    nan_mask_for_metrics_rep.loc[:mask_len-1, col_mask] = df_with_nans_reset[col_mask].iloc[:mask_len].isnull() & ~df_city_original_gt_reset_for_metrics[col_mask].iloc[:mask_len].isnull()

            for target_var_impute_loop in ALL_TARGET_COLS_VAE:
                vae_model_for_var, scaler_target_for_var = trained_vaes.get(target_var_impute_loop), scalers_target.get(target_var_impute_loop)
                if vae_model_for_var is None or scaler_target_for_var is None:
                    df_to_save_imputed_all_vars_rep[target_var_impute_loop] = df_to_save_imputed_all_vars_rep[target_var_impute_loop].interpolate(method='linear').ffill().bfill()
                    continue

                imputed_values_for_var_2d = impute_segment_with_lstm_vae_batch(
                    df_with_nans_all_vars_loaded[target_var_impute_loop].values,
                    current_scaled_exog_for_rep_df,
                    vae_model_for_var, scaler_target_for_var, N_LAGS_VAE,
                    fallback_value_unscaled=df_city_original_gt_reset_for_metrics[target_var_impute_loop].mean()
                )
                df_to_save_imputed_all_vars_rep[target_var_impute_loop] = imputed_values_for_var_2d.flatten()

            unified_imputed_filename = f"{city_name_safe_for_files}_{scenario_name_safe_vae}_rep{rep_vae}_seed{current_seed_loaded_vae}_imputed_vae_ALL.csv"
            unified_imputed_file_save_path = os.path.join(imputed_unified_scenario_dir, unified_imputed_filename)
            df_to_save_imputed_all_vars_rep.to_csv(unified_imputed_file_save_path, index=False)
            print(f"      Archivo unificado imputado guardado: {unified_imputed_filename}")

            metrics_scores_vae_rep = calculate_metrics_with_nmae_nrmse(
                df_city_original_gt_reset_for_metrics, df_to_save_imputed_all_vars_rep,
                nan_mask_for_metrics_rep, ALL_TARGET_COLS_VAE
            )
            for var_name_metric in ALL_TARGET_COLS_VAE:
                all_detailed_metrics_lstm_vae_results.append({
                    'City': current_city_name_vae, 'Scenario': scenario_name_safe_vae, 'Repetition': rep_vae,
                    'Seed': current_seed_loaded_vae, 'Imputation_Method': LSTM_VAE_METHOD_NAME,
                    'Variable': var_name_metric, 'MAE': metrics_scores_vae_rep[var_name_metric]['MAE'],
                    'RMSE': metrics_scores_vae_rep[var_name_metric]['RMSE'], 'NRMAE_range': metrics_scores_vae_rep[var_name_metric]['NRMAE_range'],
                    'NRMSE_range': metrics_scores_vae_rep[var_name_metric]['NRMSE_range']
                })
            print(f"    Métricas calculadas para rep {rep_vae}, Escenario {scenario_name_for_dir}")
            
    print(f"  Procesamiento de escenarios para {current_city_name_vae} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_lstm_vae_results:
    df_all_detailed_metrics_vae = pd.DataFrame(all_detailed_metrics_lstm_vae_results)
    print("\nGenerando resúmenes de métricas LSTM-VAE...")
    sufijo_nombre_archivo = f"_{sanitize_tf_name(TARGET_CITY_VAE)}" if TARGET_CITY_VAE else "_ALL_CITIES"
    detailed_metrics_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'ALL_metrics_detailed_{LSTM_VAE_METHOD_NAME}{sufijo_nombre_archivo}.csv')
    df_all_detailed_metrics_vae.to_csv(detailed_metrics_path_vae, index=False)
    print(f"  Métricas detalladas LSTM-VAE guardadas en: {detailed_metrics_path_vae}")
    if not df_all_detailed_metrics_vae.empty: print(df_all_detailed_metrics_vae.head())

    if not df_all_detailed_metrics_vae.empty:
        grouped_by_city_var_vae = df_all_detailed_metrics_vae.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_city_var_vae.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        for city_name_summary_val in grouped_by_city_var_vae['City'].unique():
            city_metrics_df_summary = grouped_by_city_var_vae[grouped_by_city_var_vae['City'] == city_name_summary_val]
            variables_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_variables_summary_{LSTM_VAE_METHOD_NAME}_{sanitize_tf_name(city_name_summary_val)}.csv')
            city_metrics_df_summary.to_csv(variables_summary_path_vae, index=False)
            print(f"  Resumen LSTM-VAE de variables para {city_name_summary_val} guardado en: {variables_summary_path_vae}")

        contaminant_cols_for_summary = [c for c in ALL_TARGET_COLS_VAE if c != 'AQI']
        df_contaminants_only_vae = df_all_detailed_metrics_vae[df_all_detailed_metrics_vae['Variable'].isin(contaminant_cols_for_summary)]
        if not df_contaminants_only_vae.empty:
            contaminants_combined_summary_vae = df_contaminants_only_vae.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_vae.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{LSTM_VAE_METHOD_NAME}{sufijo_nombre_archivo}.csv')
            contaminants_combined_summary_vae.to_csv(contaminants_summary_path_vae, index=False); print(f"  Resumen LSTM-VAE contaminantes guardado: {contaminants_summary_path_vae}"); print(contaminants_combined_summary_vae)

        df_aqi_only_vae = df_all_detailed_metrics_vae[df_all_detailed_metrics_vae['Variable'] == 'AQI']
        if not df_aqi_only_vae.empty:
            aqi_combined_summary_vae = df_aqi_only_vae.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            aqi_combined_summary_vae.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
            aqi_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_aqi_combined_summary_{LSTM_VAE_METHOD_NAME}{sufijo_nombre_archivo}.csv')
            aqi_combined_summary_vae.to_csv(aqi_summary_path_vae, index=False); print(f"  Resumen LSTM-VAE AQI guardado: {aqi_summary_path_vae}"); print(aqi_combined_summary_vae)
else: print("No se generaron métricas detalladas para LSTM-VAE.")

script_end_time_lstm_vae = time.time()
print(f"\nProceso LSTM-VAE completado. Tiempo: {(script_end_time_lstm_vae - script_start_time_lstm_vae) / 60:.2f} min.")

Datos originales cargados desde: CSVs\Air_Quality.csv

Convirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...
Conversión de zona horaria y extracción de características de tiempo completada.

=== PROCESANDO CIUDAD: Brasilia ===


--- Preparando VAE para VARIABLE: CO (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de CO...
    WFV Fold 1 (CO): Train 878, Val 878
    Iniciando entrenamiento del Fold 1 para CO...
Epoch 1/30
26/26 ━━━━━━━━━━━━━━━━━━━━ 7s 77ms/step - kl_loss: 0.1154 - reconstruction_loss: 0.0576 - total_loss: 0.0576 - val_kl_loss: 0.2499 - val_reconstruction_loss: 0.0125 - val_total_loss: 0.0125 - learning_rate: 5.0000e-04
Epoch 2/30
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - kl_loss: 0.2570 - reconstruction_loss: 0.0115 - total_loss: 0.0115 - val_kl_loss: 0.3733 - val_reconstruction_loss: 0.0095 - val_total_loss: 0.0096 - learning_rate: 5.0000e-04
Epoch 3/30
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - kl_loss: 0.4961 - reconstructi

MOMENT-PRE

In [2]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos Originales (Ground Truth) ---
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia'] 
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        num_rows_example = 550 
        for i in range(num_rows_example): 
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])
df_air_quality_ground_truth['Hour'] = df_air_quality_ground_truth['Date'].dt.hour
df_air_quality_ground_truth['DayOfWeek'] = df_air_quality_ground_truth['Date'].dt.dayofweek
df_air_quality_ground_truth['Month'] = df_air_quality_ground_truth['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# --- Parámetros Configurables ---
TARGET_CITY_MOMENT = "Brasilia"
CONTAMINANT_COLS_MOMENT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_MOMENT = ['AQI']
ALL_TARGET_COLS_MOMENT = CONTAMINANT_COLS_MOMENT + AQI_COL_MOMENT

SEQ_LEN_MODEL_MOMENT = 512
IMPUTATION_STEP_MOMENT = SEQ_LEN_MODEL_MOMENT
BATCH_SIZE_IMPUTE_MOMENT = 16
MODEL_NAME_MOMENT_PRETRAINED = "AutonLab/MOMENT-1-large"

N_REPETITIONS_MOMENT_EVAL = 1
NAN_SIMULATION_SCENARIOS_MOMENT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_MOMENT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME = 'MOMENT_PRE' # Nuevo nombre para esta prueba
IMPUTATIONS_MOMENT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'imputaciones', MOMENT_METHOD_NAME)
METRICS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'metricas')
IMPUTED_CSVS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'CSVs')
SCALERS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'scalers')
os.makedirs(METRICS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_MOMENT_OUTPUT_DIR, exist_ok=True)

# ======================================================================
# --- Dataset Personalizado para MOMENT (CON LA CORRECCIÓN FINAL) ---
# ======================================================================
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data_array_scaled_zeros = data_array_2d_multivar_scaled_zeros
        self.nan_mask_original = nan_mask_original_bool_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array_scaled_zeros) < self.seq_len : self.n_samples, self.indices = 1, [0]
        elif len(self.data_array_scaled_zeros) == 0: self.n_samples, self.indices = 0, []
        else:
            self.indices = list(range(0, len(self.data_array_scaled_zeros) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array_scaled_zeros) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array_scaled_zeros) >= self.seq_len: self.indices, self.n_samples = [0],1
            
    def __len__(self): return self.n_samples
    
    def __getitem__(self, idx):
        actual_start_index = self.indices[idx]
        seq_data_scaled_zeros_orig_shape = self.data_array_scaled_zeros[actual_start_index : actual_start_index + self.seq_len]
        seq_input_mask_for_moment_bool_orig_shape = ~self.nan_mask_original[actual_start_index : actual_start_index + self.seq_len]
        
        if len(seq_data_scaled_zeros_orig_shape) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_scaled_zeros_orig_shape)
            seq_data_scaled_zeros_orig_shape = np.pad(seq_data_scaled_zeros_orig_shape, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_input_mask_for_moment_bool_orig_shape = np.pad(seq_input_mask_for_moment_bool_orig_shape, ((0, pad_width), (0,0)), 'constant', constant_values=False)
            
        # El tensor de la serie temporal necesita la forma (n_channels, seq_len)
        timeseries_out = torch.tensor(seq_data_scaled_zeros_orig_shape, dtype=torch.float32).transpose(0, 1)

        # La máscara de entrada NO debe tener la dimensión de canal.
        # Su forma debe ser (seq_len), el DataLoader añadirá la dimensión de batch.
        # Usamos .squeeze() para eliminar la dimensión del canal (que es 1).
        input_mask_out = torch.tensor(seq_input_mask_for_moment_bool_orig_shape, dtype=torch.bool).squeeze()
        
        return {
            'timeseries': timeseries_out,
            'input_mask': input_mask_out
        }

# (El resto del script no necesita cambios)

# --- Función de Imputación con MOMENT ---
def impute_multivar_with_moment_pipeline(df_with_nans_city_scaled_zeros, original_nan_mask_df_bool, moment_pipeline_obj, device_str, scaler_dict_vars_city, seq_len_moment, imputation_step_moment, batch_size_moment):
    imputation_dataset = MomentImputationDataset(df_with_nans_city_scaled_zeros.values, original_nan_mask_df_bool.values, seq_len_moment, imputation_step_moment)
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_moment, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_with_nans_city_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = moment_pipeline_obj(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e:
                continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_moment + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_segment_loop = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_segment_loop = min(start_idx_in_segment_loop + seq_len_moment, n_rows_total)
                len_window_actual = end_idx_in_segment_loop - start_idx_in_segment_loop
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_segment_loop:end_idx_in_segment_loop, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_segment_loop:end_idx_in_segment_loop, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_segment_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_segment_unscaled_df = pd.DataFrame(columns=df_with_nans_city_scaled_zeros.columns, index=df_with_nans_city_scaled_zeros.index, dtype=float)
    for col_name_unscale in df_with_nans_city_scaled_zeros.columns:
        scaler_var_unscale = scaler_dict_vars_city.get(col_name_unscale)
        if scaler_var_unscale:
            col_idx_unscale = df_with_nans_city_scaled_zeros.columns.get_loc(col_name_unscale)
            imputed_full_segment_unscaled_df[col_name_unscale] = scaler_var_unscale.inverse_transform(imputed_full_segment_scaled[:, col_idx_unscale].reshape(-1, 1)).flatten()
        else:
            imputed_full_segment_unscaled_df[col_name_unscale] = imputed_full_segment_scaled[:, df_with_nans_city_scaled_zeros.columns.get_loc(col_name_unscale)]
    df_final_imputed_unscaled = df_with_nans_city_scaled_zeros.copy()
    for col_orig_unscale in df_with_nans_city_scaled_zeros.columns:
        scaler_orig_var = scaler_dict_vars_city.get(col_orig_unscale)
        if scaler_orig_var and not (df_final_imputed_unscaled[col_orig_unscale] == 0).all():
            df_final_imputed_unscaled[col_orig_unscale] = scaler_orig_var.inverse_transform(df_final_imputed_unscaled[[col_orig_unscale]].values).flatten()
    for col_final_fill in df_final_imputed_unscaled.columns:
        if col_final_fill in imputed_full_segment_unscaled_df.columns:
            col_nan_mask_original = original_nan_mask_df_bool[col_final_fill].values
            df_final_imputed_unscaled.loc[col_nan_mask_original, col_final_fill] = imputed_full_segment_unscaled_df.loc[col_nan_mask_original, col_final_fill]
    return df_final_imputed_unscaled

# --- Función de Métricas ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan)
            nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_moment_results = []
script_start_time_moment = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
print(f"Cargando modelo MOMENT pre-entrenado: {MODEL_NAME_MOMENT_PRETRAINED}")
moment_pipeline_global = None
try:
    moment_pipeline_global = MOMENTPipeline.from_pretrained(
        MODEL_NAME_MOMENT_PRETRAINED, model_kwargs={'task_name': 'reconstruction'}
    )
    moment_pipeline_global.init()
    moment_pipeline_global = moment_pipeline_global.to(device)
    print("Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.")
except Exception as e_model_load:
    print(f"Error crítico cargando modelo MOMENT: {e_model_load}"); traceback.print_exc(); exit()

df_city_ground_truth_moment = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == TARGET_CITY_MOMENT].copy()
if df_city_ground_truth_moment.empty: print(f"No hay datos GT para {TARGET_CITY_MOMENT}."); exit()
for col_gt_fill in ALL_TARGET_COLS_MOMENT: 
    if col_gt_fill in df_city_ground_truth_moment.columns: df_city_ground_truth_moment[col_gt_fill] = df_city_ground_truth_moment[col_gt_fill].ffill().bfill()
if df_city_ground_truth_moment[ALL_TARGET_COLS_MOMENT].isnull().any().any(): print(f"No se pudo rellenar GT para {TARGET_CITY_MOMENT}."); exit()
df_city_ground_truth_moment_reset = df_city_ground_truth_moment.reset_index(drop=True)

scalers_for_city_moment = {}
city_scalers_dir = os.path.join(SCALERS_MOMENT_OUTPUT_DIR, TARGET_CITY_MOMENT.replace(" ", "_")); os.makedirs(city_scalers_dir, exist_ok=True)
for col_scale in ALL_TARGET_COLS_MOMENT: 
    scaler_var = StandardScaler()
    data_to_fit_scaler = df_city_ground_truth_moment_reset[[col_scale]].dropna().values
    if len(data_to_fit_scaler) > 1:
        scaler_var.fit(data_to_fit_scaler)
        scalers_for_city_moment[col_scale] = scaler_var
        joblib.dump(scaler_var, os.path.join(city_scalers_dir, f'scaler_{col_scale}.joblib'))
    else:
        class DummyScaler:
            def fit(self, X): return self
            def transform(self, X): return np.array(X, dtype=float)
            def inverse_transform(self, X): return np.array(X, dtype=float)
        scalers_for_city_moment[col_scale] = DummyScaler()

city_name_safe_moment = TARGET_CITY_MOMENT.replace(" ", "_")
city_imputed_csv_moment_dir_root = os.path.join(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, city_name_safe_moment); os.makedirs(city_imputed_csv_moment_dir_root, exist_ok=True)

for scenario_moment in NAN_SIMULATION_SCENARIOS_MOMENT:
    scenario_name_safe_moment = scenario_moment['name'].replace("%", "percent").replace(" ", "_")
    print(f"\nProcesando Escenario: {scenario_name_safe_moment}")
    sim_data_scenario_dir_moment = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT, city_name_safe_moment, scenario_name_safe_moment)
    imputed_moment_scenario_dir = os.path.join(city_imputed_csv_moment_dir_root, scenario_name_safe_moment); os.makedirs(imputed_moment_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_moment): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_moment}."); continue
    
    for rep_moment in range(1, N_REPETITIONS_MOMENT_EVAL + 1):
        print(f"  Repetición: {rep_moment}")
        sim_files_in_dir_moment = sorted([f for f in os.listdir(sim_data_scenario_dir_moment) if f.startswith(f"{city_name_safe_moment}_{scenario_name_safe_moment}_rep{rep_moment}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_moment: print(f"    Archivo simulado no encontrado para rep {rep_moment}."); continue
        sim_file_path_to_load_moment = os.path.join(sim_data_scenario_dir_moment, sim_files_in_dir_moment[0])
        current_seed_loaded_moment = sim_files_in_dir_moment[0].split('_seed')[-1].split('_with_nans.csv')[0]
        try: df_city_with_nans_loaded_moment = pd.read_csv(sim_file_path_to_load_moment)
        except Exception as e_load_rep_m: print(f"    Error cargando {sim_file_path_to_load_moment}: {e_load_rep_m}."); continue
        df_imputed_all_cols_rep = df_city_with_nans_loaded_moment[ALL_TARGET_COLS_MOMENT].copy()
        for col_to_impute in tqdm(ALL_TARGET_COLS_MOMENT, desc=f"  Imputando variables para Rep {rep_moment}", unit="var"):
            df_to_process_univar = df_city_with_nans_loaded_moment[[col_to_impute]].copy()
            scaler_current_var = scalers_for_city_moment.get(col_to_impute)
            df_scaled_univar = pd.DataFrame(index=df_to_process_univar.index, columns=[col_to_impute])
            if scaler_current_var and not df_to_process_univar.isnull().all().iloc[0]:
                df_scaled_univar[col_to_impute] = scaler_current_var.transform(df_to_process_univar.values)
            else:
                df_scaled_univar[col_to_impute] = df_to_process_univar[col_to_impute]
            original_nan_mask_univar_df = df_to_process_univar.isnull()
            df_scaled_filled_univar = df_scaled_univar.fillna(0.0)
            if moment_pipeline_global is None: continue
            df_imputed_univar_final = impute_multivar_with_moment_pipeline(df_scaled_filled_univar, original_nan_mask_univar_df, moment_pipeline_global, device, scalers_for_city_moment, SEQ_LEN_MODEL_MOMENT, IMPUTATION_STEP_MOMENT, BATCH_SIZE_IMPUTE_MOMENT)
            if df_imputed_univar_final is not None and col_to_impute in df_imputed_univar_final.columns:
                nan_mask_for_update = df_imputed_all_cols_rep[col_to_impute].isnull()
                df_imputed_all_cols_rep.loc[nan_mask_for_update, col_to_impute] = df_imputed_univar_final.loc[nan_mask_for_update, col_to_impute]
        df_imputed_moment_final_rep = df_imputed_all_cols_rep
        if df_imputed_moment_final_rep.isnull().all().all():
            print(f"    Falló la imputación MOMENT para rep {rep_moment} (resultado vacío).")
            continue
        df_imputed_moment_final_to_save = df_city_with_nans_loaded_moment[['Date', 'City']].copy()
        for col_save in ALL_TARGET_COLS_MOMENT:
            if col_save in df_imputed_moment_final_rep.columns:
                df_imputed_moment_final_to_save[col_save] = df_imputed_moment_final_rep[col_save]
            else:
                df_imputed_moment_final_to_save[col_save] = np.nan
        imputed_moment_filename = f"{city_name_safe_moment}_{scenario_name_safe_moment}_rep{rep_moment}_seed{current_seed_loaded_moment}_imputed_momentALL.csv"
        imputed_moment_file_save_path = os.path.join(imputed_moment_scenario_dir, imputed_moment_filename)
        final_cols_moment_save = ['Date', 'City'] + ALL_TARGET_COLS_MOMENT
        df_gt_city_reset_for_tf = df_city_ground_truth_moment_reset
        time_features_to_add = [tf for tf in TIME_FEATURES if tf in df_gt_city_reset_for_tf.columns]
        for tf_col_final_m in time_features_to_add:
            if tf_col_final_m not in df_imputed_moment_final_to_save.columns and tf_col_final_m in df_gt_city_reset_for_tf.columns:
                df_imputed_moment_final_to_save[tf_col_final_m] = df_gt_city_reset_for_tf[tf_col_final_m].values[:len(df_imputed_moment_final_to_save)]
            if tf_col_final_m not in final_cols_moment_save:
                final_cols_moment_save.append(tf_col_final_m)
        final_cols_moment_save = [c for c in final_cols_moment_save if c in df_imputed_moment_final_to_save.columns]
        df_imputed_moment_final_to_save = df_imputed_moment_final_to_save[final_cols_moment_save]
        df_imputed_moment_final_to_save.to_csv(imputed_moment_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT guardado: {imputed_moment_filename}")
        nan_mask_introduced_for_metrics_rep = pd.DataFrame(False, index=df_city_ground_truth_moment_reset.index, columns=ALL_TARGET_COLS_MOMENT)
        for col_m_mask_rep in ALL_TARGET_COLS_MOMENT:
            if col_m_mask_rep in df_city_with_nans_loaded_moment.columns and col_m_mask_rep in df_city_ground_truth_moment_reset.columns:
                nan_mask_introduced_for_metrics_rep[col_m_mask_rep] = df_city_with_nans_loaded_moment[col_m_mask_rep].isnull() & ~df_city_ground_truth_moment_reset[col_m_mask_rep].isnull()
        metrics_scores_moment_rep = calculate_metrics_moment(df_city_ground_truth_moment_reset, df_imputed_moment_final_to_save, nan_mask_introduced_for_metrics_rep, ALL_TARGET_COLS_MOMENT)
        for var_name_metric_m in ALL_TARGET_COLS_MOMENT:
            all_detailed_metrics_moment_results.append({'City': TARGET_CITY_MOMENT, 'Scenario': scenario_name_safe_moment, 'Repetition': rep_moment, 'Seed': current_seed_loaded_moment, 'Imputation_Method': MOMENT_METHOD_NAME, 'Variable': var_name_metric_m, 'MAE': metrics_scores_moment_rep[var_name_metric_m]['MAE'], 'RMSE': metrics_scores_moment_rep[var_name_metric_m]['RMSE'], 'NRMAE_range': metrics_scores_moment_rep[var_name_metric_m]['NRMAE_range'], 'NRMSE_range': metrics_scores_moment_rep[var_name_metric_m]['NRMSE_range']})
        print(f"    Métricas calculadas para MOMENT, rep {rep_moment}")
    print(f"  Escenario {scenario_name_safe_moment} completado.")

# --- Guardado de Métricas ---
if all_detailed_metrics_moment_results:
    df_all_detailed_metrics_moment = pd.DataFrame(all_detailed_metrics_moment_results)
    print("\nGenerando resúmenes de métricas MOMENT...")
    sufijo_test_m = "" 
    if len(ALL_TARGET_COLS_MOMENT) == 1:
        sufijo_test_m = f"_SingleVar_{ALL_TARGET_COLS_MOMENT[0]}"
    
    detailed_metrics_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
    df_all_detailed_metrics_moment.to_csv(detailed_metrics_path_m, index=False)
    print(f"  Métricas detalladas MOMENT guardadas en: {detailed_metrics_path_m}")
    if not df_all_detailed_metrics_moment.empty:
        print(df_all_detailed_metrics_moment.head())
    
    grouped_by_var_m = df_all_detailed_metrics_moment.groupby(['Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_var_m.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    
    variables_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
    
    grouped_by_var_m.to_csv(variables_summary_path_m, index=False)
    print(f"  Resumen MOMENT de variables guardado: {variables_summary_path_m}")
    print(grouped_by_var_m)
    
    if len(ALL_TARGET_COLS_MOMENT) > 1:
        CONTAMINANT_COLS_MOMENT_ORIG, AQI_COL_MOMENT_ORIG = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10'], ['AQI']
        df_contaminants_only_m = df_all_detailed_metrics_moment[df_all_detailed_metrics_moment['Variable'].isin(CONTAMINANT_COLS_MOMENT_ORIG)]
        if not df_contaminants_only_m.empty:
            contaminants_combined_summary_m = df_contaminants_only_m.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_m.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
            contaminants_combined_summary_m.to_csv(contaminants_summary_path_m, index=False)
            print(f"  Resumen MOMENT contaminantes guardado: {contaminants_summary_path_m}")
            print(contaminants_combined_summary_m)
        
        df_aqi_only_m = df_all_detailed_metrics_moment[df_all_detailed_metrics_moment['Variable'] == AQI_COL_MOMENT_ORIG[0]]
        if not df_aqi_only_m.empty:
            aqi_combined_summary_m = df_aqi_only_m.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            aqi_combined_summary_m.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
            aqi_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
            aqi_combined_summary_m.to_csv(aqi_summary_path_m, index=False)
            print(f"  Resumen MOMENT AQI guardado: {aqi_summary_path_m}")
            print(aqi_combined_summary_m)
else:
    print("No se generaron métricas detalladas para MOMENT.")

script_end_time_moment = time.time()
print(f"\nProceso de Imputación con MOMENT Preentrenado completado. Tiempo total: {(script_end_time_moment - script_start_time_moment) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cpu
Cargando modelo MOMENT pre-entrenado: AutonLab/MOMENT-1-large
Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.

Procesando Escenario: RandomRows_20percent
  Repetición: 1


  Imputando variables para Rep 1: 100%|██████████| 7/7 [00:24<00:00,  3.55s/var]


    Archivo imputado por MOMENT guardado: Brasilia_RandomRows_20percent_rep1_seed1202032594_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Escenario RandomRows_20percent completado.

Procesando Escenario: BlockRows_12h_20percent
  Repetición: 1


  Imputando variables para Rep 1: 100%|██████████| 7/7 [00:26<00:00,  3.76s/var]


    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_12h_20percent_rep1_seed2510790048_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Escenario BlockRows_12h_20percent completado.

Procesando Escenario: BlockRows_24h_20percent
  Repetición: 1


...

    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_36h_20percent_rep1_seed1572501156_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Escenario BlockRows_36h_20percent completado.

Procesando Escenario: BlockRows_48h_20percent
  Repetición: 1


  Imputando variables para Rep 1: 100%|██████████| 7/7 [00:25<00:00,  3.69s/var]


    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_48h_20percent_rep1_seed107941404_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Escenario BlockRows_48h_20percent completado.

Procesando Escenario: BlockRows_60h_20percent
  Repetición: 1


  Imputando variables para Rep 1: 100%|██████████| 7/7 [00:25<00:00,  3.60s/var]

    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_60h_20percent_rep1_seed2179968212_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Escenario BlockRows_60h_20percent completado.

Generando resúmenes de métricas MOMENT...
  Métricas detalladas MOMENT guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE\metricas\ALL_metrics_detailed_MOMENT_PRE_Brasilia.csv
       City              Scenario  Repetition        Seed Imputation_Method  \
0  Brasilia  RandomRows_20percent           1  1202032594        MOMENT_PRE   
1  Brasilia  RandomRows_20percent           1  1202032594        MOMENT_PRE   
2  Brasilia  RandomRows_20percent           1  1202032594        MOMENT_PRE   
3  Brasilia  RandomRows_20percent           1  1202032594        MOMENT_PRE   
4  Brasilia  RandomRows_20percent           1  1202032594        MOMENT_PRE   

  Variable        MAE       RMSE  NRMAE_range  NRMSE_range  
0       CO  65.829579  97.5873

MOMENT_FT WFV

In [15]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None # Placeholder

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"


# --- 0. Carga de Datos Originales (Ground Truth) ---
TARGET_CITY_FT = "Brasilia"
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_FT]
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        num_rows_example = 1000 # Increased for WFV
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado con más datos.")

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_FT in city_timezones:
    tz = city_timezones[TARGET_CITY_FT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_FT} localized to {tz} and time features extracted.")
    else: print(f"No data for {TARGET_CITY_FT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_FT}.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date'])
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
if df_air_quality_ground_truth_localized.empty: print(f"Error: Ground truth for {TARGET_CITY_FT} is empty. Exiting."); exit()

# --- Parámetros Configurables para Fine-Tuning ---
# CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
# ALL_TARGET_COLS_FT = CONTAMINANT_COLS_FT + AQI_COL_FT
ALL_TARGET_COLS_FT = AQI_COL_FT
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 32
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"

# WFV Parameters for Fine-tuning
WFV_FT_INITIAL_TRAIN_RATIO = 0.2 # Start with 50% of data for the first FT fold
WFV_FT_VALIDATION_RATIO = 0.1  # Use 15% for validation in each FT fold
WFV_FT_STEP_RATIO = 0.1       # Expand training data by 15% in each subsequent FT fold
MAX_WFV_FT_FOLDS = 5           # Maximum number of WFV folds for fine-tuning

FINETUNE_EPOCHS_PER_FOLD = 100 # Epochs for fine-tuning within each WFV fold
FINETUNE_STEP_DATALOADER = 512 # Step for creating sequences within a fold's dataset
BATCH_SIZE_FINETUNE = 32
LEARNING_RATE_FINETUNE = 1e-1
PATIENCE_EARLY_STOPPING_FT = 16 # Early stopping patience for fine-tuning
LR_PATIENCE_FT = 6             # Patience for ReduceLROnPlateau
MIN_DELTA_EARLY_STOPPING_FT = 1e-5

N_REPETITIONS_FT_EVAL = 5
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_WFV_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_wfv')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)

# --- Dataset Personalizado para MOMENT ---
class MomentDataset(Dataset): # Renamed for clarity
    def __init__(self, data_array_2d, observed_mask_2d, seq_len, step):
        self.data_array = data_array_2d; self.observed_mask = observed_mask_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array) == 0: self.n_samples, self.indices = 0, []
        elif len(self.data_array) < self.seq_len : self.n_samples, self.indices = 1, [0]
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        if self.n_samples == 0: raise IndexError("Dataset is empty")
        actual_start_index = self.indices[idx]
        seq_data_values = self.data_array[actual_start_index : actual_start_index + self.seq_len]
        seq_observed_status = self.observed_mask[actual_start_index : actual_start_index + self.seq_len]
        if len(seq_data_values) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_values)
            seq_data_values = np.pad(seq_data_values, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_observed_status = np.pad(seq_observed_status, ((0, pad_width), (0,0)), 'constant', constant_values=False)
        timeseries_out = torch.tensor(seq_data_values, dtype=torch.float32).transpose(0, 1)
        input_mask_per_channel_tensor = torch.tensor(seq_observed_status, dtype=torch.bool)
        input_mask_out = input_mask_per_channel_tensor.all(dim=1)
        return {'timeseries': timeseries_out, 'input_mask': input_mask_out}

# --- Función de Evaluación del Modelo en Set de Validación ---
def evaluate_model_on_validation_set(model, val_loader, device, criterion):
    model.eval()
    total_val_loss = 0
    batch_count = 0
    with torch.no_grad():
        for batch_data in val_loader:
            timeseries_batch = batch_data['timeseries'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            try:
                outputs = model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                loss = criterion(outputs.reconstruction, timeseries_batch)
                if not (torch.isnan(loss) or torch.isinf(loss)):
                    total_val_loss += loss.item()
                    batch_count += 1
            except Exception as e_eval:
                print(f"    Error durante evaluación de validación: {e_eval}")
                continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

# --- Función de Entrenamiento (Fine-Tuning) con WFV Interno ---
def train_moment_model_finetune_wfv(
    model_template, # MOMENTPipeline instance (pre-loaded)
    train_data_scaled_var_fold, # Scaled training data for THIS FOLD (numpy array)
    val_data_scaled_var_fold,   # Scaled validation data for THIS FOLD (numpy array)
    device, epochs_per_fold, variable_name, fold_num,
    model_fold_save_path # Path to save the best model for THIS FOLD
    ):

    # Each fold starts with a fresh copy of the pre-trained model state
    # This ensures folds are independent in terms of fine-tuning from scratch (the pre-trained base)
    current_fold_model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    current_fold_model.init()
    current_fold_model = current_fold_model.to(device)
    # If you wanted to continue fine-tuning from previous fold's best, you'd load weights here.
    # For independent WFV folds on top of pre-trained, this is correct.

    current_fold_model.train()
    optimizer = torch.optim.Adam(current_fold_model.parameters(), lr=LEARNING_RATE_FINETUNE)
    criterion = nn.MSELoss()
    # Scheduler: Reduce LR every 3 epochs of NO improvement in val_loss
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=LR_PATIENCE_FT, verbose=True) # Changed patience

    # Create DataLoaders for this fold
    observed_mask_train_fold = np.ones_like(train_data_scaled_var_fold, dtype=bool)
    train_dataset_fold = MomentDataset(train_data_scaled_var_fold, observed_mask_train_fold, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER)
    train_loader_fold = DataLoader(train_dataset_fold, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)

    observed_mask_val_fold = np.ones_like(val_data_scaled_var_fold, dtype=bool)
    val_dataset_fold = MomentDataset(val_data_scaled_var_fold, observed_mask_val_fold, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER) # Use same step or full for val
    val_loader_fold = DataLoader(val_dataset_fold, batch_size=BATCH_SIZE_FINETUNE, shuffle=False, num_workers=0) # No shuffle for val

    best_fold_val_loss = float('inf')
    epochs_no_improve_fold = 0
    print(f"  Iniciando fine-tuning para WFV Fold {fold_num} ('{variable_name}', {epochs_per_fold} épocas máx)...")

    for epoch in range(epochs_per_fold):
        current_fold_model.train() # Ensure training mode
        total_epoch_train_loss = 0
        train_batch_count = 0
        for batch_data in tqdm(train_loader_fold, desc=f"  WFV Fold {fold_num} Epoch {epoch+1}/{epochs_per_fold} (Train)", leave=False):
            timeseries_batch = batch_data['timeseries'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            optimizer.zero_grad()
            try:
                outputs = current_fold_model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                loss = criterion(outputs.reconstruction, timeseries_batch)
            except Exception as e_train: print(f"    Error train batch: {e_train}"); continue
            if torch.isnan(loss) or torch.isinf(loss): print(f"    NaN/Inf train loss."); continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(current_fold_model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item()
            train_batch_count += 1
        
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        
        # Validation for this epoch
        avg_epoch_val_loss = evaluate_model_on_validation_set(current_fold_model, val_loader_fold, device, criterion)
        
        print(f"    WFV Fold {fold_num} Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        
        scheduler.step(avg_epoch_val_loss) # Scheduler checks val_loss
        
        if avg_epoch_val_loss < best_fold_val_loss - MIN_DELTA_EARLY_STOPPING_FT:
            best_fold_val_loss = avg_epoch_val_loss
            epochs_no_improve_fold = 0
            torch.save(current_fold_model.state_dict(), model_fold_save_path) # Save best model for this fold
            print(f"      Nuevo mejor modelo para Fold {fold_num} ('{variable_name}') guardado con Val Loss: {best_fold_val_loss:.6f}")
        else:
            epochs_no_improve_fold += 1
            
        if epochs_no_improve_fold >= PATIENCE_EARLY_STOPPING_FT: # Early stopping for THIS FOLD
            print(f"      Early stopping en Fold {fold_num} Epoch {epoch+1} para '{variable_name}'.")
            break
            
    print(f"  Fine-tuning para WFV Fold {fold_num} ('{variable_name}') completado. Mejor Val Loss del Fold: {best_fold_val_loss:.6f}")
    # Load the best model for this fold before returning (or it's already loaded if EarlyStopping hit at the best)
    if os.path.exists(model_fold_save_path):
        try: current_fold_model.load_state_dict(torch.load(model_fold_save_path, map_location=device))
        except Exception as e: print(f"  Error cargando mejor modelo del fold: {e}")

    return current_fold_model, best_fold_val_loss # Return the best model and its val_loss for this fold


# --- Función de Imputación con Modelo Fine-tuneado (Univariado) ---
def impute_univar_with_finetuned_model(
    df_univar_with_nans_scaled_zeros, original_nan_mask_univar,
    model_ft_var, device_str, scaler_var,
    seq_len_imp, step_imp, batch_size_imp
    ):
    model_ft_var.eval()
    imputation_dataset = MomentDataset(
        df_univar_with_nans_scaled_zeros.values, ~original_nan_mask_univar.values,
        seq_len_imp, step_imp
    )
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e_pred: print(f"    Error impute batch {batch_idx}: {e_pred}"); continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_imp + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_imp, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_series_unscaled = scaler_var.inverse_transform(imputed_full_series_scaled)
    df_final_imputed_unscaled = pd.DataFrame(
        scaler_var.inverse_transform(df_univar_with_nans_scaled_zeros.values),
        columns=df_univar_with_nans_scaled_zeros.columns,
        index=df_univar_with_nans_scaled_zeros.index
    )
    nan_mask_this_col_flat = original_nan_mask_univar.iloc[:,0].values
    col_name = df_univar_with_nans_scaled_zeros.columns[0]
    df_final_imputed_unscaled.loc[nan_mask_this_col_flat, col_name] = imputed_full_series_unscaled[nan_mask_this_col_flat, 0]
    return df_final_imputed_unscaled

# --- Función de Métricas ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values.astype(bool)
        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals, pred_col_vals, mask_col_flags = true_col_vals[:min_len], pred_col_vals[:min_len], mask_col_flags[:min_len]
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid); rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = 0
            if not original_col_full_series.empty: original_col_range = original_col_full_series.max() - original_col_full_series.min()
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan); nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
if MOMENTPipeline is None: print("MOMENTPipeline could not be imported. Exiting script."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns: df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print(f"Error: GT con NaNs para {TARGET_CITY_FT}."); exit()

# --- Fine-Tuning Phase con WFV ---
finetuned_models_overall_best_dict = {} # Stores the overall best fine-tuned model for each variable (across WFV folds)
scalers_for_vars_dict = {}
city_name_safe_ft_models = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_scalers_dir, exist_ok=True)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning con WFV para la variable: {var_to_finetune} =====")
    scaler_current_var = StandardScaler()
    data_for_scaler_fit = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    if len(data_for_scaler_fit) < 1: print(f"  Datos insuficientes para scaler de {var_to_finetune}."); continue
    scaler_current_var.fit(data_for_scaler_fit)
    scalers_for_vars_dict[var_to_finetune] = scaler_current_var
    joblib.dump(scaler_current_var, os.path.join(city_scalers_dir, f'scaler_ft_wfv_{sanitize_name(var_to_finetune)}.joblib'))
    
    # Get the full clean (scaled) series for this variable to perform WFV
    full_clean_data_var_scaled = scaler_current_var.transform(df_city_ground_truth_ft_reset[[var_to_finetune]].values)
    total_points_var_wfv = len(full_clean_data_var_scaled)

    # Calculate WFV split sizes based on ratios
    wfv_ft_init_train_len = int(total_points_var_wfv * WFV_FT_INITIAL_TRAIN_RATIO)
    wfv_ft_val_len = int(total_points_var_wfv * WFV_FT_VALIDATION_RATIO)
    wfv_ft_step_len = max(1, int(total_points_var_wfv * WFV_FT_STEP_RATIO))

    if total_points_var_wfv < wfv_ft_init_train_len + wfv_ft_val_len + SEQ_LEN_MODEL_FT: # Basic check
        print(f"  Datos insuficientes para WFV en {var_to_finetune} (Total: {total_points_var_wfv}). Saltando WFV.")
        # Optional: Fallback to a single fine-tuning run without WFV if desired
        # For now, we skip if WFV isn't feasible.
        finetuned_models_overall_best_dict[var_to_finetune] = None
        continue

    best_val_loss_across_all_folds = float('inf')
    path_to_overall_best_model_for_var = os.path.join(city_models_dir, f'moment_ft_wfv_overall_best_{sanitize_name(var_to_finetune)}.pth')
    
    current_train_end_idx = wfv_ft_init_train_len
    fold_count = 0

    while (current_train_end_idx + wfv_ft_val_len <= total_points_var_wfv) and (fold_count < MAX_WFV_FT_FOLDS):
        fold_count += 1
        print(f"\n--- WFV Fine-Tuning Fold {fold_count} para {var_to_finetune} ---")
        
        # Define data for this fold
        train_data_fold = full_clean_data_var_scaled[:current_train_end_idx]
        val_data_fold = full_clean_data_var_scaled[current_train_end_idx : current_train_end_idx + wfv_ft_val_len]
        
        print(f"  Tamaños - Train: {len(train_data_fold)}, Val: {len(val_data_fold)}")
        if len(train_data_fold) < SEQ_LEN_MODEL_FT or len(val_data_fold) < SEQ_LEN_MODEL_FT :
             print(f"  Datos insuficientes en Fold {fold_count} para secuencias. Train: {len(train_data_fold)}, Val: {len(val_data_fold)}, SeqLen: {SEQ_LEN_MODEL_FT}")
             current_train_end_idx += wfv_ft_step_len
             continue


        # Path to save the best model specifically for this fold (can be temporary)
        model_this_fold_save_path = os.path.join(city_models_dir, f'moment_ft_wfv_fold{fold_count}_{sanitize_name(var_to_finetune)}_temp.pth')

        # model_template is not used if we load fresh each time
        _, current_fold_best_val_loss = train_moment_model_finetune_wfv(
            None, # Pass None, function will load fresh
            train_data_fold, val_data_fold, device,
            FINETUNE_EPOCHS_PER_FOLD, var_to_finetune, fold_count,
            model_this_fold_save_path
        )
        
        if current_fold_best_val_loss < best_val_loss_across_all_folds:
            best_val_loss_across_all_folds = current_fold_best_val_loss
            print(f"  NUEVA MEJOR PÉRDIDA DE VALIDACIÓN GLOBAL para '{var_to_finetune}': {best_val_loss_across_all_folds:.6f} (desde Fold {fold_count})")
            if os.path.exists(model_this_fold_save_path):
                # Copy the best model of this fold to be the overall best for the variable
                import shutil
                shutil.copyfile(model_this_fold_save_path, path_to_overall_best_model_for_var)
                print(f"    Mejor modelo global para '{var_to_finetune}' actualizado y guardado en: {path_to_overall_best_model_for_var}")
        
        if os.path.exists(model_this_fold_save_path): # Clean up temp fold model
            os.remove(model_this_fold_save_path)
            
        current_train_end_idx += wfv_ft_step_len # Expand training window for next fold

    # After all WFV folds, load the overall best model for this variable
    if os.path.exists(path_to_overall_best_model_for_var):
        print(f"\nCargando el mejor modelo fine-tuneado general para '{var_to_finetune}' desde {path_to_overall_best_model_for_var}")
        overall_best_model_var = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
        overall_best_model_var.init()
        overall_best_model_var.load_state_dict(torch.load(path_to_overall_best_model_for_var, map_location=device))
        overall_best_model_var = overall_best_model_var.to(device)
        overall_best_model_var.eval() # Set to eval mode for imputation
        finetuned_models_overall_best_dict[var_to_finetune] = overall_best_model_var
    else:
        print(f"  ADVERTENCIA: No se encontró un modelo fine-tuneado general para '{var_to_finetune}'. La imputación podría fallar o usar fallback.")
        finetuned_models_overall_best_dict[var_to_finetune] = None

print("\n\n--- Fine-Tuning con WFV completado para todas las variables objetivo. ---")


# --- Imputation Phase using Overall Best Fine-tuned Models from WFV ---
city_name_safe_ft_impute = sanitize_name(TARGET_CITY_FT)
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft_impute)
os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)

for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = scenario_ft['name'].replace("%", "percent")
    scenario_name_for_file_part = scenario_ft['name'].replace("%", "percent")
    print(f"\nProcesando Escenario de Imputación: {scenario_name_for_dir}")
    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft_impute, scenario_name_for_dir)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_ft_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_ft): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_ft}."); continue

    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición de Imputación: {rep_ft}")
        file_pattern_to_find_imp = f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}_*_with_nans.csv"
        sim_files_in_dir_ft = sorted([f for f in os.listdir(sim_data_scenario_dir_ft) if f.startswith(f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_ft: print(f"    Archivo simulado no encontrado para '{file_pattern_to_find_imp}'."); continue
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = "unknown"
        try: current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError: print(f"    Advertencia: No se pudo extraer seed de: {sim_files_in_dir_ft[0]}")
        try:
            df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
            if 'Date' in df_city_with_nans_loaded_ft.columns: df_city_with_nans_loaded_ft['Date'] = pd.to_datetime(df_city_with_nans_loaded_ft['Date'])
        except Exception as e: print(f"    Error cargando {sim_file_path_to_load_ft}: {e}."); continue
        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()

        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var", leave=False):
            model_ft_to_use = finetuned_models_overall_best_dict.get(var_to_impute)
            scaler_var_to_use = scalers_for_vars_dict.get(var_to_impute)
            if model_ft_to_use is None or scaler_var_to_use is None:
                print(f"    Modelo/scaler no disponible para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
                continue
            df_univar_to_impute_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]].copy()
            original_nan_mask_univar_df = df_univar_to_impute_with_nans.isnull()
            df_univar_filled_for_scaling = df_univar_to_impute_with_nans.fillna(0)
            df_univar_scaled_zeros = pd.DataFrame(scaler_var_to_use.transform(df_univar_filled_for_scaling.values), columns=[var_to_impute], index=df_univar_to_impute_with_nans.index)
            df_imputed_univar_unscaled = impute_univar_with_finetuned_model(
                df_univar_scaled_zeros, original_nan_mask_univar_df, model_ft_to_use, device, scaler_var_to_use,
                SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT
            )
            if df_imputed_univar_unscaled is not None and var_to_impute in df_imputed_univar_unscaled.columns:
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_unscaled[var_to_impute]
            else:
                print(f"    Imputación falló para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
        
        df_to_save_final_ft = df_imputed_all_cols_rep_ft.copy()
        final_cols_for_saving_ft = ['Date', 'City'] + ALL_TARGET_COLS_FT
        for tf_col in TIME_FEATURES:
            if tf_col in df_city_ground_truth_ft_reset.columns:
                df_to_save_final_ft[tf_col] = df_city_ground_truth_ft_reset[tf_col].values[:len(df_to_save_final_ft)]
                if tf_col not in final_cols_for_saving_ft: final_cols_for_saving_ft.append(tf_col)
        df_to_save_final_ft = df_to_save_final_ft[[c for c in final_cols_for_saving_ft if c in df_to_save_final_ft.columns]]
        imputed_ft_filename = f"{city_name_safe_ft_impute}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT_WFV.csv"
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        df_to_save_final_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT-WFV guardado: {imputed_ft_filename}")
        
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        df_loaded_nans_reset_ft = df_city_with_nans_loaded_ft.reset_index(drop=True)
        for col_m_mask in ALL_TARGET_COLS_FT:
            if col_m_mask in df_loaded_nans_reset_ft.columns and col_m_mask in df_city_ground_truth_ft_reset.columns:
                len_min_mask_ft = min(len(df_loaded_nans_reset_ft), len(df_city_ground_truth_ft_reset))
                mask_series_metric_ft = (df_loaded_nans_reset_ft[col_m_mask].iloc[:len_min_mask_ft].isnull() & ~df_city_ground_truth_ft_reset[col_m_mask].iloc[:len_min_mask_ft].isnull())
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask].iloc[:len_min_mask_ft] = mask_series_metric_ft
        metrics_scores_ft_rep = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_to_save_final_ft.reset_index(drop=True), nan_mask_introduced_for_metrics_rep_ft, ALL_TARGET_COLS_FT)
        for var_name_metric_ft in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep_ft, 'Seed': current_seed_loaded_ft, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name_metric_ft, 'MAE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('MAE', np.nan), 'RMSE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('RMSE', np.nan), 'NRMAE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMAE_range', np.nan), 'NRMSE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMSE_range', np.nan)})
        print(f"    Métricas calculadas para MOMENT-FT-WFV, rep {rep_ft}")
    print(f"  Escenario de Imputación {scenario_name_for_dir} completado.")

# --- Guardado de Métricas Agregadas ---
if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT-WFV...")
    sufijo_test_ft = f"_{sanitize_name(TARGET_CITY_FT)}"
    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT-WFV guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())
    if not df_all_detailed_metrics_ft.empty:
        grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
        grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-WFV de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)
        df_contaminants_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(CONTAMINANT_COLS_FT)]
        if not df_contaminants_only_ft.empty:
            contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
            contaminants_combined_summary_ft.to_csv(contaminants_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-WFV contaminantes guardado: {contaminants_summary_path_ft}"); print(contaminants_combined_summary_ft)
        if AQI_COL_FT[0] in ALL_TARGET_COLS_FT:
            df_aqi_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'] == AQI_COL_FT[0]]
            if not df_aqi_only_ft.empty:
                aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
                aqi_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                aqi_combined_summary_ft.to_csv(aqi_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-WFV AQI guardado: {aqi_summary_path_ft}"); print(aqi_combined_summary_ft)
else: print("No se generaron métricas detalladas para MOMENT-FT-WFV.")
script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning con WFV e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Ground truth for Brasilia localized to America/Sao_Paulo and time features extracted.
Usando dispositivo: cpu

===== Iniciando Fine-Tuning con WFV para la variable: AQI =====

--- WFV Fine-Tuning Fold 1 para AQI ---
  Tamaños - Train: 1756, Val: 878
  Iniciando fine-tuning para WFV Fold 1 ('AQI', 100 épocas máx)...


    WFV Fold 1 Epoch 1: Train Loss: 0.196002, Val Loss: 6.986301 (LR: 1.00e-01)
      Nuevo mejor modelo para Fold 1 ('AQI') guardado con Val Loss: 6.986301


    WFV Fold 1 Epoch 2: Train Loss: 9.862823, Val Loss: 1.144508 (LR: 1.00e-01)
      Nuevo mejor modelo para Fold 1 ('AQI') guardado con Val Loss: 1.144508


    WFV Fold 1 Epoch 3: Train Loss: 1.601706, Val Loss: 1.755931 (LR: 1.00e-01)


    WFV Fold 1 Epoch 4: Train Loss: 2.656158, Val Loss: 3.343397 (LR: 1.00e-01)


...

    Archivo imputado por MOMENT-FT-WFV guardado: Brasilia_BlockRows_60h_20percent_rep1_seed2179968212_imputed_momentFT_WFV.csv
    Métricas calculadas para MOMENT-FT-WFV, rep 1
  Repetición de Imputación: 2


    Archivo imputado por MOMENT-FT-WFV guardado: Brasilia_BlockRows_60h_20percent_rep2_seed2179898580_imputed_momentFT_WFV.csv
    Métricas calculadas para MOMENT-FT-WFV, rep 2
  Repetición de Imputación: 3


    Archivo imputado por MOMENT-FT-WFV guardado: Brasilia_BlockRows_60h_20percent_rep3_seed2179895508_imputed_momentFT_WFV.csv
    Métricas calculadas para MOMENT-FT-WFV, rep 3
  Repetición de Imputación: 4


    Archivo imputado por MOMENT-FT-WFV guardado: Brasilia_BlockRows_60h_20percent_rep4_seed2179898580_imputed_momentFT_WFV.csv
    Métricas calculadas para MOMENT-FT-WFV, rep 4
  Repetición de Imputación: 5


    Archivo imputado por MOMENT-FT-WFV guardado: Brasilia_BlockRows_60h_20percent_rep5_seed2179896532_imputed_momentFT_WFV.csv
    Métricas calculadas para MOMENT-FT-WFV, rep 5
  Escenario de Imputación BlockRows_60h_20percent completado.

Generando resúmenes de métricas MOMENT-FT-WFV...
  Métricas detalladas MOMENT-FT-WFV guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_WFV_MOMENT_1_large\metricas\ALL_metrics_detailed_MOMENT_FT_WFV_MOMENT_1_large_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_20percent           1  1202032594   
1  Brasilia  RandomRows_20percent           2  1202079186   
2  Brasilia  RandomRows_20percent           3  1202059218   
3  Brasilia  RandomRows_20percent           4  1202005970   
4  Brasilia  RandomRows_20percent           5  1202063826   

              Imputation_Method Variable       MAE      RMSE  NRMAE_range  \
0  MOMENT_FT_WFV_MOMENT_1_large      AQI 

MOMENT_FT WFV INCREASE WEIGHTS

In [24]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil # Importado para shutil.copyfile
import glob # <<<--- IMPORTACIÓN AÑADIDA

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None # Placeholder

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"


# --- 0. Carga de Datos Originales (Ground Truth) ---
TARGET_CITY_FT = "Brasilia"
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_FT]
    _start_date = pd.to_datetime('2024-01-01')
    # Asumiendo que tu dataset original tiene ~8780 filas para Brasilia
    # num_rows_example = 8780
    # Para que el WFV tenga sentido con los ratios dados:
    # Initial train: 0.2 * N = al menos SEQ_LEN_MODEL_FT (512) => N > 2560
    # Val: 0.1 * N = al menos SEQ_LEN_MODEL_FT (512) => N > 5120
    # Step: 0.1 * N
    # Para 5 folds, y que el último train + val quepa:
    # (0.2 + 4*0.1 + 0.1) * N = 0.7 * N < N. Esto siempre se cumple.
    # Necesitamos que wfv_ft_init_train_len + wfv_ft_val_len sea manejable.
    # Si N = 8780:
    # Init train = 1756
    # Val = 878
    # Step = 878
    # Fold 1: Train 0-1755, Val 1756-2633
    # Fold 2: Train 0-2633, Val 2634-3511
    # ...
    # Fold 5: Train 0-(1756 + 4*878) = 0-5268, Val 5269-6146. Total usado 6147. Cabe en 8780.
    num_rows_example = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == 'Brasilia'].shape[0] if 'Brasilia' in df_air_quality_ground_truth_raw['City'].unique() else 8780 # Usar la longitud real si está disponible
    print(f"Usando num_rows_example: {num_rows_example}")

    for _city_ex in _cities_example:
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print(f"DataFrame de ejemplo creado con {num_rows_example} filas para {TARGET_CITY_FT}.")


df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_FT in city_timezones:
    tz = city_timezones[TARGET_CITY_FT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_FT} localized to {tz} and time features extracted. Shape: {df_air_quality_ground_truth_localized.shape}")
    else: print(f"No data for {TARGET_CITY_FT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_FT}.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date'])
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
if df_air_quality_ground_truth_localized.empty: print(f"Error: Ground truth for {TARGET_CITY_FT} is empty. Exiting."); exit()

# --- Parámetros Configurables para Fine-Tuning ---
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
# ALL_TARGET_COLS_FT = CONTAMINANT_COLS_FT + AQI_COL_FT
ALL_TARGET_COLS_FT = AQI_COL_FT # Para probar solo con AQI
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 32
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-base"

# WFV Parameters for Fine-tuning
WFV_FT_INITIAL_TRAIN_RATIO = 0.2
WFV_FT_VALIDATION_RATIO = 0.1
WFV_FT_STEP_RATIO = 0.1
MAX_WFV_FT_FOLDS = 5

FINETUNE_EPOCHS_PER_FOLD = 10 # Ajustado para que coincida con el log
FINETUNE_STEP_DATALOADER = 64 # Este es el 'step' para crear secuencias en el DataLoader
BATCH_SIZE_FINETUNE = 32 # Este es el tamaño del batch para el entrenamiento fine-tuneado
LEARNING_RATE_FINETUNE = 1e-3 # Coincide con el script; el log mostraba 1e-2, lo cual es extraño. Se usará 1e-4.
PATIENCE_EARLY_STOPPING_FT = 50 
LR_PATIENCE_FT = 20 
MIN_DELTA_EARLY_STOPPING_FT = 1e-5

N_REPETITIONS_FT_EVAL = 5
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_WFV_Inc_Base_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_wfv_incremental')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)

# --- Dataset Personalizado para MOMENT ---
class MomentDataset(Dataset):
    def __init__(self, data_array_2d, observed_mask_2d, seq_len, step):
        self.data_array = data_array_2d; self.observed_mask = observed_mask_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array) == 0: self.n_samples, self.indices = 0, []
        elif len(self.data_array) < self.seq_len : self.n_samples, self.indices = 1, [0]
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        if self.n_samples == 0: raise IndexError("Dataset is empty")
        actual_start_index = self.indices[idx]
        seq_data_values = self.data_array[actual_start_index : actual_start_index + self.seq_len]
        seq_observed_status = self.observed_mask[actual_start_index : actual_start_index + self.seq_len]
        if len(seq_data_values) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_values)
            seq_data_values = np.pad(seq_data_values, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_observed_status = np.pad(seq_observed_status, ((0, pad_width), (0,0)), 'constant', constant_values=False)
        timeseries_out = torch.tensor(seq_data_values, dtype=torch.float32).transpose(0, 1)
        input_mask_per_channel_tensor = torch.tensor(seq_observed_status, dtype=torch.bool)
        input_mask_out = input_mask_per_channel_tensor.all(dim=1)
        return {'timeseries': timeseries_out, 'input_mask': input_mask_out}

# --- Función de Evaluación del Modelo en Set de Validación ---
def evaluate_model_on_validation_set(model, val_loader, device, criterion):
    model.eval()
    total_val_loss = 0
    batch_count = 0
    with torch.no_grad():
        for batch_data in val_loader:
            timeseries_batch = batch_data['timeseries'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            try:
                outputs = model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                loss = criterion(outputs.reconstruction, timeseries_batch)
                if not (torch.isnan(loss) or torch.isinf(loss)):
                    total_val_loss += loss.item()
                    batch_count += 1
            except Exception as e_eval:
                print(f"    Error durante evaluación de validación: {e_eval}")
                continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

# --- Función de Entrenamiento (Fine-Tuning) con WFV Interno e Incremental ---
def train_moment_model_finetune_wfv_incremental(
    train_data_scaled_var_fold,
    val_data_scaled_var_fold,
    device, epochs_per_fold, variable_name, fold_num,
    model_fold_save_path,
    previous_fold_best_weights_path=None
    ):
    current_fold_model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    current_fold_model.init()
    current_fold_model = current_fold_model.to(device)
    if previous_fold_best_weights_path and os.path.exists(previous_fold_best_weights_path):
        try:
            current_fold_model.load_state_dict(torch.load(previous_fold_best_weights_path, map_location=device))
            print(f"  Fold {fold_num}: Pesos cargados desde el fold anterior: {previous_fold_best_weights_path}")
        except Exception as e_load_prev:
            print(f"  Fold {fold_num}: Error cargando pesos del fold anterior ({e_load_prev}). Iniciando desde pre-entrenado base.")
    else:
        if fold_num > 1: print(f"  Fold {fold_num}: No se encontraron pesos del fold anterior o es el primer fold. Iniciando desde pre-entrenado base.")
    current_fold_model.train()
    optimizer = torch.optim.Adam(current_fold_model.parameters(), lr=LEARNING_RATE_FINETUNE)
    criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=LR_PATIENCE_FT, verbose=True)
    observed_mask_train_fold = np.ones_like(train_data_scaled_var_fold, dtype=bool)
    train_dataset_fold = MomentDataset(train_data_scaled_var_fold, observed_mask_train_fold, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER)
    train_loader_fold = DataLoader(train_dataset_fold, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)
    observed_mask_val_fold = np.ones_like(val_data_scaled_var_fold, dtype=bool)
    val_dataset_fold = MomentDataset(val_data_scaled_var_fold, observed_mask_val_fold, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER)
    val_loader_fold = DataLoader(val_dataset_fold, batch_size=BATCH_SIZE_FINETUNE, shuffle=False, num_workers=0)
    best_fold_val_loss = float('inf')
    epochs_no_improve_fold = 0
    print(f"  Iniciando fine-tuning para WFV Fold {fold_num} ('{variable_name}', {epochs_per_fold} épocas máx)...")
    for epoch in range(epochs_per_fold):
        current_fold_model.train()
        total_epoch_train_loss = 0
        train_batch_count = 0
        for batch_data in tqdm(train_loader_fold, desc=f"  WFV Fold {fold_num} Epoch {epoch+1}/{epochs_per_fold} (Train)", leave=False):
            timeseries_batch = batch_data['timeseries'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            optimizer.zero_grad()
            try:
                outputs = current_fold_model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                loss = criterion(outputs.reconstruction, timeseries_batch)
            except Exception as e_train: print(f"    Error train batch: {e_train}"); continue
            if torch.isnan(loss) or torch.isinf(loss): print(f"    NaN/Inf train loss."); continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(current_fold_model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item()
            train_batch_count += 1
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        avg_epoch_val_loss = evaluate_model_on_validation_set(current_fold_model, val_loader_fold, device, criterion)
        print(f"    WFV Fold {fold_num} Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_fold_val_loss - MIN_DELTA_EARLY_STOPPING_FT:
            best_fold_val_loss = avg_epoch_val_loss
            epochs_no_improve_fold = 0
            torch.save(current_fold_model.state_dict(), model_fold_save_path)
            print(f"      Nuevo mejor modelo para Fold {fold_num} ('{variable_name}') guardado con Val Loss: {best_fold_val_loss:.6f}")
        else: epochs_no_improve_fold += 1
        if epochs_no_improve_fold >= PATIENCE_EARLY_STOPPING_FT: print(f"      Early stopping en Fold {fold_num} Epoch {epoch+1} para '{variable_name}'."); break
    print(f"  Fine-tuning para WFV Fold {fold_num} ('{variable_name}') completado. Mejor Val Loss del Fold: {best_fold_val_loss:.6f}")
    if os.path.exists(model_fold_save_path):
        try: current_fold_model.load_state_dict(torch.load(model_fold_save_path, map_location=device))
        except Exception as e: print(f"  Error cargando mejor modelo del fold: {e}")
    return current_fold_model, best_fold_val_loss

# --- Función de Imputación con Modelo Fine-tuneado (Univariado) ---
def impute_univar_with_finetuned_model(
    df_univar_with_nans_scaled_zeros, original_nan_mask_univar,
    model_ft_var, device_str, scaler_var,
    seq_len_imp, step_imp, batch_size_imp
    ):
    model_ft_var.eval()
    imputation_dataset = MomentDataset(df_univar_with_nans_scaled_zeros.values, ~original_nan_mask_univar.values, seq_len_imp, step_imp)
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e_pred: print(f"    Error impute batch {batch_idx}: {e_pred}"); continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_imp + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_imp, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_series_unscaled = scaler_var.inverse_transform(imputed_full_series_scaled)
    df_final_imputed_unscaled = pd.DataFrame(scaler_var.inverse_transform(df_univar_with_nans_scaled_zeros.values), columns=df_univar_with_nans_scaled_zeros.columns, index=df_univar_with_nans_scaled_zeros.index)
    nan_mask_this_col_flat = original_nan_mask_univar.iloc[:,0].values
    col_name = df_univar_with_nans_scaled_zeros.columns[0]
    df_final_imputed_unscaled.loc[nan_mask_this_col_flat, col_name] = imputed_full_series_unscaled[nan_mask_this_col_flat, 0]
    return df_final_imputed_unscaled

# --- Función de Métricas ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values.astype(bool)
        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals, pred_col_vals, mask_col_flags = true_col_vals[:min_len], pred_col_vals[:min_len], mask_col_flags[:min_len]
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid); rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = 0
            if not original_col_full_series.empty: original_col_range = original_col_full_series.max() - original_col_full_series.min()
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan); nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
if MOMENTPipeline is None: print("MOMENTPipeline could not be imported. Exiting script."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns: df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print(f"Error: GT con NaNs para {TARGET_CITY_FT}."); exit()

finetuned_models_overall_best_dict = {}
scalers_for_vars_dict = {}
city_name_safe_ft_models = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_scalers_dir, exist_ok=True)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning con WFV Incremental para la variable: {var_to_finetune} =====")
    scaler_current_var = StandardScaler()
    data_for_scaler_fit = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    if len(data_for_scaler_fit) < 1: print(f"  Datos insuficientes para scaler de {var_to_finetune}."); continue
    scaler_current_var.fit(data_for_scaler_fit)
    scalers_for_vars_dict[var_to_finetune] = scaler_current_var
    joblib.dump(scaler_current_var, os.path.join(city_scalers_dir, f'scaler_ft_wfv_inc_{sanitize_name(var_to_finetune)}.joblib'))
    
    full_clean_data_var_scaled = scaler_current_var.transform(df_city_ground_truth_ft_reset[[var_to_finetune]].values)
    total_points_var_wfv = len(full_clean_data_var_scaled)
    wfv_ft_init_train_len = int(total_points_var_wfv * WFV_FT_INITIAL_TRAIN_RATIO)
    wfv_ft_val_len = int(total_points_var_wfv * WFV_FT_VALIDATION_RATIO)
    wfv_ft_step_len = max(1, int(total_points_var_wfv * WFV_FT_STEP_RATIO))

    if total_points_var_wfv < wfv_ft_init_train_len + wfv_ft_val_len + SEQ_LEN_MODEL_FT:
        print(f"  Datos insuficientes para WFV en {var_to_finetune}. Saltando WFV."); finetuned_models_overall_best_dict[var_to_finetune] = None; continue

    best_val_loss_across_all_folds = float('inf')
    path_to_overall_best_model_for_var = os.path.join(city_models_dir, f'moment_ft_wfv_inc_overall_best_{sanitize_name(var_to_finetune)}.pth')
    previous_fold_best_weights_path = None 
    
    current_train_end_idx = wfv_ft_init_train_len
    fold_count = 0

    while (current_train_end_idx + wfv_ft_val_len <= total_points_var_wfv) and (fold_count < MAX_WFV_FT_FOLDS):
        fold_count += 1
        print(f"\n--- WFV Fine-Tuning Fold {fold_count} para {var_to_finetune} ---")
        train_data_fold = full_clean_data_var_scaled[:current_train_end_idx]
        val_data_fold = full_clean_data_var_scaled[current_train_end_idx : current_train_end_idx + wfv_ft_val_len]
        print(f"  Tamaños - Train: {len(train_data_fold)}, Val: {len(val_data_fold)}")
        if len(train_data_fold) < SEQ_LEN_MODEL_FT or len(val_data_fold) < SEQ_LEN_MODEL_FT:
             print(f"  Datos insuficientes en Fold {fold_count} para crear secuencias (Train: {len(train_data_fold)}, Val: {len(val_data_fold)}). Avanzando."); 
             current_train_end_idx += wfv_ft_step_len; continue

        model_this_fold_temp_save_path = os.path.join(city_models_dir, f'moment_ft_wfv_inc_fold{fold_count}_{sanitize_name(var_to_finetune)}_temp.pth')

        _, current_fold_best_val_loss = train_moment_model_finetune_wfv_incremental(
            train_data_fold, val_data_fold, device,
            FINETUNE_EPOCHS_PER_FOLD, var_to_finetune, fold_count,
            model_this_fold_temp_save_path, 
            previous_fold_best_weights_path
        )
        
        if os.path.exists(model_this_fold_temp_save_path):
            # This temp path now holds the best model from the current fold.
            # It becomes the candidate for the next fold's starting point.
            # And also a candidate for the overall best model.
            path_for_next_fold_init = model_this_fold_temp_save_path
        else: 
            print(f"    Advertencia: No se guardó ningún modelo para el Fold {fold_count}. El siguiente fold (si existe) comenzará desde el pre-entrenado base (o el último 'overall best' si esa was the logic).")
            # If current fold failed, next fold should start from scratch or from the previous *overall* best
            path_for_next_fold_init = path_to_overall_best_model_for_var if os.path.exists(path_to_overall_best_model_for_var) else None


        if current_fold_best_val_loss < best_val_loss_across_all_folds:
            best_val_loss_across_all_folds = current_fold_best_val_loss
            print(f"  NUEVA MEJOR PÉRDIDA DE VALIDACIÓN GLOBAL para '{var_to_finetune}': {best_val_loss_across_all_folds:.6f} (desde Fold {fold_count})")
            if os.path.exists(model_this_fold_temp_save_path): 
                 shutil.copyfile(model_this_fold_temp_save_path, path_to_overall_best_model_for_var)
                 print(f"    Mejor modelo global para '{var_to_finetune}' actualizado y guardado en: {path_to_overall_best_model_for_var}")
        
        # The weights for the *next* fold should be from the best model saved *in this fold*
        previous_fold_best_weights_path = path_for_next_fold_init # Corrected logic for next fold's input
            
        current_train_end_idx += wfv_ft_step_len
    
    if os.path.exists(path_to_overall_best_model_for_var):
        print(f"\nCargando el mejor modelo fine-tuneado general para '{var_to_finetune}' desde {path_to_overall_best_model_for_var}")
        overall_best_model_var = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
        overall_best_model_var.init()
        overall_best_model_var.load_state_dict(torch.load(path_to_overall_best_model_for_var, map_location=device))
        overall_best_model_var = overall_best_model_var.to(device)
        overall_best_model_var.eval()
        finetuned_models_overall_best_dict[var_to_finetune] = overall_best_model_var
    else:
        print(f"  ADVERTENCIA: No se encontró un modelo fine-tuneado general para '{var_to_finetune}'. La imputación podría usar fallback.")
        finetuned_models_overall_best_dict[var_to_finetune] = None
    
    for f_cleanup in glob.glob(os.path.join(city_models_dir, f'moment_ft_wfv_inc_fold*_{sanitize_name(var_to_finetune)}_temp.pth')):
        if not os.path.exists(path_to_overall_best_model_for_var) or not os.path.samefile(f_cleanup, path_to_overall_best_model_for_var):
            try:
                # print(f"  Limpiando modelo temporal: {f_cleanup}") # Descomentar para depuración
                os.remove(f_cleanup)
            except OSError as e:
                print(f"  Error limpiando archivo temporal {f_cleanup}: {e}")


print("\n\n--- Fine-Tuning con WFV Incremental completado para todas las variables objetivo. ---")

city_name_safe_ft_impute = sanitize_name(TARGET_CITY_FT)
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft_impute)
os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)
for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = scenario_ft['name'].replace("%", "percent")
    scenario_name_for_file_part = scenario_ft['name'].replace("%", "percent")
    print(f"\nProcesando Escenario de Imputación: {scenario_name_for_dir}")
    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft_impute, scenario_name_for_dir)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_ft_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_ft): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_ft}."); continue
    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición de Imputación: {rep_ft}")
        file_pattern_to_find_imp = f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}_*_with_nans.csv"
        sim_files_in_dir_ft = sorted([f for f in os.listdir(sim_data_scenario_dir_ft) if f.startswith(f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_ft: print(f"    Archivo simulado no encontrado para '{file_pattern_to_find_imp}'."); continue
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = "unknown"
        try: current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError: print(f"    Advertencia: No se pudo extraer seed de: {sim_files_in_dir_ft[0]}")
        try:
            df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
            if 'Date' in df_city_with_nans_loaded_ft.columns: df_city_with_nans_loaded_ft['Date'] = pd.to_datetime(df_city_with_nans_loaded_ft['Date'])
        except Exception as e: print(f"    Error cargando {sim_file_path_to_load_ft}: {e}."); continue
        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()
        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var", leave=False):
            model_ft_to_use = finetuned_models_overall_best_dict.get(var_to_impute)
            scaler_var_to_use = scalers_for_vars_dict.get(var_to_impute)
            if model_ft_to_use is None or scaler_var_to_use is None:
                print(f"    Modelo/scaler no disponible para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
                continue
            df_univar_to_impute_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]].copy()
            original_nan_mask_univar_df = df_univar_to_impute_with_nans.isnull()
            df_univar_filled_for_scaling = df_univar_to_impute_with_nans.fillna(0)
            df_univar_scaled_zeros = pd.DataFrame(scaler_var_to_use.transform(df_univar_filled_for_scaling.values), columns=[var_to_impute], index=df_univar_to_impute_with_nans.index)
            df_imputed_univar_unscaled = impute_univar_with_finetuned_model(df_univar_scaled_zeros, original_nan_mask_univar_df, model_ft_to_use, device, scaler_var_to_use, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT)
            if df_imputed_univar_unscaled is not None and var_to_impute in df_imputed_univar_unscaled.columns:
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_unscaled[var_to_impute]
            else:
                print(f"    Imputación falló para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
        
        df_to_save_final_ft = df_imputed_all_cols_rep_ft.copy()
        final_cols_for_saving_ft = ['Date', 'City'] + ALL_TARGET_COLS_FT
        for tf_col in TIME_FEATURES:
            if tf_col in df_city_ground_truth_ft_reset.columns:
                max_len = len(df_to_save_final_ft)
                df_to_save_final_ft[tf_col] = df_city_ground_truth_ft_reset[tf_col].values[:max_len]
                if tf_col not in final_cols_for_saving_ft: final_cols_for_saving_ft.append(tf_col)
        
        df_to_save_final_ft = df_to_save_final_ft[[c for c in final_cols_for_saving_ft if c in df_to_save_final_ft.columns]]

        imputed_ft_filename = f"{city_name_safe_ft_impute}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT_WFV_Inc.csv"
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        df_to_save_final_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT-WFV-Inc guardado: {imputed_ft_filename}")
        
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        df_loaded_nans_reset_ft = df_city_with_nans_loaded_ft.reset_index(drop=True)
        for col_m_mask in ALL_TARGET_COLS_FT:
            if col_m_mask in df_loaded_nans_reset_ft.columns and col_m_mask in df_city_ground_truth_ft_reset.columns:
                len_min_mask_ft = min(len(df_loaded_nans_reset_ft), len(df_city_ground_truth_ft_reset))
                mask_series_metric_ft = (df_loaded_nans_reset_ft[col_m_mask].iloc[:len_min_mask_ft].isnull() & ~df_city_ground_truth_ft_reset[col_m_mask].iloc[:len_min_mask_ft].isnull())
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask].iloc[:len_min_mask_ft] = mask_series_metric_ft
        metrics_scores_ft_rep = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_to_save_final_ft.reset_index(drop=True), nan_mask_introduced_for_metrics_rep_ft, ALL_TARGET_COLS_FT)
        for var_name_metric_ft in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep_ft, 'Seed': current_seed_loaded_ft, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name_metric_ft, 'MAE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('MAE', np.nan), 'RMSE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('RMSE', np.nan), 'NRMAE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMAE_range', np.nan), 'NRMSE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMSE_range', np.nan)})
        print(f"    Métricas calculadas para MOMENT-FT-WFV-Inc, rep {rep_ft}")
    print(f"  Escenario de Imputación {scenario_name_for_dir} completado.")

if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT-WFV-Inc...")
    sufijo_test_ft = f"_{sanitize_name(TARGET_CITY_FT)}"
    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT-WFV-Inc guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())
    if not df_all_detailed_metrics_ft.empty:
        grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
        grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-WFV-Inc de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)
        
        current_contaminant_cols = [col for col in ALL_TARGET_COLS_FT if col in CONTAMINANT_COLS_FT] if isinstance(CONTAMINANT_COLS_FT, list) else ([CONTAMINANT_COLS_FT] if isinstance(CONTAMINANT_COLS_FT, str) else [])
        current_aqi_cols = [col for col in ALL_TARGET_COLS_FT if col in AQI_COL_FT] if isinstance(AQI_COL_FT, list) else ([AQI_COL_FT] if isinstance(AQI_COL_FT, str) else [])

        if current_contaminant_cols:
            df_contaminants_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(current_contaminant_cols)]
            if not df_contaminants_only_ft.empty:
                contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
                contaminants_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                contaminants_combined_summary_ft.to_csv(contaminants_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-WFV-Inc contaminantes guardado: {contaminants_summary_path_ft}"); print(contaminants_combined_summary_ft)
        
        if current_aqi_cols:
            df_aqi_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(current_aqi_cols)]
            if not df_aqi_only_ft.empty:
                aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
                aqi_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                aqi_combined_summary_ft.to_csv(aqi_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-WFV-Inc AQI guardado: {aqi_summary_path_ft}"); print(aqi_combined_summary_ft)
else: print("No se generaron métricas detalladas para MOMENT-FT-WFV-Inc.")
script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning con WFV Incremental e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Ground truth for Brasilia localized to America/Sao_Paulo and time features extracted. Shape: (8784, 12)
Usando dispositivo: cpu

===== Iniciando Fine-Tuning con WFV Incremental para la variable: AQI =====

--- WFV Fine-Tuning Fold 1 para AQI ---
  Tamaños - Train: 1756, Val: 878
  Iniciando fine-tuning para WFV Fold 1 ('AQI', 10 épocas máx)...


    WFV Fold 1 Epoch 1: Train Loss: 0.827678, Val Loss: 0.485423 (LR: 1.00e-03)
      Nuevo mejor modelo para Fold 1 ('AQI') guardado con Val Loss: 0.485423


    WFV Fold 1 Epoch 2: Train Loss: 0.763302, Val Loss: 0.443827 (LR: 1.00e-03)
      Nuevo mejor modelo para Fold 1 ('AQI') guardado con Val Loss: 0.443827


    WFV Fold 1 Epoch 3: Train Loss: 0.696514, Val Loss: 0.404813 (LR: 1.00e-03)
      Nuevo mejor modelo para Fold 1 ('AQI') guardado con Val Loss: 0.404813


    WFV Fold 1 Epoch 4: Train Loss: 0.641476, Val Loss: 0.368396 (LR: 1.00e-03)
      Nuevo mejor modelo para Fold 1 ('AQI') guardado con Val Loss: 0.368396


...

    Archivo imputado por MOMENT-FT-WFV-Inc guardado: Brasilia_BlockRows_60h_20percent_rep1_seed2179968212_imputed_momentFT_WFV_Inc.csv
    Métricas calculadas para MOMENT-FT-WFV-Inc, rep 1
  Repetición de Imputación: 2


    Archivo imputado por MOMENT-FT-WFV-Inc guardado: Brasilia_BlockRows_60h_20percent_rep2_seed2179898580_imputed_momentFT_WFV_Inc.csv
    Métricas calculadas para MOMENT-FT-WFV-Inc, rep 2
  Repetición de Imputación: 3


    Archivo imputado por MOMENT-FT-WFV-Inc guardado: Brasilia_BlockRows_60h_20percent_rep3_seed2179895508_imputed_momentFT_WFV_Inc.csv
    Métricas calculadas para MOMENT-FT-WFV-Inc, rep 3
  Repetición de Imputación: 4


    Archivo imputado por MOMENT-FT-WFV-Inc guardado: Brasilia_BlockRows_60h_20percent_rep4_seed2179898580_imputed_momentFT_WFV_Inc.csv
    Métricas calculadas para MOMENT-FT-WFV-Inc, rep 4
  Repetición de Imputación: 5


    Archivo imputado por MOMENT-FT-WFV-Inc guardado: Brasilia_BlockRows_60h_20percent_rep5_seed2179896532_imputed_momentFT_WFV_Inc.csv
    Métricas calculadas para MOMENT-FT-WFV-Inc, rep 5
  Escenario de Imputación BlockRows_60h_20percent completado.

Generando resúmenes de métricas MOMENT-FT-WFV-Inc...
  Métricas detalladas MOMENT-FT-WFV-Inc guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_WFV_Inc_Base_MOMENT_1_base\metricas\ALL_metrics_detailed_MOMENT_FT_WFV_Inc_Base_MOMENT_1_base_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_20percent           1  1202032594   
1  Brasilia  RandomRows_20percent           2  1202079186   
2  Brasilia  RandomRows_20percent           3  1202059218   
3  Brasilia  RandomRows_20percent           4  1202005970   
4  Brasilia  RandomRows_20percent           5  1202063826   

                      Imputation_Method Variable       MAE       RMSE  \
0  MOMENT

MOMENT_FT tutorial - el mejor hasta ahora

In [24]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
    # from momentfm.common.masking import Masking # Ya no es estrictamente necesario para este nuevo FT
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None
    # Masking = None

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"


# --- 0. Carga de Datos Originales (Ground Truth) ---
TARGET_CITY_FT = "Brasilia"
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_FT]
    _start_date = pd.to_datetime('2024-01-01')
    num_rows_example = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == 'Brasilia'].shape[0] if not df_air_quality_ground_truth_raw.empty and 'Brasilia' in df_air_quality_ground_truth_raw['City'].unique() else 8780
    print(f"Usando num_rows_example: {num_rows_example}")
    for _city_ex in _cities_example:
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print(f"DataFrame de ejemplo creado con {num_rows_example} filas para {TARGET_CITY_FT}.")

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_FT in city_timezones:
    tz = city_timezones[TARGET_CITY_FT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_FT} localized to {tz} and time features extracted. Shape: {df_air_quality_ground_truth_localized.shape}")
    else: print(f"No data for {TARGET_CITY_FT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_FT}.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date'])
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
if df_air_quality_ground_truth_localized.empty: print(f"Error: Ground truth for {TARGET_CITY_FT} is empty. Exiting."); exit()

# --- Parámetros Configurables para Fine-Tuning ---
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = AQI_COL_FT # Solo AQI para fine-tuning
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 64
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"

# Fine-tuning Parameters (NUEVA Estrategia)
FINETUNE_EPOCHS = 5000 # Aumentar si es necesario
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 512
BATCH_SIZE_FINETUNE_NEW = 32
LEARNING_RATE_FINETUNE_NEW = 1e-4
SIMULATED_MISSING_RATIO_FOR_FT = 0.25 # Porcentaje de valores a enmascarar en cada secuencia de FT
PATIENCE_EARLY_STOPPING_NEW = 1250
LR_PATIENCE_NEW = 500
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 5
NAN_SIMULATION_SCENARIOS_FT = [ # Estos son para la evaluación final de imputación
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_SimImpute_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}' # Nuevo nombre
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute') # Nuevo dir
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)

# --- Dataset Personalizado para MOMENT (NUEVA Estrategia Fine-tuning) ---
class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_2d_scaled_clean, seq_len, step, missing_ratio):
        self.data_array = data_array_2d_scaled_clean
        self.seq_len = seq_len
        self.step = step
        self.missing_ratio = missing_ratio # Porcentaje de puntos a enmascarar en cada secuencia

        if len(self.data_array) == 0: self.n_samples, self.indices = 0, []
        elif len(self.data_array) < self.seq_len : self.n_samples, self.indices = 1, [0]
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len: self.indices, self.n_samples = [0],1

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        if self.n_samples == 0: raise IndexError("Dataset is empty")
        actual_start_index = self.indices[idx]
        seq_data_clean_numpy = self.data_array[actual_start_index : actual_start_index + self.seq_len].copy() # (seq_len, n_channels)
        
        if len(seq_data_clean_numpy) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_clean_numpy)
            seq_data_clean_numpy = np.pad(seq_data_clean_numpy, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)

        # Simular NaNs para esta secuencia (solo en el canal objetivo si es univariado)
        # Para univariado (n_channels=1), enmascaramos directamente en seq_data_clean_numpy
        n_points_to_mask = int(self.seq_len * self.missing_ratio)
        mask_indices = np.random.choice(self.seq_len, n_points_to_mask, replace=False)
        
        simulated_nan_mask_numpy = np.zeros_like(seq_data_clean_numpy, dtype=bool) # False por defecto
        simulated_nan_mask_numpy[mask_indices, :] = True # True donde simulamos NaN

        input_mask_for_moment = ~simulated_nan_mask_numpy.all(axis=1) # True si el timestep es observado (ningún canal es NaN simulado)

        # Crear x_enc (con 0s donde simulamos NaNs) y el target (la secuencia limpia)
        x_enc_numpy = seq_data_clean_numpy.copy()
        x_enc_numpy[simulated_nan_mask_numpy] = 0.0 # Rellenar NaNs simulados con 0 para la entrada del modelo

        return {
            'x_enc': torch.tensor(x_enc_numpy, dtype=torch.float32).transpose(0, 1),         # (n_channels, seq_len)
            'input_mask': torch.tensor(input_mask_for_moment, dtype=torch.bool),             # (seq_len,)
            'target_clean': torch.tensor(seq_data_clean_numpy, dtype=torch.float32).transpose(0,1), # (n_channels, seq_len)
            'simulated_nan_mask': torch.tensor(simulated_nan_mask_numpy, dtype=torch.bool).transpose(0,1) # (n_channels, seq_len)
        }

# --- Dataset Personalizado para Imputación (igual que antes) ---
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data_array_scaled_zeros = data_array_2d_multivar_scaled_zeros
        self.nan_mask_original = nan_mask_original_bool_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array_scaled_zeros) < self.seq_len : self.n_samples, self.indices = 1, [0]
        elif len(self.data_array_scaled_zeros) == 0: self.n_samples, self.indices = 0, []
        else:
            self.indices = list(range(0, len(self.data_array_scaled_zeros) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array_scaled_zeros) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array_scaled_zeros) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        actual_start_index = self.indices[idx]
        seq_data_scaled_zeros_orig_shape = self.data_array_scaled_zeros[actual_start_index : actual_start_index + self.seq_len]
        seq_observed_mask_bool_per_channel = ~self.nan_mask_original[actual_start_index : actual_start_index + self.seq_len]
        if len(seq_data_scaled_zeros_orig_shape) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_scaled_zeros_orig_shape)
            seq_data_scaled_zeros_orig_shape = np.pad(seq_data_scaled_zeros_orig_shape, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_observed_mask_bool_per_channel = np.pad(seq_observed_mask_bool_per_channel, ((0, pad_width), (0,0)), 'constant', constant_values=False)
        timeseries_out = torch.tensor(seq_data_scaled_zeros_orig_shape, dtype=torch.float32).transpose(0, 1)
        input_mask_out = torch.tensor(seq_observed_mask_bool_per_channel, dtype=torch.bool).all(dim=1)
        return {'timeseries': timeseries_out, 'input_mask': input_mask_out }

# --- Función de Evaluación del Modelo en Set de Validación (NUEVA Estrategia) ---
def evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion):
    model.eval()
    total_val_loss = 0
    batch_count = 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch = batch_data['x_enc'].to(device) # Ya contiene 0s en NaNs simulados
            input_mask_batch = batch_data['input_mask'].to(device) # Máscara de observados (después de NaNs simulados)
            target_clean_batch = batch_data['target_clean'].to(device) # Secuencia limpia original
            simulated_nan_mask_batch = batch_data['simulated_nan_mask'].to(device) # True donde se simularon NaNs
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch) # Sin el argumento 'mask'
                reconstruction_all = outputs.reconstruction
                
                # Calcular pérdida solo en las posiciones donde simulamos NaNs
                # reconstruction_loss_at_nans es (batch*n_channels, 1, seq_len)
                reconstruction_loss_at_nans = criterion(reconstruction_all, target_clean_batch) 
                
                # Asegurar que simulated_nan_mask_batch tiene la misma forma para el producto elemento a elemento
                # Si n_channels > 1, y x_enc_batch fue (B*C, 1, L), entonces simulated_nan_mask_batch es (B*C, 1, L)
                # Si es univariado, n_channels=1, x_enc_batch es (B, 1, L), simulated_nan_mask_batch es (B,1,L)
                if reconstruction_loss_at_nans.shape != simulated_nan_mask_batch.shape: # Necesario si x_enc_batch es (B,C,L) y mask (B,C,L)
                     simulated_nan_mask_batch = simulated_nan_mask_batch.reshape(reconstruction_loss_at_nans.shape)

                masked_loss_components = reconstruction_loss_at_nans * simulated_nan_mask_batch.float() # Ponderar por máscara
                loss = masked_loss_components.nansum() / (simulated_nan_mask_batch.nansum() + 1e-9)

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    total_val_loss += loss.item()
                    batch_count += 1
            except Exception as e_eval: print(f"    Error en validación (strat. sim impute): {e_eval}"); continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

# --- Función de Entrenamiento (Fine-Tuning) con NUEVA Estrategia ---
def train_moment_model_finetune_simulated_imputation(
    model, train_loader, val_loader, device, epochs, variable_name, model_save_path,
    learning_rate, lr_patience, es_patience, min_delta_es
    ):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss(reduction='none') # Para calcular pérdida solo en posiciones enmascaradas
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.8, patience=lr_patience, verbose=True)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    print(f"  Iniciando fine-tuning (estrategia imputación simulada) para '{variable_name}' ({epochs} épocas máx)...")

    for epoch in range(epochs):
        model.train()
        total_epoch_train_loss = 0; train_batch_count = 0
        pbar_train = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar_train:
            x_enc_batch = batch_data['x_enc'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            target_clean_batch = batch_data['target_clean'].to(device)
            simulated_nan_mask_batch = batch_data['simulated_nan_mask'].to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch) # Sin el argumento `mask`
                reconstruction_all = outputs.reconstruction
                reconstruction_loss_at_nans = criterion(reconstruction_all, target_clean_batch)
                
                if reconstruction_loss_at_nans.shape != simulated_nan_mask_batch.shape:
                     simulated_nan_mask_batch = simulated_nan_mask_batch.reshape(reconstruction_loss_at_nans.shape)
                
                masked_loss_components = reconstruction_loss_at_nans * simulated_nan_mask_batch.float()
                loss = masked_loss_components.nansum() / (simulated_nan_mask_batch.nansum() + 1e-9)
            except Exception as e_train: print(f"    Error train batch: {e_train}"); continue
            if torch.isnan(loss) or torch.isinf(loss) or (simulated_nan_mask_batch.nansum() < 1e-7): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item(); train_batch_count += 1
            pbar_train.set_postfix(loss=loss.item())
        
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        # Para la validación, usamos la misma estrategia de pérdida (solo en NaNs simulados)
        avg_epoch_val_loss = evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion)
        
        print(f"    Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_val_loss - min_delta_es:
            best_val_loss = avg_epoch_val_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"      Nuevo mejor modelo para '{variable_name}' (strat. sim impute) guardado con Val Loss: {best_val_loss:.6f}")
        else: epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1} para '{variable_name}' (strat. sim impute)."); break
    print(f"  Fine-tuning (strat. sim impute) para '{variable_name}' completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path):
        try: model.load_state_dict(torch.load(model_save_path, map_location=device))
        except Exception as e: print(f"  Error cargando mejor modelo (strat. sim impute): {e}")
    return model

# --- Función de Imputación (igual que antes) ---
def impute_univar_with_finetuned_model(
    df_univar_with_nans_scaled_zeros, original_nan_mask_univar,
    model_ft_var, device_str, scaler_var,
    seq_len_imp, step_imp, batch_size_imp
    ):
    model_ft_var.eval()
    imputation_dataset = MomentImputationDataset(
        df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values,
        seq_len_imp, step_imp
    )
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e_pred: print(f"    Error impute batch {batch_idx}: {e_pred}"); continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_imp + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_imp, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_series_unscaled = scaler_var.inverse_transform(imputed_full_series_scaled)
    df_final_imputed_unscaled = pd.DataFrame(scaler_var.inverse_transform(df_univar_with_nans_scaled_zeros.values), columns=df_univar_with_nans_scaled_zeros.columns, index=df_univar_with_nans_scaled_zeros.index)
    nan_mask_this_col_flat = original_nan_mask_univar.iloc[:,0].values
    col_name = df_univar_with_nans_scaled_zeros.columns[0]
    df_final_imputed_unscaled.loc[nan_mask_this_col_flat, col_name] = imputed_full_series_unscaled[nan_mask_this_col_flat, 0]
    return df_final_imputed_unscaled

# --- Función de Métricas (igual que antes) ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values.astype(bool)
        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals, pred_col_vals, mask_col_flags = true_col_vals[:min_len], pred_col_vals[:min_len], mask_col_flags[:min_len]
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid); rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = 0
            if not original_col_full_series.empty: original_col_range = original_col_full_series.max() - original_col_full_series.min()
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan); nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
if MOMENTPipeline is None: print("MOMENTPipeline o Masking no pudieron ser importadas. Exiting script."); exit() # Masking ya no es necesario aquí
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns: df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print(f"Error: GT con NaNs para {TARGET_CITY_FT} en columnas objetivo."); exit()

finetuned_models_sim_impute_dict = {} # Nuevo dict para esta estrategia
scalers_for_vars_sim_impute_dict = {} # Nuevo dict para esta estrategia
city_name_safe_ft_models = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_scalers_dir, exist_ok=True)

# mask_generator_ft ya no se usa para el entrenamiento, solo para evaluación si quisiéramos esa estrategia
# mask_generator_ft = Masking(mask_ratio=MASK_RATIO_FINETUNE_AUTHORS) # Comentado/Eliminado

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning (Estrategia Imputación Simulada) para la variable: {var_to_finetune} =====")
    scaler_current_var = StandardScaler()
    data_for_scaler_fit = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    if len(data_for_scaler_fit) < 1: print(f"  Datos insuficientes para scaler de {var_to_finetune}."); continue
    scaler_current_var.fit(data_for_scaler_fit)
    scalers_for_vars_sim_impute_dict[var_to_finetune] = scaler_current_var
    joblib.dump(scaler_current_var, os.path.join(city_scalers_dir, f'scaler_ft_sim_impute_{sanitize_name(var_to_finetune)}.joblib'))
    
    full_clean_data_var_scaled = scaler_current_var.transform(df_city_ground_truth_ft_reset[[var_to_finetune]].values)
    
    train_val_split_idx = int(len(full_clean_data_var_scaled) * FINETUNE_TRAIN_RATIO_NEW) # Usar ratio para nueva estrategia
    train_data_ft_var = full_clean_data_var_scaled[:train_val_split_idx]
    val_data_ft_var = full_clean_data_var_scaled[train_val_split_idx:]

    if len(train_data_ft_var) < SEQ_LEN_MODEL_FT or len(val_data_ft_var) < SEQ_LEN_MODEL_FT:
        print(f"  Datos insuficientes para split train/val en {var_to_finetune}. Saltando fine-tuning."); finetuned_models_sim_impute_dict[var_to_finetune] = None; continue

    # Usar MomentFTSimulatedImputationDataset
    train_dataset_sim_impute = MomentFTSimulatedImputationDataset(train_data_ft_var, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    train_loader_sim_impute = DataLoader(train_dataset_sim_impute, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0)
    val_dataset_sim_impute = MomentFTSimulatedImputationDataset(val_data_ft_var, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    val_loader_sim_impute = DataLoader(val_dataset_sim_impute, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False, num_workers=0)

    if len(train_loader_sim_impute) == 0 or len(val_loader_sim_impute) == 0 :
        print(f"  Train o Val loader vacío para {var_to_finetune}. Saltando fine-tuning."); finetuned_models_sim_impute_dict[var_to_finetune] = None; continue
        
    model_to_finetune_sim_impute = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model_to_finetune_sim_impute.init(); model_to_finetune_sim_impute = model_to_finetune_sim_impute.to(device)
    model_save_path_var_sim_impute = os.path.join(city_models_dir, f'moment_ft_sim_impute_best_{sanitize_name(var_to_finetune)}.pth')
    
    finetuned_model_var = train_moment_model_finetune_simulated_imputation(
        model_to_finetune_sim_impute, train_loader_sim_impute, val_loader_sim_impute, device,
        FINETUNE_EPOCHS, var_to_finetune, model_save_path_var_sim_impute,
        LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW
    )
    finetuned_models_sim_impute_dict[var_to_finetune] = finetuned_model_var
    if finetuned_model_var is not None:
        print(f"  Modelo para {var_to_finetune} (strat. sim impute) afinado y guardado/cargado.")
    else:
        print(f"  Fallo el fine-tuning (strat. sim impute) para {var_to_finetune}.")

print("\n\n--- Fine-Tuning (Estrategia Imputación Simulada) completado para todas las variables objetivo. ---")

# --- Imputation Phase ---
city_name_safe_ft_impute = sanitize_name(TARGET_CITY_FT)
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft_impute)
os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)
for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = scenario_ft['name'].replace("%", "percent")
    scenario_name_for_file_part = scenario_ft['name'].replace("%", "percent")
    print(f"\nProcesando Escenario de Imputación: {scenario_name_for_dir}")
    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft_impute, scenario_name_for_dir)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_ft_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_ft): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_ft}."); continue
    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición de Imputación: {rep_ft}")
        file_pattern_to_find_imp = f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}_*_with_nans.csv"
        sim_files_in_dir_ft = sorted([f for f in os.listdir(sim_data_scenario_dir_ft) if f.startswith(f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_ft: print(f"    Archivo simulado no encontrado para '{file_pattern_to_find_imp}'."); continue
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = "unknown"
        try: current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError: print(f"    Advertencia: No se pudo extraer seed de: {sim_files_in_dir_ft[0]}")
        try:
            df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
            if 'Date' in df_city_with_nans_loaded_ft.columns: df_city_with_nans_loaded_ft['Date'] = pd.to_datetime(df_city_with_nans_loaded_ft['Date'])
        except Exception as e: print(f"    Error cargando {sim_file_path_to_load_ft}: {e}."); continue
        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()
        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var", leave=False):
            model_ft_to_use = finetuned_models_sim_impute_dict.get(var_to_impute) # Usar el dict correcto
            scaler_var_to_use = scalers_for_vars_sim_impute_dict.get(var_to_impute) # Usar el dict correcto
            if model_ft_to_use is None or scaler_var_to_use is None:
                print(f"    Modelo/scaler (Estrat. Sim Impute) no disponible para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
                continue
            df_univar_to_impute_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]].copy()
            original_nan_mask_univar_df = df_univar_to_impute_with_nans.isnull()
            df_univar_filled_for_scaling = df_univar_to_impute_with_nans.fillna(0)
            df_univar_scaled_zeros = pd.DataFrame(scaler_var_to_use.transform(df_univar_filled_for_scaling.values), columns=[var_to_impute], index=df_univar_to_impute_with_nans.index)
            df_imputed_univar_unscaled = impute_univar_with_finetuned_model(df_univar_scaled_zeros, original_nan_mask_univar_df, model_ft_to_use, device, scaler_var_to_use, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT)
            if df_imputed_univar_unscaled is not None and var_to_impute in df_imputed_univar_unscaled.columns:
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_unscaled[var_to_impute]
            else:
                print(f"    Imputación falló para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
        df_to_save_final_ft = df_imputed_all_cols_rep_ft.copy()
        final_cols_for_saving_ft = ['Date', 'City'] + ALL_TARGET_COLS_FT
        for tf_col in TIME_FEATURES:
            if tf_col in df_city_ground_truth_ft_reset.columns:
                max_len = len(df_to_save_final_ft)
                df_to_save_final_ft[tf_col] = df_city_ground_truth_ft_reset[tf_col].values[:max_len]
                if tf_col not in final_cols_for_saving_ft: final_cols_for_saving_ft.append(tf_col)
        df_to_save_final_ft = df_to_save_final_ft[[c for c in final_cols_for_saving_ft if c in df_to_save_final_ft.columns]]
        imputed_ft_filename = f"{city_name_safe_ft_impute}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT_SimImpute.csv" # Nombre actualizado
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        df_to_save_final_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT-SimImpute guardado: {imputed_ft_filename}")
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        df_loaded_nans_reset_ft = df_city_with_nans_loaded_ft.reset_index(drop=True)
        for col_m_mask in ALL_TARGET_COLS_FT:
            if col_m_mask in df_loaded_nans_reset_ft.columns and col_m_mask in df_city_ground_truth_ft_reset.columns:
                len_min_mask_ft = min(len(df_loaded_nans_reset_ft), len(df_city_ground_truth_ft_reset))
                mask_series_metric_ft = (df_loaded_nans_reset_ft[col_m_mask].iloc[:len_min_mask_ft].isnull() & ~df_city_ground_truth_ft_reset[col_m_mask].iloc[:len_min_mask_ft].isnull())
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask].iloc[:len_min_mask_ft] = mask_series_metric_ft
        metrics_scores_ft_rep = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_to_save_final_ft.reset_index(drop=True), nan_mask_introduced_for_metrics_rep_ft, ALL_TARGET_COLS_FT)
        for var_name_metric_ft in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep_ft, 'Seed': current_seed_loaded_ft, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name_metric_ft, 'MAE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('MAE', np.nan), 'RMSE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('RMSE', np.nan), 'NRMAE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMAE_range', np.nan), 'NRMSE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMSE_range', np.nan)})
        print(f"    Métricas calculadas para MOMENT-FT-SimImpute, rep {rep_ft}")
    print(f"  Escenario de Imputación {scenario_name_for_dir} completado.")

if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT-SimImpute...") # Nombre actualizado
    sufijo_test_ft = f"_{sanitize_name(TARGET_CITY_FT)}"
    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT-SimImpute guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())
    if not df_all_detailed_metrics_ft.empty:
        grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
        grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-SimImpute de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)
        current_contaminant_cols_summary = [col for col in ALL_TARGET_COLS_FT if col in CONTAMINANT_COLS_FT] if isinstance(CONTAMINANT_COLS_FT, list) else ([CONTAMINANT_COLS_FT] if isinstance(CONTAMINANT_COLS_FT, str) else [])
        current_aqi_cols_summary = [col for col in ALL_TARGET_COLS_FT if col in AQI_COL_FT] if isinstance(AQI_COL_FT, list) else ([AQI_COL_FT] if isinstance(AQI_COL_FT, str) else [])
        if current_contaminant_cols_summary:
            df_contaminants_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(current_contaminant_cols_summary)]
            if not df_contaminants_only_ft.empty:
                contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
                contaminants_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                contaminants_combined_summary_ft.to_csv(contaminants_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-SimImpute contaminantes guardado: {contaminants_summary_path_ft}"); print(contaminants_combined_summary_ft)
        if current_aqi_cols_summary:
            df_aqi_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(current_aqi_cols_summary)]
            if not df_aqi_only_ft.empty:
                aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
                aqi_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                aqi_combined_summary_ft.to_csv(aqi_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-SimImpute AQI guardado: {aqi_summary_path_ft}"); print(aqi_combined_summary_ft)
else: print("No se generaron métricas detalladas para MOMENT-FT-SimImpute.")
script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning (Estrategia Imputación Simulada) e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Ground truth for Brasilia localized to America/Sao_Paulo and time features extracted. Shape: (8784, 12)
Usando dispositivo: cpu

===== Iniciando Fine-Tuning (Estrategia Imputación Simulada) para la variable: AQI =====
  Iniciando fine-tuning (estrategia imputación simulada) para 'AQI' (5000 épocas máx)...


    Epoch 1: Train Loss: 0.888715, Val Loss: 1.245021 (LR: 1.00e-04)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 1.245021


    Epoch 2: Train Loss: 1.110758, Val Loss: 1.212173 (LR: 1.00e-04)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 1.212173


    Epoch 3: Train Loss: 0.907643, Val Loss: 1.199014 (LR: 1.00e-04)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 1.199014


    Epoch 4: Train Loss: 0.984477, Val Loss: 1.189994 (LR: 1.00e-04)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 1.189994


...

    Archivo imputado por MOMENT-FT-SimImpute guardado: Brasilia_BlockRows_60h_20percent_rep1_seed2179968212_imputed_momentFT_SimImpute.csv
    Métricas calculadas para MOMENT-FT-SimImpute, rep 1
  Repetición de Imputación: 2


    Archivo imputado por MOMENT-FT-SimImpute guardado: Brasilia_BlockRows_60h_20percent_rep2_seed2179898580_imputed_momentFT_SimImpute.csv
    Métricas calculadas para MOMENT-FT-SimImpute, rep 2
  Repetición de Imputación: 3


    Archivo imputado por MOMENT-FT-SimImpute guardado: Brasilia_BlockRows_60h_20percent_rep3_seed2179895508_imputed_momentFT_SimImpute.csv
    Métricas calculadas para MOMENT-FT-SimImpute, rep 3
  Repetición de Imputación: 4


    Archivo imputado por MOMENT-FT-SimImpute guardado: Brasilia_BlockRows_60h_20percent_rep4_seed2179898580_imputed_momentFT_SimImpute.csv
    Métricas calculadas para MOMENT-FT-SimImpute, rep 4
  Repetición de Imputación: 5


    Archivo imputado por MOMENT-FT-SimImpute guardado: Brasilia_BlockRows_60h_20percent_rep5_seed2179896532_imputed_momentFT_SimImpute.csv
    Métricas calculadas para MOMENT-FT-SimImpute, rep 5
  Escenario de Imputación BlockRows_60h_20percent completado.

Generando resúmenes de métricas MOMENT-FT-SimImpute...
  Métricas detalladas MOMENT-FT-SimImpute guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_SimImpute_MOMENT_1_large\metricas\ALL_metrics_detailed_MOMENT_FT_SimImpute_MOMENT_1_large_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_20percent           1  1202032594   
1  Brasilia  RandomRows_20percent           2  1202079186   
2  Brasilia  RandomRows_20percent           3  1202059218   
3  Brasilia  RandomRows_20percent           4  1202005970   
4  Brasilia  RandomRows_20percent           5  1202063826   

                    Imputation_Method Variable  MAE  RMSE  NRMAE_range  \
0 

moment_ft con log

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys # Para redirigir la salida

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
    from momentfm.utils.masking import Masking 
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None
    Masking = None

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables para Fine-Tuning ---
TARGET_CITY_FT = "Brasilia"
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = AQI_COL_FT # Solo AQI para fine-tuning
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 64
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"

# Fine-tuning Parameters (NUEVA Estrategia)
FINETUNE_EPOCHS = 5000 # Reducido para pruebas más rápidas, ajustar
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 512
BATCH_SIZE_FINETUNE_NEW = 32
LEARNING_RATE_FINETUNE_NEW = 1e-5
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 1250 # Reducido
LR_PATIENCE_NEW = 500             # Reducido
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 1 # Reducido para pruebas
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_Log_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs') # Nueva carpeta para logs
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_FT_OUTPUT_DIR, exist_ok=True) # Crear carpeta de logs


# --- Redirigir stdout y stderr a un archivo de log ---
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{sanitize_name(TARGET_CITY_FT)}.txt')
original_stdout = sys.stdout
original_stderr = sys.stderr

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush() # Asegurar que se escriba inmediatamente
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file)
sys.stderr = Tee(sys.stderr, log_file)

print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
print(f"Log guardado en: {log_file_path}")


# --- 0. Carga de Datos Originales (Ground Truth) ---
print(f"\n--- 0. Carga de Datos Originales (Ground Truth) ---")
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_FT]
    _start_date = pd.to_datetime('2024-01-01')
    num_rows_example = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == 'Brasilia'].shape[0] if not df_air_quality_ground_truth_raw.empty and 'Brasilia' in df_air_quality_ground_truth_raw['City'].unique() else 8780
    print(f"Usando num_rows_example: {num_rows_example}")
    for _city_ex in _cities_example:
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print(f"DataFrame de ejemplo creado con {num_rows_example} filas para {TARGET_CITY_FT}.")

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_FT in city_timezones:
    tz = city_timezones[TARGET_CITY_FT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_FT} localized to {tz} and time features extracted. Shape: {df_air_quality_ground_truth_localized.shape}")
    else: print(f"No data for {TARGET_CITY_FT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_FT}.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date'])
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
if df_air_quality_ground_truth_localized.empty: print(f"Error: Ground truth for {TARGET_CITY_FT} is empty. Exiting."); exit()


# --- Dataset Personalizado para MOMENT (NUEVA Estrategia Fine-tuning) ---
class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_2d_scaled_clean, seq_len, step, missing_ratio):
        self.data_array = data_array_2d_scaled_clean
        self.seq_len = seq_len
        self.step = step
        self.missing_ratio = missing_ratio

        if len(self.data_array) == 0: self.n_samples, self.indices = 0, []
        elif len(self.data_array) < self.seq_len : self.n_samples, self.indices = 1, [0]
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        if self.n_samples == 0: raise IndexError("Dataset is empty")
        actual_start_index = self.indices[idx]
        seq_data_clean_numpy = self.data_array[actual_start_index : actual_start_index + self.seq_len].copy()
        
        if len(seq_data_clean_numpy) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_clean_numpy)
            seq_data_clean_numpy = np.pad(seq_data_clean_numpy, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)

        n_points_to_mask = int(self.seq_len * self.missing_ratio)
        mask_indices = np.random.choice(self.seq_len, n_points_to_mask, replace=False)
        
        simulated_nan_mask_numpy = np.zeros_like(seq_data_clean_numpy, dtype=bool)
        simulated_nan_mask_numpy[mask_indices, :] = True

        input_mask_for_moment = ~simulated_nan_mask_numpy.all(axis=1)

        x_enc_numpy = seq_data_clean_numpy.copy()
        x_enc_numpy[simulated_nan_mask_numpy] = 0.0

        return {
            'x_enc': torch.tensor(x_enc_numpy, dtype=torch.float32).transpose(0, 1),
            'input_mask': torch.tensor(input_mask_for_moment, dtype=torch.bool),
            'target_clean': torch.tensor(seq_data_clean_numpy, dtype=torch.float32).transpose(0,1),
            'simulated_nan_mask': torch.tensor(simulated_nan_mask_numpy, dtype=torch.bool).transpose(0,1)
        }

# --- Dataset Personalizado para Imputación ---
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data_array_scaled_zeros = data_array_2d_multivar_scaled_zeros
        self.nan_mask_original = nan_mask_original_bool_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array_scaled_zeros) < self.seq_len : self.n_samples, self.indices = 1, [0]
        elif len(self.data_array_scaled_zeros) == 0: self.n_samples, self.indices = 0, []
        else:
            self.indices = list(range(0, len(self.data_array_scaled_zeros) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array_scaled_zeros) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array_scaled_zeros) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        actual_start_index = self.indices[idx]
        seq_data_scaled_zeros_orig_shape = self.data_array_scaled_zeros[actual_start_index : actual_start_index + self.seq_len]
        seq_observed_mask_bool_per_channel = ~self.nan_mask_original[actual_start_index : actual_start_index + self.seq_len]
        if len(seq_data_scaled_zeros_orig_shape) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_scaled_zeros_orig_shape)
            seq_data_scaled_zeros_orig_shape = np.pad(seq_data_scaled_zeros_orig_shape, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_observed_mask_bool_per_channel = np.pad(seq_observed_mask_bool_per_channel, ((0, pad_width), (0,0)), 'constant', constant_values=False)
        timeseries_out = torch.tensor(seq_data_scaled_zeros_orig_shape, dtype=torch.float32).transpose(0, 1)
        input_mask_out = torch.tensor(seq_observed_mask_bool_per_channel, dtype=torch.bool).all(dim=1)
        return {'timeseries': timeseries_out, 'input_mask': input_mask_out }

# --- Función de Evaluación del Modelo en Set de Validación (NUEVA Estrategia) ---
def evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion):
    model.eval()
    total_val_loss = 0
    batch_count = 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch = batch_data['x_enc'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            target_clean_batch = batch_data['target_clean'].to(device)
            simulated_nan_mask_batch = batch_data['simulated_nan_mask'].to(device)
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                reconstruction_loss_at_nans = criterion(reconstruction_all, target_clean_batch)
                if reconstruction_loss_at_nans.shape != simulated_nan_mask_batch.shape:
                     simulated_nan_mask_batch = simulated_nan_mask_batch.reshape(reconstruction_loss_at_nans.shape)
                masked_loss_components = reconstruction_loss_at_nans * simulated_nan_mask_batch.float()
                loss = masked_loss_components.nansum() / (simulated_nan_mask_batch.nansum() + 1e-9)
                if not (torch.isnan(loss) or torch.isinf(loss)):
                    total_val_loss += loss.item()
                    batch_count += 1
            except Exception as e_eval: print(f"    Error en validación (strat. sim impute): {e_eval}"); continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

# --- Función de Entrenamiento (Fine-Tuning) con NUEVA Estrategia ---
def train_moment_model_finetune_simulated_imputation(
    model, train_loader, val_loader, device, epochs, variable_name, model_save_path,
    learning_rate, lr_patience, es_patience, min_delta_es
    ):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.8, patience=lr_patience, verbose=True)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    print(f"  Iniciando fine-tuning (estrategia imputación simulada) para '{variable_name}' ({epochs} épocas máx)...")
    for epoch in range(epochs):
        model.train()
        total_epoch_train_loss = 0; train_batch_count = 0
        pbar_train = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar_train:
            x_enc_batch = batch_data['x_enc'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            target_clean_batch = batch_data['target_clean'].to(device)
            simulated_nan_mask_batch = batch_data['simulated_nan_mask'].to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                reconstruction_loss_at_nans = criterion(reconstruction_all, target_clean_batch)
                if reconstruction_loss_at_nans.shape != simulated_nan_mask_batch.shape:
                     simulated_nan_mask_batch = simulated_nan_mask_batch.reshape(reconstruction_loss_at_nans.shape)
                masked_loss_components = reconstruction_loss_at_nans * simulated_nan_mask_batch.float()
                loss = masked_loss_components.nansum() / (simulated_nan_mask_batch.nansum() + 1e-9)
            except Exception as e_train: print(f"    Error train batch: {e_train}"); continue
            if torch.isnan(loss) or torch.isinf(loss) or (simulated_nan_mask_batch.nansum() < 1e-7): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item(); train_batch_count += 1
            pbar_train.set_postfix(loss=loss.item())
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        avg_epoch_val_loss = evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_val_loss - min_delta_es:
            best_val_loss = avg_epoch_val_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"      Nuevo mejor modelo para '{variable_name}' (strat. sim impute) guardado con Val Loss: {best_val_loss:.6f}")
        else: epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1} para '{variable_name}' (strat. sim impute)."); break
    print(f"  Fine-tuning (strat. sim impute) para '{variable_name}' completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path):
        try: model.load_state_dict(torch.load(model_save_path, map_location=device))
        except Exception as e: print(f"  Error cargando mejor modelo (strat. sim impute): {e}")
    return model

# --- Función de Imputación (igual que antes) ---
def impute_univar_with_finetuned_model(
    df_univar_with_nans_scaled_zeros, original_nan_mask_univar,
    model_ft_var, device_str, scaler_var,
    seq_len_imp, step_imp, batch_size_imp
    ):
    model_ft_var.eval()
    imputation_dataset = MomentImputationDataset(
        df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values,
        seq_len_imp, step_imp
    )
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e_pred: print(f"    Error impute batch {batch_idx}: {e_pred}"); continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_imp + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_imp, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_series_unscaled = scaler_var.inverse_transform(imputed_full_series_scaled)
    df_final_imputed_unscaled = pd.DataFrame(scaler_var.inverse_transform(df_univar_with_nans_scaled_zeros.values), columns=df_univar_with_nans_scaled_zeros.columns, index=df_univar_with_nans_scaled_zeros.index)
    nan_mask_this_col_flat = original_nan_mask_univar.iloc[:,0].values
    col_name = df_univar_with_nans_scaled_zeros.columns[0]
    df_final_imputed_unscaled.loc[nan_mask_this_col_flat, col_name] = imputed_full_series_unscaled[nan_mask_this_col_flat, 0]
    return df_final_imputed_unscaled

# --- Función de Métricas (igual que antes) ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values.astype(bool)
        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals, pred_col_vals, mask_col_flags = true_col_vals[:min_len], pred_col_vals[:min_len], mask_col_flags[:min_len]
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid); rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = 0
            if not original_col_full_series.empty: original_col_range = original_col_full_series.max() - original_col_full_series.min()
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan); nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
if MOMENTPipeline is None : print("MOMENTPipeline o Masking no pudieron ser importadas. Exiting script."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns: df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print(f"Error: GT con NaNs para {TARGET_CITY_FT} en columnas objetivo."); exit()

finetuned_models_sim_impute_dict = {}
scalers_for_vars_sim_impute_dict = {}
city_name_safe_ft_models = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_scalers_dir, exist_ok=True)

# mask_generator_ft ya no es necesario para esta estrategia de fine-tuning
# mask_generator_ft = Masking(mask_ratio=MASK_RATIO_FINETUNE_AUTHORS)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning (Estrategia Imputación Simulada) para la variable: {var_to_finetune} =====")
    scaler_current_var = StandardScaler()
    data_for_scaler_fit = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    if len(data_for_scaler_fit) < 1: print(f"  Datos insuficientes para scaler de {var_to_finetune}."); continue
    scaler_current_var.fit(data_for_scaler_fit)
    scalers_for_vars_sim_impute_dict[var_to_finetune] = scaler_current_var
    joblib.dump(scaler_current_var, os.path.join(city_scalers_dir, f'scaler_ft_sim_impute_{sanitize_name(var_to_finetune)}.joblib'))
    
    full_clean_data_var_scaled = scaler_current_var.transform(df_city_ground_truth_ft_reset[[var_to_finetune]].values)
    
    train_val_split_idx = int(len(full_clean_data_var_scaled) * FINETUNE_TRAIN_RATIO_NEW)
    train_data_ft_var = full_clean_data_var_scaled[:train_val_split_idx]
    val_data_ft_var = full_clean_data_var_scaled[train_val_split_idx:]

    if len(train_data_ft_var) < SEQ_LEN_MODEL_FT or len(val_data_ft_var) < SEQ_LEN_MODEL_FT:
        print(f"  Datos insuficientes para split train/val en {var_to_finetune}. Saltando fine-tuning."); finetuned_models_sim_impute_dict[var_to_finetune] = None; continue

    train_dataset_sim_impute = MomentFTSimulatedImputationDataset(train_data_ft_var, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    train_loader_sim_impute = DataLoader(train_dataset_sim_impute, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0)
    val_dataset_sim_impute = MomentFTSimulatedImputationDataset(val_data_ft_var, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    val_loader_sim_impute = DataLoader(val_dataset_sim_impute, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False, num_workers=0)

    if len(train_loader_sim_impute) == 0 or len(val_loader_sim_impute) == 0 :
        print(f"  Train o Val loader vacío para {var_to_finetune}. Saltando fine-tuning."); finetuned_models_sim_impute_dict[var_to_finetune] = None; continue
        
    model_to_finetune_sim_impute = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model_to_finetune_sim_impute.init(); model_to_finetune_sim_impute = model_to_finetune_sim_impute.to(device)
    model_save_path_var_sim_impute = os.path.join(city_models_dir, f'moment_ft_sim_impute_best_{sanitize_name(var_to_finetune)}.pth')
    
    print(f"Learning Rate para {var_to_finetune}: {LEARNING_RATE_FINETUNE_NEW}") # Verificar LR
    finetuned_model_var = train_moment_model_finetune_simulated_imputation(
        model_to_finetune_sim_impute, train_loader_sim_impute, val_loader_sim_impute, device,
        FINETUNE_EPOCHS, var_to_finetune, model_save_path_var_sim_impute,
        LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW
    )
    finetuned_models_sim_impute_dict[var_to_finetune] = finetuned_model_var
    if finetuned_model_var is not None:
        print(f"  Modelo para {var_to_finetune} (strat. sim impute) afinado y guardado/cargado.")
    else:
        print(f"  Fallo el fine-tuning (strat. sim impute) para {var_to_finetune}.")

print("\n\n--- Fine-Tuning (Estrategia Imputación Simulada) completado para todas las variables objetivo. ---")

# --- Imputation Phase ---
city_name_safe_ft_impute = sanitize_name(TARGET_CITY_FT)
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft_impute)
os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)
for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = scenario_ft['name'].replace("%", "percent")
    scenario_name_for_file_part = scenario_ft['name'].replace("%", "percent")
    print(f"\nProcesando Escenario de Imputación: {scenario_name_for_dir}")
    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft_impute, scenario_name_for_dir)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_ft_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_ft): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_ft}."); continue
    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición de Imputación: {rep_ft}")
        file_pattern_to_find_imp = f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}_*_with_nans.csv"
        sim_files_in_dir_ft = sorted([f for f in os.listdir(sim_data_scenario_dir_ft) if f.startswith(f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_ft: print(f"    Archivo simulado no encontrado para '{file_pattern_to_find_imp}'."); continue
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = "unknown"
        try: current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError: print(f"    Advertencia: No se pudo extraer seed de: {sim_files_in_dir_ft[0]}")
        try:
            df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
            if 'Date' in df_city_with_nans_loaded_ft.columns: df_city_with_nans_loaded_ft['Date'] = pd.to_datetime(df_city_with_nans_loaded_ft['Date'])
        except Exception as e: print(f"    Error cargando {sim_file_path_to_load_ft}: {e}."); continue
        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()
        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var", leave=False):
            model_ft_to_use = finetuned_models_sim_impute_dict.get(var_to_impute)
            scaler_var_to_use = scalers_for_vars_sim_impute_dict.get(var_to_impute)
            if model_ft_to_use is None or scaler_var_to_use is None:
                print(f"    Modelo/scaler (Estrat. Sim Impute) no disponible para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
                continue
            df_univar_to_impute_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]].copy()
            original_nan_mask_univar_df = df_univar_to_impute_with_nans.isnull()
            df_univar_filled_for_scaling = df_univar_to_impute_with_nans.fillna(0)
            df_univar_scaled_zeros = pd.DataFrame(scaler_var_to_use.transform(df_univar_filled_for_scaling.values), columns=[var_to_impute], index=df_univar_to_impute_with_nans.index)
            df_imputed_univar_unscaled = impute_univar_with_finetuned_model(df_univar_scaled_zeros, original_nan_mask_univar_df, model_ft_to_use, device, scaler_var_to_use, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT)
            if df_imputed_univar_unscaled is not None and var_to_impute in df_imputed_univar_unscaled.columns:
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_unscaled[var_to_impute]
            else:
                print(f"    Imputación falló para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
        df_to_save_final_ft = df_imputed_all_cols_rep_ft.copy()
        final_cols_for_saving_ft = ['Date', 'City'] + ALL_TARGET_COLS_FT
        for tf_col in TIME_FEATURES:
            if tf_col in df_city_ground_truth_ft_reset.columns:
                max_len = len(df_to_save_final_ft)
                df_to_save_final_ft[tf_col] = df_city_ground_truth_ft_reset[tf_col].values[:max_len]
                if tf_col not in final_cols_for_saving_ft: final_cols_for_saving_ft.append(tf_col)
        df_to_save_final_ft = df_to_save_final_ft[[c for c in final_cols_for_saving_ft if c in df_to_save_final_ft.columns]]
        imputed_ft_filename = f"{city_name_safe_ft_impute}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT_SimImpute.csv"
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        df_to_save_final_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT-SimImpute guardado: {imputed_ft_filename}")
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        df_loaded_nans_reset_ft = df_city_with_nans_loaded_ft.reset_index(drop=True)
        for col_m_mask in ALL_TARGET_COLS_FT:
            if col_m_mask in df_loaded_nans_reset_ft.columns and col_m_mask in df_city_ground_truth_ft_reset.columns:
                len_min_mask_ft = min(len(df_loaded_nans_reset_ft), len(df_city_ground_truth_ft_reset))
                mask_series_metric_ft = (df_loaded_nans_reset_ft[col_m_mask].iloc[:len_min_mask_ft].isnull() & ~df_city_ground_truth_ft_reset[col_m_mask].iloc[:len_min_mask_ft].isnull())
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask].iloc[:len_min_mask_ft] = mask_series_metric_ft
        metrics_scores_ft_rep = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_to_save_final_ft.reset_index(drop=True), nan_mask_introduced_for_metrics_rep_ft, ALL_TARGET_COLS_FT)
        for var_name_metric_ft in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep_ft, 'Seed': current_seed_loaded_ft, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name_metric_ft, 'MAE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('MAE', np.nan), 'RMSE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('RMSE', np.nan), 'NRMAE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMAE_range', np.nan), 'NRMSE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMSE_range', np.nan)})
        print(f"    Métricas calculadas para MOMENT-FT-SimImpute, rep {rep_ft}")
    print(f"  Escenario de Imputación {scenario_name_for_dir} completado.")

if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT-SimImpute...")
    sufijo_test_ft = f"_{sanitize_name(TARGET_CITY_FT)}"
    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT-SimImpute guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())
    if not df_all_detailed_metrics_ft.empty:
        grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
        grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-SimImpute de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)
        current_contaminant_cols_summary = [col for col in ALL_TARGET_COLS_FT if col in CONTAMINANT_COLS_FT] if isinstance(CONTAMINANT_COLS_FT, list) else ([CONTAMINANT_COLS_FT] if isinstance(CONTAMINANT_COLS_FT, str) else [])
        current_aqi_cols_summary = [col for col in ALL_TARGET_COLS_FT if col in AQI_COL_FT] if isinstance(AQI_COL_FT, list) else ([AQI_COL_FT] if isinstance(AQI_COL_FT, str) else [])
        if current_contaminant_cols_summary:
            df_contaminants_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(current_contaminant_cols_summary)]
            if not df_contaminants_only_ft.empty:
                contaminants_combined_summary_ft = df_contaminants_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                contaminants_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
                contaminants_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                contaminants_combined_summary_ft.to_csv(contaminants_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-SimImpute contaminantes guardado: {contaminants_summary_path_ft}"); print(contaminants_combined_summary_ft)
        if current_aqi_cols_summary:
            df_aqi_only_ft = df_all_detailed_metrics_ft[df_all_detailed_metrics_ft['Variable'].isin(current_aqi_cols_summary)]
            if not df_aqi_only_ft.empty:
                aqi_combined_summary_ft = df_aqi_only_ft.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
                aqi_combined_summary_ft.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
                aqi_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
                aqi_combined_summary_ft.to_csv(aqi_summary_path_ft, index=False); print(f"  Resumen MOMENT-FT-SimImpute AQI guardado: {aqi_summary_path_ft}"); print(aqi_combined_summary_ft)
else: print("No se generaron métricas detalladas para MOMENT-FT-SimImpute.")
script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning (Estrategia Imputación Simulada) e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
sys.stdout = original_stdout
sys.stderr = original_stderr
log_file.close()
print(f"Salida de log completada y guardada en: {log_file_path}")

--- INICIO DEL SCRIPT: 2025-06-15 11:48:40 ---
Log guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_Log_MOMENT_1_large\logs\log_MOMENT_FT_Log_MOMENT_1_large_Brasilia.txt

--- 0. Carga de Datos Originales (Ground Truth) ---
Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Ground truth for Brasilia localized to America/Sao_Paulo and time features extracted. Shape: (8784, 12)
Usando dispositivo: cpu

===== Iniciando Fine-Tuning (Estrategia Imputación Simulada) para la variable: AQI =====
Learning Rate para AQI: 1e-05
  Iniciando fine-tuning (estrategia imputación simulada) para 'AQI' (5000 épocas máx)...


    Epoch 1: Train Loss: 1.030294, Val Loss: 1.260165 (LR: 1.00e-05)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 1.260165


    Epoch 2: Train Loss: 1.045842, Val Loss: 1.315095 (LR: 1.00e-05)


    Epoch 3: Train Loss: 0.901036, Val Loss: 1.205551 (LR: 1.00e-05)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 1.205551


    Epoch 4: Train Loss: 0.897863, Val Loss: 0.996334 (LR: 1.00e-05)
      Nuevo mejor modelo para 'AQI' (strat. sim impute) guardado con Val Loss: 0.996334


...

    Archivo imputado guardado: Brasilia_BlockRows_48h_20percent_rep1_seed107941404_imputed.csv

Procesando Escenario: BlockRows_60h_20percent
  Repetición: 1


  Imputando Rep 1:   0%|          | 0/1 [00:00<?, ?it/s]

    Error impute batch 0: index 1 is out of bounds for dimension 1 with size 1


  Imputando Rep 1: 100%|##########| 1/1 [00:00<00:00, 109.67it/s]


    Archivo imputado guardado: Brasilia_BlockRows_60h_20percent_rep1_seed2179968212_imputed.csv

Generando resúmenes de métricas...

Proceso completado. Tiempo total: 1.03 minutos.
--- FIN DEL SCRIPT: 2025-06-15 12:15:32 ---
Salida de log completada y guardada en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_Log_MOMENT_1_large\logs\log_MOMENT_FT_Log_MOMENT_1_large_Brasilia.txt


a ver si solucionamos la perdida 0

In [6]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys # Para redirigir la salida

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
    from momentfm.utils.masking import Masking 
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None
    Masking = None

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables para Fine-Tuning ---
TARGET_CITY_FT = "Brasilia"
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = AQI_COL_FT # Solo AQI para fine-tuning
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 64
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-base"

# SOLUCIÓN: Parámetro para replicación de canal
EXPECTED_MODEL_CHANNELS = 8

# Fine-tuning Parameters (NUEVA Estrategia)
FINETUNE_EPOCHS = 5000
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 256 # Generar más muestras
BATCH_SIZE_FINETUNE_NEW = 32       # Batch size más pequeño
LEARNING_RATE_FINETUNE_NEW = 1e-3
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 250
LR_PATIENCE_NEW = 50
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 1
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_24h_20percent'},
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_Loggers_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_FT_OUTPUT_DIR, exist_ok=True)

# --- Redirigir stdout y stderr a un archivo de log ---
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{sanitize_name(TARGET_CITY_FT)}.txt')
original_stdout = sys.stdout
original_stderr = sys.stderr
class Tee:
    def __init__(self, *files): self.files = files
    def write(self, obj):
        for f in self.files: f.write(obj); f.flush()
    def flush(self):
        for f in self.files: f.flush()
log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file)
sys.stderr = Tee(sys.stderr, log_file)
print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
print(f"Log guardado en: {log_file_path}")

# --- 0. Carga de Datos Originales (Ground Truth) ---
# ... (código de carga de datos sin cambios, se asume correcto) ...
print(f"\n--- 0. Carga de Datos Originales (Ground Truth) ---")
df_air_quality_ground_truth_raw = pd.DataFrame() # Initialize
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except Exception as e:
    print(f"Advertencia: {e}. Saliendo."); exit()

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {'Brasilia': 'America/Sao_Paulo'}
df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
if df_city_part_gt.empty: print(f"Error: No hay datos para {TARGET_CITY_FT}. Saliendo."); exit()
if TARGET_CITY_FT in city_timezones:
    df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(city_timezones[TARGET_CITY_FT])
df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
df_air_quality_ground_truth_localized = df_city_part_gt
print(f"Ground truth para {TARGET_CITY_FT} procesado. Shape: {df_air_quality_ground_truth_localized.shape}")


# SOLUCIÓN: Datasets adaptados para replicar el canal univariado
class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_1d, seq_len, step, missing_ratio, n_replications):
        self.data = data_array_1d.squeeze()
        self.seq_len, self.step, self.missing_ratio, self.n_replications = seq_len, step, missing_ratio, n_replications
        if len(self.data) < self.seq_len: self.indices = [0]
        else: self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq_1d = self.data[start_idx : start_idx + self.seq_len]
        if len(seq_1d) < self.seq_len:
            seq_1d = np.pad(seq_1d, (0, self.seq_len - len(seq_1d)), 'constant')
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        mask_indices = np.random.choice(self.seq_len, int(self.seq_len * self.missing_ratio), replace=False)
        sim_nan_mask = np.zeros_like(seq_replicated, dtype=bool)
        sim_nan_mask[mask_indices, :] = True
        input_mask = ~sim_nan_mask.all(axis=1)
        x_enc = seq_replicated.copy(); x_enc[sim_nan_mask] = 0.0
        return {
            'x_enc': torch.tensor(x_enc, dtype=torch.float32).transpose(0, 1),
            'input_mask': torch.tensor(input_mask, dtype=torch.bool),
            'target_clean': torch.tensor(seq_replicated, dtype=torch.float32).transpose(0, 1),
            'simulated_nan_mask': torch.tensor(sim_nan_mask, dtype=torch.bool).transpose(0, 1)
        }

class MomentImputationDataset(Dataset):
    def __init__(self, data_array_1d, nan_mask_1d, seq_len, step, n_replications):
        self.data = data_array_1d.squeeze()
        self.mask = nan_mask_1d.squeeze()
        self.seq_len, self.step, self.n_replications = seq_len, step, n_replications
        if len(self.data) < self.seq_len: self.indices = [0]
        else: self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start = self.indices[idx]
        seq_1d = self.data[start:start + self.seq_len]
        mask_1d = self.mask[start:start + self.seq_len]
        if len(seq_1d) < self.seq_len:
            pad_len = self.seq_len - len(seq_1d)
            seq_1d = np.pad(seq_1d, (0, pad_len), 'constant')
            mask_1d = np.pad(mask_1d, (0, pad_len), 'constant', constant_values=True)
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        return {
            'timeseries': torch.tensor(seq_replicated, dtype=torch.float32).transpose(0, 1),
            'input_mask': torch.tensor(~mask_1d, dtype=torch.bool)
        }

# --- Funciones de Entrenamiento y Evaluación (restauradas a la versión original) ---
def evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion):
    model.eval()
    total_val_loss, batch_count = 0, 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (
                batch_data['x_enc'].to(device), batch_data['input_mask'].to(device),
                batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device)
            )
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                if not torch.isnan(loss):
                    total_val_loss += loss.item()
                    batch_count += 1
            except Exception as e: print(f"    Error en validación: {e}"); continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

def train_moment_model_finetune_simulated_imputation(model, train_loader, val_loader, device, epochs, variable_name, model_save_path, learning_rate, lr_patience, es_patience, min_delta_es):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=lr_patience, verbose=True)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    print(f"  Iniciando fine-tuning para '{variable_name}'...")
    for epoch in range(epochs):
        model.train()
        total_epoch_train_loss, train_batch_count = 0, 0
        pbar_train = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar_train:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (
                batch_data['x_enc'].to(device), batch_data['input_mask'].to(device),
                batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device)
            )
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
            except Exception as e: print(f"    Error train batch: {e}"); continue
            if torch.isnan(loss) or simulated_nan_mask_batch.sum() < 1: continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item(); train_batch_count += 1
            pbar_train.set_postfix(loss=loss.item())
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        avg_epoch_val_loss = evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_val_loss - min_delta_es:
            best_val_loss = avg_epoch_val_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"      Nuevo mejor modelo guardado con Val Loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1}."); break
    print(f"  Fine-tuning completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

# SOLUCIÓN: Función de imputación que promedia los canales replicados
def impute_univar_with_finetuned_model(model, df_univar_with_nans_scaled_zeros, original_nan_mask_univar, scaler_var, seq_len_imp, step_imp, batch_size_imp, n_replications_imp):
    model.eval()
    imputation_dataset = MomentImputationDataset(df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values, seq_len_imp, step_imp, n_replications_imp)
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows = len(df_univar_with_nans_scaled_zeros)
    preds_sum, counts = np.zeros(n_rows), np.zeros(n_rows)
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(imputation_loader, desc=f"  Imputando", leave=False)):
            timeseries_batch = batch_data['timeseries'].to(model.device)
            input_mask_batch = batch_data['input_mask'].to(model.device)
            try:
                outputs = model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                reconstruction = outputs.reconstruction.mean(dim=1).cpu().numpy() # Promediar canales
            except Exception as e: print(f"    Error impute batch {i}: {e}"); continue
            
            for j in range(reconstruction.shape[0]):
                start_idx = imputation_loader.dataset.indices[i * batch_size_imp + j]
                end_idx = min(start_idx + seq_len_imp, n_rows)
                pred_len = reconstruction.shape[1]
                actual_len = end_idx - start_idx
                len_to_add = min(pred_len, actual_len)
                preds_sum[start_idx : start_idx + len_to_add] += reconstruction[j, :len_to_add]
                counts[start_idx : start_idx + len_to_add] += 1
    counts[counts == 0] = 1
    imputed_1d_scaled = preds_sum / counts
    imputed_1d_unscaled = scaler_var.inverse_transform(imputed_1d_scaled.reshape(-1, 1)).squeeze()
    
    df_final = df_univar_with_nans_scaled_zeros.copy()
    df_final.iloc[:, 0] = scaler_var.inverse_transform(df_final.values)
    mask = original_nan_mask_univar.iloc[:, 0].values
    df_final.iloc[mask, 0] = imputed_1d_unscaled[mask]
    return df_final

def calculate_metrics_moment(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    results = {}
    y_true_df, y_pred_df, nan_mask_df = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        true_vals, pred_vals, mask = y_true_df[col], y_pred_df[col], nan_mask_df[col].astype(bool)
        if mask.sum() == 0: continue
        true_at_nans, pred_at_nans = true_vals[mask], pred_vals[mask]
        mae = mean_absolute_error(true_at_nans, pred_at_nans)
        rmse = np.sqrt(mean_squared_error(true_at_nans, pred_at_nans))
        data_range = y_true_df[col].max() - y_true_df[col].min()
        results[col] = {'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 0 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 0 else np.nan}
    return results

# --- Script Principal ---
if MOMENTPipeline is None: print("MOMENTPipeline not imported. Exiting."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col in ALL_TARGET_COLS_FT:
    df_city_ground_truth_ft_reset[col] = df_city_ground_truth_ft_reset[col].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print("Error: GT con NaNs."); exit()

finetuned_models_dict = {}
scalers_dict = {}
city_name_safe = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_scalers_dir, exist_ok=True)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning para: {var_to_finetune} =====")
    scaler = StandardScaler()
    data_to_scale = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    scaler.fit(data_to_scale)
    scalers_dict[var_to_finetune] = scaler
    joblib.dump(scaler, os.path.join(city_scalers_dir, f'scaler_{sanitize_name(var_to_finetune)}.joblib'))
    
    scaled_data = scaler.transform(data_to_scale)
    split_idx = int(len(scaled_data) * FINETUNE_TRAIN_RATIO_NEW)
    train_data, val_data = scaled_data[:split_idx], scaled_data[split_idx:]
    
    train_dataset = MomentFTSimulatedImputationDataset(train_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0, drop_last=False)
    val_dataset = MomentFTSimulatedImputationDataset(val_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False)

    if len(train_loader) == 0: print(f"Train loader vacío para {var_to_finetune}. Saltando."); continue
        
    model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model.init() # Inicializar o re-inicializar
    model = model.to(device)
    model_save_path = os.path.join(city_models_dir, f'moment_ft_best_{sanitize_name(var_to_finetune)}.pth')
    
    finetuned_model = train_moment_model_finetune_simulated_imputation(
        model, train_loader, val_loader, device, FINETUNE_EPOCHS, var_to_finetune, model_save_path,
        LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW
    )
    finetuned_models_dict[var_to_finetune] = finetuned_model
    if finetuned_model: print(f"  Modelo para {var_to_finetune} afinado.")

# --- Imputation Phase ---
print("\n--- Iniciando Fase de Imputación ---")
city_imputed_csv_dir = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_imputed_csv_dir, exist_ok=True)
for scenario in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = sanitize_name(scenario['name'])
    scenario_name_for_file = scenario['name'].replace('%', 'percent')
    print(f"\nProcesando Escenario: {scenario_name_for_dir}")
    sim_data_dir = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe, scenario_name_for_dir)
    imputed_scenario_dir = os.path.join(city_imputed_csv_dir, scenario_name_for_dir)
    os.makedirs(imputed_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_dir): print(f"    Dir simulado no encontrado: {sim_data_dir}."); continue
    for rep in range(1, N_REPETITIONS_FT_EVAL + 1):
        sim_files = [f for f in os.listdir(sim_data_dir) if f.startswith(f"{city_name_safe}_{scenario_name_for_file}_rep{rep}") and f.endswith("_with_nans.csv")]
        if not sim_files: print(f"    Archivo no encontrado para rep {rep}."); continue
        sim_file_path = os.path.join(sim_data_dir, sim_files[0])
        seed = sim_files[0].split('_seed')[-1].split('_with_nans.csv')[0] if '_seed' in sim_files[0] else "unknown"
        df_with_nans = pd.read_csv(sim_file_path, parse_dates=['Date'])
        
        df_imputed = df_with_nans.copy()
        for var in ALL_TARGET_COLS_FT:
            model, scaler = finetuned_models_dict.get(var), scalers_dict.get(var)
            if not model or not scaler: print(f"    Modelo/scaler no disponible para {var}."); continue
            
            univar_with_nans = df_with_nans[[var]].copy()
            nan_mask = univar_with_nans.isnull()
            filled_for_scaling = univar_with_nans.fillna(0)
            scaled_zeros = pd.DataFrame(scaler.transform(filled_for_scaling), columns=[var], index=univar_with_nans.index)
            
            imputed_univar = impute_univar_with_finetuned_model(model, scaled_zeros, nan_mask, scaler, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT, EXPECTED_MODEL_CHANNELS)
            if imputed_univar is not None: df_imputed[var] = imputed_univar[var]
        
        imputed_filename = f"{city_name_safe}_{scenario_name_for_dir}_rep{rep}_seed{seed}_imputed.csv"
        imputed_save_path = os.path.join(imputed_scenario_dir, imputed_filename)
        df_imputed.to_csv(imputed_save_path, index=False)
        print(f"    Archivo imputado guardado: {imputed_filename}")
        
        nan_mask_for_metrics = df_with_nans[ALL_TARGET_COLS_FT].isnull() & ~df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull()
        metrics = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_imputed, nan_mask_for_metrics, ALL_TARGET_COLS_FT)
        for var, scores in metrics.items():
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep, 'Seed': seed, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var, **scores})

# --- Guardar Métricas ---
if all_detailed_metrics_ft_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas...")
    sufijo = f"_{city_name_safe}"
    df_metrics.to_csv(os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo}.csv'), index=False)
    print(df_metrics)
else: print("No se generaron métricas.")
print(f"\nProceso completado. Tiempo total: {(time.time() - script_start_time_ft) / 60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
if log_file:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

AttributeError: 'MOMENTPipeline' object has no attribute 'device'

In [9]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys # Para redirigir la salida

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
    from momentfm.utils.masking import Masking 
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None
    Masking = None

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables ---
TARGET_CITY_FT = "Brasilia"
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = AQI_COL_FT
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT // 4
BATCH_SIZE_IMPUTE_FT = 64
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"
EXPECTED_MODEL_CHANNELS = 8

# Fine-tuning Parameters
FINETUNE_EPOCHS = 5000
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 256
BATCH_SIZE_FINETUNE_NEW = 32
LEARNING_RATE_FINETUNE_NEW = 1e-3 # Un poco más alto para acelerar convergencia
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 250
LR_PATIENCE_NEW = 50
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 5
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_Log_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_FT_OUTPUT_DIR, exist_ok=True)

# --- Redirigir stdout y stderr a un archivo de log ---
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{sanitize_name(TARGET_CITY_FT)}.txt')
original_stdout, original_stderr = sys.stdout, sys.stderr
class Tee:
    def __init__(self, *files): self.files = files
    def write(self, obj):
        for f in self.files: f.write(obj); f.flush()
    def flush(self):
        for f in self.files: f.flush()
log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file)
sys.stderr = Tee(sys.stderr, log_file)
print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")

# --- 0. Carga de Datos Originales (Ground Truth) ---
print(f"\n--- 0. Carga de Datos Originales (Ground Truth) ---")
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales cargados. Shape: {df_air_quality_ground_truth_raw.shape}")
except Exception as e:
    print(f"Advertencia: {e}. Saliendo."); exit()

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
city_timezones = {'Brasilia': 'America/Sao_Paulo'}
df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
if df_city_part_gt.empty: print(f"Error: No hay datos para {TARGET_CITY_FT}. Saliendo."); exit()
if TARGET_CITY_FT in city_timezones:
    df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(city_timezones[TARGET_CITY_FT])
df_air_quality_ground_truth_localized = df_city_part_gt
print(f"Ground truth para {TARGET_CITY_FT} procesado. Shape: {df_air_quality_ground_truth_localized.shape}")

# --- Datasets con replicación de canal ---
class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_1d, seq_len, step, missing_ratio, n_replications):
        self.data = data_array_1d.squeeze()
        self.seq_len, self.step, self.missing_ratio, self.n_replications = seq_len, step, missing_ratio, n_replications
        if len(self.data) < self.seq_len: self.indices = [0]
        else: self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq_1d = self.data[start_idx : start_idx + self.seq_len]
        if len(seq_1d) < self.seq_len:
            seq_1d = np.pad(seq_1d, (0, self.seq_len - len(seq_1d)), 'constant')
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        mask_indices = np.random.choice(self.seq_len, int(self.seq_len * self.missing_ratio), replace=False)
        sim_nan_mask = np.zeros_like(seq_replicated, dtype=bool); sim_nan_mask[mask_indices, :] = True
        input_mask = ~sim_nan_mask.all(axis=1)
        x_enc = seq_replicated.copy(); x_enc[sim_nan_mask] = 0.0
        return {'x_enc': torch.tensor(x_enc, dtype=torch.float32).transpose(0, 1), 'input_mask': torch.tensor(input_mask, dtype=torch.bool), 'target_clean': torch.tensor(seq_replicated, dtype=torch.float32).transpose(0, 1), 'simulated_nan_mask': torch.tensor(sim_nan_mask, dtype=torch.bool).transpose(0, 1)}

class MomentImputationDataset(Dataset):
    def __init__(self, data_array_1d, nan_mask_1d, seq_len, step, n_replications):
        self.data, self.mask, self.seq_len, self.step, self.n_replications = data_array_1d.squeeze(), nan_mask_1d.squeeze(), seq_len, step, n_replications
        if len(self.data) < self.seq_len: self.indices = [0]
        else: self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start = self.indices[idx]
        seq_1d = self.data[start:start + self.seq_len]
        mask_1d = self.mask[start:start + self.seq_len]
        if len(seq_1d) < self.seq_len:
            pad_len = self.seq_len - len(seq_1d)
            seq_1d = np.pad(seq_1d, (0, pad_len), 'constant')
            mask_1d = np.pad(mask_1d, (0, pad_len), 'constant', constant_values=True)
        seq_replicated = np.tile(seq_1d[:, np.newaxis], (1, self.n_replications))
        return {'timeseries': torch.tensor(seq_replicated, dtype=torch.float32).transpose(0, 1), 'input_mask': torch.tensor(~mask_1d, dtype=torch.bool)}

# --- Funciones de Entrenamiento y Evaluación ---
def evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion):
    model.eval()
    total_val_loss, batch_count = 0, 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                if simulated_nan_mask_batch.sum() > 0:
                    loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                    if not torch.isnan(loss): total_val_loss += loss.item(); batch_count += 1
            except Exception as e: print(f"    Error en validación: {e}"); continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

def train_moment_model_finetune_simulated_imputation(model, train_loader, val_loader, device, epochs, variable_name, model_save_path, learning_rate, lr_patience, es_patience, min_delta_es):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=lr_patience, verbose=True)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    print(f"  Iniciando fine-tuning para '{variable_name}'...")
    for epoch in range(epochs):
        model.train()
        total_epoch_train_loss, train_batch_count = 0, 0
        pbar_train = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar_train:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                if simulated_nan_mask_batch.sum() > 0:
                    loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                else: continue
            except Exception as e: print(f"    Error train batch: {e}"); continue
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item(); train_batch_count += 1
            pbar_train.set_postfix(loss=loss.item())
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        avg_epoch_val_loss = evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_val_loss - min_delta_es:
            best_val_loss = avg_epoch_val_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"      Nuevo mejor modelo guardado con Val Loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1}."); break
    print(f"  Fine-tuning completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

# SOLUCIÓN: La función de imputación ahora toma `device` como argumento
def impute_univar_with_finetuned_model(model, df_univar_with_nans_scaled_zeros, original_nan_mask_univar, scaler_var, seq_len_imp, step_imp, batch_size_imp, n_replications_imp, device):
    model.eval()
    imputation_dataset = MomentImputationDataset(df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values, seq_len_imp, step_imp, n_replications_imp)
    if not imputation_dataset.indices: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows = len(df_univar_with_nans_scaled_zeros)
    preds_sum, counts = np.zeros(n_rows), np.zeros(n_rows)
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(imputation_loader, desc=f"  Imputando", leave=False)):
            timeseries_batch = batch_data['timeseries'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            try:
                outputs = model(x_enc=timeseries_batch, input_mask=input_mask_batch)
                reconstruction = outputs.reconstruction.mean(dim=1).cpu().numpy()
            except Exception as e: print(f"    Error impute batch {i}: {e}"); continue
            for j in range(reconstruction.shape[0]):
                start_idx = imputation_loader.dataset.indices[i * batch_size_imp + j]
                end_idx = min(start_idx + seq_len_imp, n_rows)
                len_to_add = min(reconstruction.shape[1], end_idx - start_idx)
                preds_sum[start_idx : start_idx + len_to_add] += reconstruction[j, :len_to_add]
                counts[start_idx : start_idx + len_to_add] += 1
    counts[counts == 0] = 1
    imputed_1d_scaled = preds_sum / counts
    imputed_1d_unscaled = scaler_var.inverse_transform(imputed_1d_scaled.reshape(-1, 1)).squeeze()
    df_final = df_univar_with_nans_scaled_zeros.copy()
    df_final.iloc[:, 0] = scaler_var.inverse_transform(df_final.values)
    mask = original_nan_mask_univar.iloc[:, 0].values
    df_final.iloc[mask, 0] = imputed_1d_unscaled[mask]
    return df_final

def calculate_metrics_moment(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    results = {}
    y_true_df, y_pred_df, nan_mask_df = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        true_vals, pred_vals, mask = y_true_df[col], y_pred_df[col], nan_mask_df[col].astype(bool)
        if mask.sum() == 0: continue
        true_at_nans, pred_at_nans = true_vals[mask], pred_vals[mask]
        mae = mean_absolute_error(true_at_nans, pred_at_nans)
        rmse = np.sqrt(mean_squared_error(true_at_nans, pred_at_nans))
        data_range = y_true_df[col].max() - y_true_df[col].min()
        results[col] = {'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 0 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 0 else np.nan}
    return results

# --- Script Principal ---
if MOMENTPipeline is None: print("MOMENTPipeline not imported. Exiting."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col in ALL_TARGET_COLS_FT:
    df_city_ground_truth_ft_reset[col] = df_city_ground_truth_ft_reset[col].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print("Error: GT con NaNs."); exit()

finetuned_models_dict = {}
scalers_dict = {}
city_name_safe = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_scalers_dir, exist_ok=True)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning para: {var_to_finetune} =====")
    scaler = StandardScaler()
    data_to_scale = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    scaler.fit(data_to_scale)
    scalers_dict[var_to_finetune] = scaler
    joblib.dump(scaler, os.path.join(city_scalers_dir, f'scaler_{sanitize_name(var_to_finetune)}.joblib'))
    
    scaled_data = scaler.transform(data_to_scale)
    split_idx = int(len(scaled_data) * FINETUNE_TRAIN_RATIO_NEW)
    train_data, val_data = scaled_data[:split_idx], scaled_data[split_idx:]
    
    train_dataset = MomentFTSimulatedImputationDataset(train_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0, drop_last=False)
    val_dataset = MomentFTSimulatedImputationDataset(val_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT, EXPECTED_MODEL_CHANNELS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False)

    if len(train_loader) == 0: print(f"Train loader vacío para {var_to_finetune}. Saltando."); continue
        
    model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model.init()
    model = model.to(device)
    model_save_path = os.path.join(city_models_dir, f'moment_ft_best_{sanitize_name(var_to_finetune)}.pth')
    
    finetuned_model = train_moment_model_finetune_simulated_imputation(
        model, train_loader, val_loader, device, FINETUNE_EPOCHS, var_to_finetune, model_save_path,
        LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW
    )
    finetuned_models_dict[var_to_finetune] = finetuned_model
    if finetuned_model: print(f"  Modelo para {var_to_finetune} afinado.")

# --- Imputation Phase ---
print("\n--- Iniciando Fase de Imputación ---")
city_imputed_csv_dir = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_imputed_csv_dir, exist_ok=True)
for scenario in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = sanitize_name(scenario['name'])
    scenario_name_for_file = scenario['name'].replace('%', 'percent')
    print(f"\nProcesando Escenario: {scenario_name_for_dir}")
    sim_data_dir = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe, scenario_name_for_dir)
    imputed_scenario_dir = os.path.join(city_imputed_csv_dir, scenario_name_for_dir)
    os.makedirs(imputed_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_dir): print(f"    Dir simulado no encontrado: {sim_data_dir}."); continue
    for rep in range(1, N_REPETITIONS_FT_EVAL + 1):
        sim_files = [f for f in os.listdir(sim_data_dir) if f.startswith(f"{city_name_safe}_{scenario_name_for_file}_rep{rep}") and f.endswith("_with_nans.csv")]
        if not sim_files: print(f"    Archivo no encontrado para rep {rep}."); continue
        sim_file_path = os.path.join(sim_data_dir, sim_files[0])
        seed = sim_files[0].split('_seed')[-1].split('_with_nans.csv')[0] if '_seed' in sim_files[0] else "unknown"
        df_with_nans = pd.read_csv(sim_file_path, parse_dates=['Date'])
        
        df_imputed = df_with_nans.copy()
        for var in ALL_TARGET_COLS_FT:
            model, scaler = finetuned_models_dict.get(var), scalers_dict.get(var)
            if not model or not scaler: print(f"    Modelo/scaler no disponible para {var}."); continue
            
            univar_with_nans = df_with_nans[[var]].copy()
            nan_mask = univar_with_nans.isnull()
            filled_for_scaling = univar_with_nans.fillna(0)
            scaled_zeros = pd.DataFrame(scaler.transform(filled_for_scaling), columns=[var], index=univar_with_nans.index)
            
            # SOLUCIÓN: Pasar la variable `device` a la función de imputación
            imputed_univar = impute_univar_with_finetuned_model(model, scaled_zeros, nan_mask, scaler, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT, EXPECTED_MODEL_CHANNELS, device)
            if imputed_univar is not None: df_imputed[var] = imputed_univar[var]
        
        imputed_filename = f"{city_name_safe}_{scenario_name_for_dir}_rep{rep}_seed{seed}_imputed.csv"
        imputed_save_path = os.path.join(imputed_scenario_dir, imputed_filename)
        df_imputed.to_csv(imputed_save_path, index=False)
        print(f"    Archivo imputado guardado: {imputed_filename}")
        
        nan_mask_for_metrics = df_with_nans[ALL_TARGET_COLS_FT].isnull() & ~df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull()
        metrics = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_imputed, nan_mask_for_metrics, ALL_TARGET_COLS_FT)
        for var, scores in metrics.items():
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep, 'Seed': seed, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var, **scores})

# --- Guardar Métricas ---
if all_detailed_metrics_ft_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas...")
    sufijo = f"_{city_name_safe}"
    df_metrics.to_csv(os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo}.csv'), index=False)
    print(df_metrics)
else: print("No se generaron métricas.")
print(f"\nProceso completado. Tiempo total: {(time.time() - script_start_time_ft) / 60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
if log_file:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

el que mejor fue con log

In [25]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys # Para redirigir la salida

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
    from momentfm.utils.masking import Masking 
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None
    Masking = None

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables para Fine-Tuning ---
TARGET_CITY_FT = "Brasilia"
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = AQI_COL_FT # Solo AQI para fine-tuning
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT
BATCH_SIZE_IMPUTE_FT = 64
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"

# Fine-tuning Parameters (NUEVA Estrategia)
FINETUNE_EPOCHS = 5000 # Reducido para pruebas más rápidas, ajustar
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 512
BATCH_SIZE_FINETUNE_NEW = 16
LEARNING_RATE_FINETUNE_NEW = 1e-2
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 250 # Reducido
LR_PATIENCE_NEW = 80             # Reducido
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 1 # Reducido para pruebas
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_final_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs') # Nueva carpeta para logs
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_FT_OUTPUT_DIR, exist_ok=True) # Crear carpeta de logs


# --- Redirigir stdout y stderr a un archivo de log ---
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{sanitize_name(TARGET_CITY_FT)}_{time.strftime("%Y%m%d-%H%M%S")}.txt')
original_stdout = sys.stdout
original_stderr = sys.stderr

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            try:
                f.write(obj)
                f.flush() # Asegurar que se escriba inmediatamente
            except (IOError, ValueError):
                # Manejar el caso de que el archivo se cierre inesperadamente
                pass
    def flush(self):
        for f in self.files:
            try:
                f.flush()
            except (IOError, ValueError):
                pass

log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file)
sys.stderr = Tee(sys.stderr, log_file)

print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
print(f"Log guardado en: {log_file_path}")


# --- 0. Carga de Datos Originales (Ground Truth) ---
print(f"\n--- 0. Carga de Datos Originales (Ground Truth) ---")
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = [TARGET_CITY_FT]
    _start_date = pd.to_datetime('2024-01-01')
    num_rows_example = 8780 # Default
    try:
        # Intenta obtener el número de filas del dataframe real si se cargó un dummy antes
        if 'df_air_quality_ground_truth_raw' in locals() and not df_air_quality_ground_truth_raw.empty and 'Brasilia' in df_air_quality_ground_truth_raw['City'].unique():
             num_rows_example = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == 'Brasilia'].shape[0]
    except Exception:
        pass
    print(f"Usando num_rows_example: {num_rows_example}")
    for _city_ex in _cities_example:
        for i in range(num_rows_example):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth_raw = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print(f"DataFrame de ejemplo creado con {num_rows_example} filas para {TARGET_CITY_FT}.")

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {
    'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai',
    'London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'
}
df_air_quality_ground_truth_localized = pd.DataFrame()
if TARGET_CITY_FT in city_timezones:
    tz = city_timezones[TARGET_CITY_FT]
    df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_city_part_gt.empty:
        df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(tz)
        df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
        df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
        df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
        df_air_quality_ground_truth_localized = df_city_part_gt
        print(f"Ground truth for {TARGET_CITY_FT} localized to {tz} and time features extracted. Shape: {df_air_quality_ground_truth_localized.shape}")
    else: print(f"No data for {TARGET_CITY_FT} in ground truth for timezone conversion.")
else:
    print(f"Timezone not defined for {TARGET_CITY_FT}.")
    df_air_quality_ground_truth_localized = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
    if not df_air_quality_ground_truth_localized.empty and 'Date' in df_air_quality_ground_truth_localized.columns:
        df_air_quality_ground_truth_localized['Date'] = pd.to_datetime(df_air_quality_ground_truth_localized['Date'])
        df_air_quality_ground_truth_localized['Hour'] = df_air_quality_ground_truth_localized['Date'].dt.hour
        df_air_quality_ground_truth_localized['DayOfWeek'] = df_air_quality_ground_truth_localized['Date'].dt.dayofweek
        df_air_quality_ground_truth_localized['Month'] = df_air_quality_ground_truth_localized['Date'].dt.month
if df_air_quality_ground_truth_localized.empty: print(f"Error: Ground truth for {TARGET_CITY_FT} is empty. Exiting."); exit()


# --- Dataset Personalizado para MOMENT (NUEVA Estrategia Fine-tuning) ---
class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_2d_scaled_clean, seq_len, step, missing_ratio):
        self.data_array = data_array_2d_scaled_clean
        self.seq_len = seq_len
        self.step = step
        self.missing_ratio = missing_ratio

        if len(self.data_array) == 0: self.n_samples, self.indices = 0, []
        elif len(self.data_array) < self.seq_len : self.n_samples, self.indices = 1, [0]
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        if self.n_samples == 0: raise IndexError("Dataset is empty")
        actual_start_index = self.indices[idx]
        seq_data_clean_numpy = self.data_array[actual_start_index : actual_start_index + self.seq_len].copy()
        
        if len(seq_data_clean_numpy) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_clean_numpy)
            seq_data_clean_numpy = np.pad(seq_data_clean_numpy, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)

        n_points_to_mask = int(self.seq_len * self.missing_ratio)
        mask_indices = np.random.choice(self.seq_len, n_points_to_mask, replace=False)
        
        simulated_nan_mask_numpy = np.zeros_like(seq_data_clean_numpy, dtype=bool)
        simulated_nan_mask_numpy[mask_indices, :] = True

        input_mask_for_moment = ~simulated_nan_mask_numpy.all(axis=1)

        x_enc_numpy = seq_data_clean_numpy.copy()
        x_enc_numpy[simulated_nan_mask_numpy] = 0.0

        return {
            'x_enc': torch.tensor(x_enc_numpy, dtype=torch.float32).transpose(0, 1),
            'input_mask': torch.tensor(input_mask_for_moment, dtype=torch.bool),
            'target_clean': torch.tensor(seq_data_clean_numpy, dtype=torch.float32).transpose(0,1),
            'simulated_nan_mask': torch.tensor(simulated_nan_mask_numpy, dtype=torch.bool).transpose(0,1)
        }

# --- Dataset Personalizado para Imputación ---
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data_array_scaled_zeros = data_array_2d_multivar_scaled_zeros
        self.nan_mask_original = nan_mask_original_bool_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array_scaled_zeros) < self.seq_len : self.n_samples, self.indices = 1, [0]
        elif len(self.data_array_scaled_zeros) == 0: self.n_samples, self.indices = 0, []
        else:
            self.indices = list(range(0, len(self.data_array_scaled_zeros) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array_scaled_zeros) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array_scaled_zeros) >= self.seq_len: self.indices, self.n_samples = [0],1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        actual_start_index = self.indices[idx]
        seq_data_scaled_zeros_orig_shape = self.data_array_scaled_zeros[actual_start_index : actual_start_index + self.seq_len]
        seq_observed_mask_bool_per_channel = ~self.nan_mask_original[actual_start_index : actual_start_index + self.seq_len]
        if len(seq_data_scaled_zeros_orig_shape) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_scaled_zeros_orig_shape)
            seq_data_scaled_zeros_orig_shape = np.pad(seq_data_scaled_zeros_orig_shape, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_observed_mask_bool_per_channel = np.pad(seq_observed_mask_bool_per_channel, ((0, pad_width), (0,0)), 'constant', constant_values=False)
        timeseries_out = torch.tensor(seq_data_scaled_zeros_orig_shape, dtype=torch.float32).transpose(0, 1)
        input_mask_out = torch.tensor(seq_observed_mask_bool_per_channel, dtype=torch.bool).all(dim=1)
        return {'timeseries': timeseries_out, 'input_mask': input_mask_out }

# --- Función de Evaluación del Modelo en Set de Validación (NUEVA Estrategia) ---
def evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion):
    model.eval()
    total_val_loss = 0
    batch_count = 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch = batch_data['x_enc'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            target_clean_batch = batch_data['target_clean'].to(device)
            simulated_nan_mask_batch = batch_data['simulated_nan_mask'].to(device)
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                reconstruction_loss_at_nans = criterion(reconstruction_all, target_clean_batch)
                if reconstruction_loss_at_nans.shape != simulated_nan_mask_batch.shape:
                     simulated_nan_mask_batch = simulated_nan_mask_batch.reshape(reconstruction_loss_at_nans.shape)
                masked_loss_components = reconstruction_loss_at_nans * simulated_nan_mask_batch.float()
                loss = masked_loss_components.nansum() / (simulated_nan_mask_batch.nansum() + 1e-9)
                if not (torch.isnan(loss) or torch.isinf(loss)):
                    total_val_loss += loss.item()
                    batch_count += 1
            except Exception as e_eval: print(f"    Error en validación (strat. sim impute): {e_eval}"); continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

# --- Función de Entrenamiento (Fine-Tuning) con NUEVA Estrategia ---
def train_moment_model_finetune_simulated_imputation(
    model, train_loader, val_loader, device, epochs, variable_name, model_save_path,
    learning_rate, lr_patience, es_patience, min_delta_es
    ):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.8, patience=lr_patience, verbose=True)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    print(f"  Iniciando fine-tuning (estrategia imputación simulada) para '{variable_name}' ({epochs} épocas máx)...")
    for epoch in range(epochs):
        model.train()
        total_epoch_train_loss = 0; train_batch_count = 0
        pbar_train = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar_train:
            x_enc_batch = batch_data['x_enc'].to(device)
            input_mask_batch = batch_data['input_mask'].to(device)
            target_clean_batch = batch_data['target_clean'].to(device)
            simulated_nan_mask_batch = batch_data['simulated_nan_mask'].to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                reconstruction_loss_at_nans = criterion(reconstruction_all, target_clean_batch)
                if reconstruction_loss_at_nans.shape != simulated_nan_mask_batch.shape:
                     simulated_nan_mask_batch = simulated_nan_mask_batch.reshape(reconstruction_loss_at_nans.shape)
                masked_loss_components = reconstruction_loss_at_nans * simulated_nan_mask_batch.float()
                loss = masked_loss_components.nansum() / (simulated_nan_mask_batch.nansum() + 1e-9)
            except Exception as e_train: print(f"    Error train batch: {e_train}"); continue
            if torch.isnan(loss) or torch.isinf(loss) or (simulated_nan_mask_batch.nansum() < 1e-7): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item(); train_batch_count += 1
            pbar_train.set_postfix(loss=loss.item())
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        avg_epoch_val_loss = evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_val_loss - min_delta_es:
            best_val_loss = avg_epoch_val_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"      Nuevo mejor modelo para '{variable_name}' (strat. sim impute) guardado con Val Loss: {best_val_loss:.6f}")
        else: epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1} para '{variable_name}' (strat. sim impute)."); break
    print(f"  Fine-tuning (strat. sim impute) para '{variable_name}' completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path):
        try: model.load_state_dict(torch.load(model_save_path, map_location=device))
        except Exception as e: print(f"  Error cargando mejor modelo (strat. sim impute): {e}")
    return model

# --- Función de Imputación (igual que antes) ---
def impute_univar_with_finetuned_model(
    df_univar_with_nans_scaled_zeros, original_nan_mask_univar,
    model_ft_var, device_str, scaler_var,
    seq_len_imp, step_imp, batch_size_imp
    ):
    model_ft_var.eval()
    imputation_dataset = MomentImputationDataset(
        df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values,
        seq_len_imp, step_imp
    )
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e_pred: print(f"    Error impute batch {batch_idx}: {e_pred}"); continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_imp + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_series = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_series = min(start_idx_in_series + seq_len_imp, n_rows_total)
                len_window_actual = end_idx_in_series - start_idx_in_series
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_series:end_idx_in_series, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_series:end_idx_in_series, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_series_unscaled = scaler_var.inverse_transform(imputed_full_series_scaled)
    df_final_imputed_unscaled = pd.DataFrame(scaler_var.inverse_transform(df_univar_with_nans_scaled_zeros.values), columns=df_univar_with_nans_scaled_zeros.columns, index=df_univar_with_nans_scaled_zeros.index)
    nan_mask_this_col_flat = original_nan_mask_univar.iloc[:,0].values
    col_name = df_univar_with_nans_scaled_zeros.columns[0]
    df_final_imputed_unscaled.loc[nan_mask_this_col_flat, col_name] = imputed_full_series_unscaled[nan_mask_this_col_flat, 0]
    return df_final_imputed_unscaled

# --- Función de Métricas (igual que antes) ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values.astype(bool)
        min_len = min(len(true_col_vals), len(pred_col_vals), len(mask_col_flags))
        true_col_vals, pred_col_vals, mask_col_flags = true_col_vals[:min_len], pred_col_vals[:min_len], mask_col_flags[:min_len]
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid); rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = 0
            if not original_col_full_series.empty: original_col_range = original_col_full_series.max() - original_col_full_series.min()
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan); nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
if MOMENTPipeline is None : print("MOMENTPipeline o Masking no pudieron ser importadas. Exiting script."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns: df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print(f"Error: GT con NaNs para {TARGET_CITY_FT} en columnas objetivo."); exit()

finetuned_models_sim_impute_dict = {}
scalers_for_vars_sim_impute_dict = {}
city_name_safe_ft_models = sanitize_name(TARGET_CITY_FT)
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_models_dir, exist_ok=True)
city_scalers_dir = os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe_ft_models)
os.makedirs(city_scalers_dir, exist_ok=True)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning (Estrategia Imputación Simulada) para la variable: {var_to_finetune} =====")
    scaler_current_var = StandardScaler()
    data_for_scaler_fit = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    if len(data_for_scaler_fit) < 1: print(f"  Datos insuficientes para scaler de {var_to_finetune}."); continue
    scaler_current_var.fit(data_for_scaler_fit)
    scalers_for_vars_sim_impute_dict[var_to_finetune] = scaler_current_var
    joblib.dump(scaler_current_var, os.path.join(city_scalers_dir, f'scaler_ft_sim_impute_{sanitize_name(var_to_finetune)}.joblib'))
    
    full_clean_data_var_scaled = scaler_current_var.transform(df_city_ground_truth_ft_reset[[var_to_finetune]].values)
    
    train_val_split_idx = int(len(full_clean_data_var_scaled) * FINETUNE_TRAIN_RATIO_NEW)
    train_data_ft_var = full_clean_data_var_scaled[:train_val_split_idx]
    val_data_ft_var = full_clean_data_var_scaled[train_val_split_idx:]

    if len(train_data_ft_var) < SEQ_LEN_MODEL_FT or len(val_data_ft_var) < SEQ_LEN_MODEL_FT:
        print(f"  Datos insuficientes para split train/val en {var_to_finetune}. Saltando fine-tuning."); finetuned_models_sim_impute_dict[var_to_finetune] = None; continue

    train_dataset_sim_impute = MomentFTSimulatedImputationDataset(train_data_ft_var, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    train_loader_sim_impute = DataLoader(train_dataset_sim_impute, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0)
    val_dataset_sim_impute = MomentFTSimulatedImputationDataset(val_data_ft_var, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    val_loader_sim_impute = DataLoader(val_dataset_sim_impute, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False, num_workers=0)

    if len(train_loader_sim_impute) == 0 or len(val_loader_sim_impute) == 0 :
        print(f"  Train o Val loader vacío para {var_to_finetune}. Saltando fine-tuning."); finetuned_models_sim_impute_dict[var_to_finetune] = None; continue
        
    model_to_finetune_sim_impute = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model_to_finetune_sim_impute.init(); model_to_finetune_sim_impute = model_to_finetune_sim_impute.to(device)
    model_save_path_var_sim_impute = os.path.join(city_models_dir, f'moment_ft_sim_impute_best_{sanitize_name(var_to_finetune)}.pth')
    
    finetuned_model_var = train_moment_model_finetune_simulated_imputation(
        model_to_finetune_sim_impute, train_loader_sim_impute, val_loader_sim_impute, device,
        FINETUNE_EPOCHS, var_to_finetune, model_save_path_var_sim_impute,
        LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW
    )
    finetuned_models_sim_impute_dict[var_to_finetune] = finetuned_model_var
    if finetuned_model_var is not None:
        print(f"  Modelo para {var_to_finetune} (strat. sim impute) afinado y guardado/cargado.")
    else:
        print(f"  Fallo el fine-tuning (strat. sim impute) para {var_to_finetune}.")

print("\n\n--- Fine-Tuning (Estrategia Imputación Simulada) completado para todas las variables objetivo. ---")

# --- Imputation Phase ---
city_name_safe_ft_impute = sanitize_name(TARGET_CITY_FT)
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft_impute)
os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)
for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = scenario_ft['name'].replace("%", "percent")
    scenario_name_for_file_part = scenario_ft['name'].replace("%", "percent")
    print(f"\nProcesando Escenario de Imputación: {scenario_name_for_dir}")
    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft_impute, scenario_name_for_dir)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, sanitize_name(scenario_name_for_dir))
    os.makedirs(imputed_ft_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_ft): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_ft}."); continue
    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición de Imputación: {rep_ft}")
        file_pattern_to_find_imp = f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}_*_with_nans.csv"
        sim_files_in_dir_ft = sorted([f for f in os.listdir(sim_data_scenario_dir_ft) if f.startswith(f"{city_name_safe_ft_impute}_{scenario_name_for_file_part}_rep{rep_ft}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_ft: print(f"    Archivo simulado no encontrado para '{file_pattern_to_find_imp}'."); continue
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = "unknown"
        try: current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        except IndexError: print(f"    Advertencia: No se pudo extraer seed de: {sim_files_in_dir_ft[0]}")
        try:
            df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
            if 'Date' in df_city_with_nans_loaded_ft.columns: df_city_with_nans_loaded_ft['Date'] = pd.to_datetime(df_city_with_nans_loaded_ft['Date'])
        except Exception as e: print(f"    Error cargando {sim_file_path_to_load_ft}: {e}."); continue
        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()
        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var", leave=False):
            model_ft_to_use = finetuned_models_sim_impute_dict.get(var_to_impute)
            scaler_var_to_use = scalers_for_vars_sim_impute_dict.get(var_to_impute)
            if model_ft_to_use is None or scaler_var_to_use is None:
                print(f"    Modelo/scaler (Estrat. Sim Impute) no disponible para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
                continue
            df_univar_to_impute_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]].copy()
            original_nan_mask_univar_df = df_univar_to_impute_with_nans.isnull()
            df_univar_filled_for_scaling = df_univar_to_impute_with_nans.fillna(0)
            df_univar_scaled_zeros = pd.DataFrame(scaler_var_to_use.transform(df_univar_filled_for_scaling.values), columns=[var_to_impute], index=df_univar_to_impute_with_nans.index)
            df_imputed_univar_unscaled = impute_univar_with_finetuned_model(df_univar_scaled_zeros, original_nan_mask_univar_df, model_ft_to_use, device, scaler_var_to_use, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT)
            if df_imputed_univar_unscaled is not None and var_to_impute in df_imputed_univar_unscaled.columns:
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_unscaled[var_to_impute]
            else:
                print(f"    Imputación falló para {var_to_impute}. Usando ffill/bfill.")
                if var_to_impute in df_imputed_all_cols_rep_ft.columns: df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_all_cols_rep_ft[var_to_impute].ffill().bfill()
        df_to_save_final_ft = df_imputed_all_cols_rep_ft.copy()
        # Añadir TIME_FEATURES si existen en el ground truth
        time_features_to_add = [tf for tf in TIME_FEATURES if tf in df_city_ground_truth_ft_reset.columns]
        if time_features_to_add:
            df_to_save_final_ft = pd.merge(
                df_to_save_final_ft, 
                df_city_ground_truth_ft_reset[['Date'] + time_features_to_add], 
                on='Date', 
                how='left'
            )
        final_cols_for_saving_ft = ['Date', 'City'] + ALL_TARGET_COLS_FT + time_features_to_add
        df_to_save_final_ft = df_to_save_final_ft[[c for c in final_cols_for_saving_ft if c in df_to_save_final_ft.columns]]
        imputed_ft_filename = f"{city_name_safe_ft_models}_{sanitize_name(scenario_name_for_file_part)}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT_SimImpute.csv"
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        df_to_save_final_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT-SimImpute guardado: {imputed_ft_filename}")
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        df_loaded_nans_reset_ft = df_city_with_nans_loaded_ft.reset_index(drop=True)
        for col_m_mask in ALL_TARGET_COLS_FT:
            if col_m_mask in df_loaded_nans_reset_ft.columns and col_m_mask in df_city_ground_truth_ft_reset.columns:
                len_min_mask_ft = min(len(df_loaded_nans_reset_ft), len(df_city_ground_truth_ft_reset))
                mask_series_metric_ft = (df_loaded_nans_reset_ft[col_m_mask].iloc[:len_min_mask_ft].isnull() & ~df_city_ground_truth_ft_reset[col_m_mask].iloc[:len_min_mask_ft].isnull())
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask].iloc[:len_min_mask_ft] = mask_series_metric_ft
        metrics_scores_ft_rep = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_to_save_final_ft.reset_index(drop=True), nan_mask_introduced_for_metrics_rep_ft, ALL_TARGET_COLS_FT)
        for var_name_metric_ft in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep_ft, 'Seed': current_seed_loaded_ft, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name_metric_ft, 'MAE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('MAE', np.nan), 'RMSE': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('RMSE', np.nan), 'NRMAE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMAE_range', np.nan), 'NRMSE_range': metrics_scores_ft_rep.get(var_name_metric_ft, {}).get('NRMSE_range', np.nan)})
        print(f"    Métricas calculadas para MOMENT-FT-SimImpute, rep {rep_ft}")
    print(f"  Escenario de Imputación {scenario_name_for_dir} completado.")

if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT-SimImpute...")
    sufijo_test_ft = f"_{sanitize_name(TARGET_CITY_FT)}"
    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT-SimImpute guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())
    if not df_all_detailed_metrics_ft.empty:
        # Resumen por Variable
        grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
        grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
        variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}{sufijo_test_ft}.csv')
        grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False); print(f"  Resumen de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)
else: print("No se generaron métricas detalladas para MOMENT-FT-SimImpute.")
script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning (Estrategia Imputación Simulada) e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
if 'log_file' in locals() and not log_file.closed:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

KeyboardInterrupt: 

In [26]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings
import shutil
import glob
import sys # Para redirigir la salida

# Attempt to import MOMENT, handle if not found
try:
    from momentfm import MOMENTPipeline
    from momentfm.utils.masking import Masking 
except ImportError:
    print("MOMENTfm library not found. Please install it: pip install momentfm")
    MOMENTPipeline = None
    Masking = None

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing File/Path Names ---
def sanitize_name(name_str):
    import re
    s = str(name_str)
    s = re.sub(r'[^\w\s-]', '', s).strip()
    s = re.sub(r'[-\s]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s if s else "default_sanitized_name"

# --- Parámetros Configurables para Fine-Tuning ---
TARGET_CITY_FT = "Brasilia"
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = AQI_COL_FT # Solo AQI para fine-tuning
SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = SEQ_LEN_MODEL_FT // 4 # Usar solapamiento para mejor imputación
BATCH_SIZE_IMPUTE_FT = 64
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"

# Fine-tuning Parameters
FINETUNE_EPOCHS = 5000
FINETUNE_TRAIN_RATIO_NEW = 0.8
FINETUNE_STEP_DATALOADER_NEW = 128 # Generar más muestras
BATCH_SIZE_FINETUNE_NEW = 16       # Batch size más pequeño
# SOLUCIÓN: Usar un learning rate bajo y estable para fine-tuning
LEARNING_RATE_FINETUNE_NEW = 5e-5
SIMULATED_MISSING_RATIO_FOR_FT = 0.25
PATIENCE_EARLY_STOPPING_NEW = 250 # Un poco más corto
LR_PATIENCE_NEW = 80             # Un poco más corto
MIN_DELTA_EARLY_STOPPING_NEW = 1e-5

N_REPETITIONS_FT_EVAL = 1
NAN_SIMULATION_SCENARIOS_FT = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_24h_20percent'},
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = f'MOMENT_FT_final_{sanitize_name(MODEL_NAME_PRETRAINED_FT.split("/")[-1])}'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos_ft_sim_impute')
SCALERS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'scalers')
LOGS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'logs')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_FT_OUTPUT_DIR, exist_ok=True)

# --- Redirigir stdout y stderr a un archivo de log ---
log_file_path = os.path.join(LOGS_FT_OUTPUT_DIR, f'log_{MOMENT_METHOD_NAME_FT}_{sanitize_name(TARGET_CITY_FT)}_{time.strftime("%Y%m%d-%H%M%S")}.txt')
original_stdout = sys.stdout
original_stderr = sys.stderr

class Tee:
    def __init__(self, *files): self.files = files
    def write(self, obj):
        for f in self.files:
            try: f.write(obj); f.flush()
            except (IOError, ValueError): pass
    def flush(self):
        for f in self.files:
            try: f.flush()
            except (IOError, ValueError): pass

log_file = open(log_file_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.stdout, log_file)
sys.stderr = Tee(sys.stderr, log_file)

print(f"--- INICIO DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
print(f"Log guardado en: {log_file_path}")

# --- 0. Carga de Datos Originales (Ground Truth) ---
print(f"\n--- 0. Carga de Datos Originales (Ground Truth) ---")
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth_raw = pd.read_csv(file_path_load_orig)
    print(f"Datos originales cargados desde: {file_path_load_orig}")
except Exception as e:
    print(f"Advertencia: {e}. Saliendo."); exit()

df_air_quality_ground_truth_raw['Date'] = pd.to_datetime(df_air_quality_ground_truth_raw['Date'], utc=True)
if 'CO2' in df_air_quality_ground_truth_raw.columns:
    df_air_quality_ground_truth_raw = df_air_quality_ground_truth_raw.drop(columns=['CO2'])
df_air_quality_ground_truth_raw['City'] = df_air_quality_ground_truth_raw['City'].replace('New_York', 'New York')
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
city_timezones = {'Brasilia': 'America/Sao_Paulo'}
df_city_part_gt = df_air_quality_ground_truth_raw[df_air_quality_ground_truth_raw['City'] == TARGET_CITY_FT].copy()
if df_city_part_gt.empty: print(f"Error: No hay datos para {TARGET_CITY_FT}. Saliendo."); exit()
if TARGET_CITY_FT in city_timezones:
    df_city_part_gt['Date'] = df_city_part_gt['Date'].dt.tz_convert(city_timezones[TARGET_CITY_FT])
df_city_part_gt['Hour'] = df_city_part_gt['Date'].dt.hour
df_city_part_gt['DayOfWeek'] = df_city_part_gt['Date'].dt.dayofweek
df_city_part_gt['Month'] = df_city_part_gt['Date'].dt.month
df_air_quality_ground_truth_localized = df_city_part_gt
print(f"Ground truth para {TARGET_CITY_FT} procesado. Shape: {df_air_quality_ground_truth_localized.shape}")

# --- Datasets (Sin cambios, ya que el problema no estaba aquí) ---
class MomentFTSimulatedImputationDataset(Dataset):
    def __init__(self, data_array_2d_scaled_clean, seq_len, step, missing_ratio):
        self.data_array = data_array_2d_scaled_clean
        self.seq_len, self.step, self.missing_ratio = seq_len, step, missing_ratio
        if len(self.data_array) < self.seq_len: self.indices = [0]
        else: self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq_data = self.data_array[start_idx : start_idx + self.seq_len].copy()
        if len(seq_data) < self.seq_len:
            seq_data = np.pad(seq_data, ((0, self.seq_len - len(seq_data)), (0, 0)), 'constant')
        mask_indices = np.random.choice(self.seq_len, int(self.seq_len * self.missing_ratio), replace=False)
        sim_nan_mask = np.zeros_like(seq_data, dtype=bool); sim_nan_mask[mask_indices, :] = True
        input_mask = ~sim_nan_mask.all(axis=1)
        x_enc = seq_data.copy(); x_enc[sim_nan_mask] = 0.0
        return {'x_enc': torch.tensor(x_enc, dtype=torch.float32).transpose(0, 1), 'input_mask': torch.tensor(input_mask, dtype=torch.bool), 'target_clean': torch.tensor(seq_data, dtype=torch.float32).transpose(0, 1), 'simulated_nan_mask': torch.tensor(sim_nan_mask, dtype=torch.bool).transpose(0, 1)}

class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data, self.mask, self.seq_len, self.step = data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step
        if len(self.data) < self.seq_len: self.indices = [0]
        else: self.indices = list(range(0, len(self.data) - self.seq_len + 1, self.step))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq_data = self.data[start_idx : start_idx + self.seq_len]
        seq_mask = self.mask[start_idx : start_idx + self.seq_len]
        if len(seq_data) < self.seq_len:
            pad_len = self.seq_len - len(seq_data)
            seq_data = np.pad(seq_data, ((0, pad_len), (0, 0)), 'constant')
            seq_mask = np.pad(seq_mask, ((0, pad_len), (0, 0)), 'constant', constant_values=True)
        return {'timeseries': torch.tensor(seq_data, dtype=torch.float32).transpose(0, 1), 'input_mask': torch.tensor(~seq_mask, dtype=torch.bool).all(dim=1)}

# --- Funciones de Entrenamiento y Evaluación (con la lógica de pérdida corregida) ---
def evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion):
    model.eval()
    total_val_loss, batch_count = 0, 0
    with torch.no_grad():
        for batch_data in val_loader:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                if simulated_nan_mask_batch.sum() > 0:
                    loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                    if not torch.isnan(loss): total_val_loss += loss.item(); batch_count += 1
            except Exception as e: print(f"    Error en validación: {e}"); continue
    return total_val_loss / batch_count if batch_count > 0 else float('inf')

def train_moment_model_finetune_simulated_imputation(model, train_loader, val_loader, device, epochs, variable_name, model_save_path, learning_rate, lr_patience, es_patience, min_delta_es):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    # SOLUCIÓN: Usar MSELoss con reducción 'mean' y aplicarlo solo a los tensores enmascarados
    criterion = nn.MSELoss() 
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.8, patience=lr_patience, verbose=True)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    print(f"  Iniciando fine-tuning (estrategia imputación simulada) para '{variable_name}'...")
    for epoch in range(epochs):
        model.train()
        total_epoch_train_loss, train_batch_count = 0, 0
        pbar_train = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{epochs} (Train)", leave=False)
        for batch_data in pbar_train:
            x_enc_batch, input_mask_batch, target_clean_batch, simulated_nan_mask_batch = (batch_data['x_enc'].to(device), batch_data['input_mask'].to(device), batch_data['target_clean'].to(device), batch_data['simulated_nan_mask'].to(device))
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=x_enc_batch, input_mask=input_mask_batch)
                reconstruction_all = outputs.reconstruction
                if simulated_nan_mask_batch.sum() > 0:
                    # SOLUCIÓN: Calcular pérdida solo en los elementos enmascarados
                    loss = criterion(reconstruction_all[simulated_nan_mask_batch], target_clean_batch[simulated_nan_mask_batch])
                else: continue
            except Exception as e: print(f"    Error train batch: {e}"); continue
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_train_loss += loss.item(); train_batch_count += 1
            pbar_train.set_postfix(loss=loss.item())
        avg_epoch_train_loss = total_epoch_train_loss / train_batch_count if train_batch_count > 0 else float('inf')
        avg_epoch_val_loss = evaluate_model_simulated_imputation_strategy(model, val_loader, device, criterion)
        print(f"    Epoch {epoch+1}: Train Loss: {avg_epoch_train_loss:.6f}, Val Loss: {avg_epoch_val_loss:.6f} (LR: {optimizer.param_groups[0]['lr']:.2e})")
        scheduler.step(avg_epoch_val_loss)
        if avg_epoch_val_loss < best_val_loss - min_delta_es:
            best_val_loss = avg_epoch_val_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
            print(f"      Nuevo mejor modelo para '{variable_name}' guardado con Val Loss: {best_val_loss:.6f}")
        else: epochs_no_improve += 1
        if epochs_no_improve >= es_patience: print(f"      Early stopping en Epoch {epoch+1}."); break
    print(f"  Fine-tuning completado. Mejor Val Loss: {best_val_loss:.6f}")
    if os.path.exists(model_save_path):
        try: model.load_state_dict(torch.load(model_save_path, map_location=device))
        except Exception as e: print(f"  Error cargando mejor modelo: {e}")
    return model

# --- Función de Imputación (sin cambios) ---
def impute_univar_with_finetuned_model(model_ft_var, df_univar_with_nans_scaled_zeros, original_nan_mask_univar, scaler_var, seq_len_imp, step_imp, batch_size_imp, device_str):
    model_ft_var.eval()
    imputation_dataset = MomentImputationDataset(df_univar_with_nans_scaled_zeros.values, original_nan_mask_univar.values, seq_len_imp, step_imp)
    if not imputation_dataset.indices: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_imp, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_univar_with_nans_scaled_zeros.shape
    accumulated_preds_scaled, accumulated_counts = np.zeros((n_rows_total, n_features_total)), np.zeros((n_rows_total, n_features_total))
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor, input_mask_batch_tensor = batch_data['timeseries'].to(device_str), batch_data['input_mask'].to(device_str)
            try:
                outputs_obj = model_ft_var(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e: print(f"    Error impute batch {batch_idx}: {e}"); continue
            for i in range(preds_batch_scaled_np.shape[0]):
                start_idx, end_idx = imputation_dataset.indices[batch_idx * batch_size_imp + i], min(start_idx + seq_len_imp, n_rows_total)
                len_actual = end_idx - start_idx
                if len_actual > 0:
                    accumulated_preds_scaled[start_idx:end_idx, :] += preds_batch_scaled_np[i, :, :len_actual].T
                    accumulated_counts[start_idx:end_idx, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_series_scaled = accumulated_preds_scaled / accumulated_counts
    df_final = df_univar_with_nans_scaled_zeros.copy()
    df_final.iloc[:,:] = scaler_var.inverse_transform(df_final.values)
    mask = original_nan_mask_univar.iloc[:,0].values
    df_final.iloc[mask, 0] = scaler_var.inverse_transform(imputed_full_series_scaled)[mask, 0]
    return df_final

# --- Función de Métricas (sin cambios) ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_vals, pred_vals, mask = y_true_df_reset[col], y_pred_df_reset[col], nan_mask_for_metrics_df_reset[col].astype(bool)
        if mask.sum() == 0: continue
        true_at_nans, pred_at_nans = true_vals[mask], pred_vals[mask]
        mae = mean_absolute_error(true_at_nans, pred_at_nans)
        rmse = np.sqrt(mean_squared_error(true_at_nans, pred_at_nans))
        data_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
        results[col] = {'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 0 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 0 else np.nan}
    return results

# --- Script Principal ---
if MOMENTPipeline is None: print("MOMENTPipeline no importado. Saliendo."); exit()
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
df_city_ground_truth_ft_reset = df_air_quality_ground_truth_localized.reset_index(drop=True)
for col_gt_fill in ALL_TARGET_COLS_FT:
    if col_gt_fill in df_city_ground_truth_ft_reset.columns: df_city_ground_truth_ft_reset[col_gt_fill] = df_city_ground_truth_ft_reset[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull().any().any(): print("Error: GT con NaNs."); exit()

finetuned_models_dict = {}
scalers_dict = {}
city_name_safe = sanitize_name(TARGET_CITY_FT)
city_models_dir, city_scalers_dir = os.path.join(MODELS_FT_OUTPUT_DIR, city_name_safe), os.path.join(SCALERS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_models_dir, exist_ok=True); os.makedirs(city_scalers_dir, exist_ok=True)

for var_to_finetune in ALL_TARGET_COLS_FT:
    print(f"\n===== Iniciando Fine-Tuning para: {var_to_finetune} =====")
    scaler = StandardScaler()
    data_to_scale = df_city_ground_truth_ft_reset[[var_to_finetune]].values
    scaler.fit(data_to_scale)
    scalers_dict[var_to_finetune] = scaler
    joblib.dump(scaler, os.path.join(city_scalers_dir, f'scaler_{sanitize_name(var_to_finetune)}.joblib'))
    
    scaled_data = scaler.transform(data_to_scale)
    split_idx = int(len(scaled_data) * FINETUNE_TRAIN_RATIO_NEW)
    train_data, val_data = scaled_data[:split_idx], scaled_data[split_idx:]

    if len(train_data) < SEQ_LEN_MODEL_FT or len(val_data) < SEQ_LEN_MODEL_FT: print(f"Datos insuficientes para {var_to_finetune}."); continue

    train_dataset = MomentFTSimulatedImputationDataset(train_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=True, num_workers=0, drop_last=False)
    val_dataset = MomentFTSimulatedImputationDataset(val_data, SEQ_LEN_MODEL_FT, FINETUNE_STEP_DATALOADER_NEW, SIMULATED_MISSING_RATIO_FOR_FT)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_FINETUNE_NEW, shuffle=False)

    if len(train_loader) == 0: print(f"Train loader vacío para {var_to_finetune}. Saltando."); continue
        
    model = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model.init(); model = model.to(device)
    model_save_path = os.path.join(city_models_dir, f'moment_ft_best_{sanitize_name(var_to_finetune)}.pth')
    
    finetuned_model = train_moment_model_finetune_simulated_imputation(
        model, train_loader, val_loader, device, FINETUNE_EPOCHS, var_to_finetune, model_save_path,
        LEARNING_RATE_FINETUNE_NEW, LR_PATIENCE_NEW, PATIENCE_EARLY_STOPPING_NEW, MIN_DELTA_EARLY_STOPPING_NEW
    )
    finetuned_models_dict[var_to_finetune] = finetuned_model
    if finetuned_model: print(f"  Modelo para {var_to_finetune} afinado.")

# --- Imputation Phase ---
print("\n--- Iniciando Fase de Imputación ---")
city_imputed_csv_dir = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe)
os.makedirs(city_imputed_csv_dir, exist_ok=True)
for scenario in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_for_dir = sanitize_name(scenario['name'])
    scenario_name_for_file = scenario['name'].replace('%', 'percent')
    print(f"\nProcesando Escenario: {scenario_name_for_dir}")
    sim_data_dir = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe, scenario_name_for_dir)
    imputed_scenario_dir = os.path.join(city_imputed_csv_dir, scenario_name_for_dir)
    os.makedirs(imputed_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_dir): print(f"    Dir simulado no encontrado: {sim_data_dir}."); continue
    for rep in range(1, N_REPETITIONS_FT_EVAL + 1):
        sim_files = [f for f in os.listdir(sim_data_dir) if f.startswith(f"{city_name_safe}_{scenario_name_for_file}_rep{rep}") and f.endswith("_with_nans.csv")]
        if not sim_files: print(f"    Archivo no encontrado para rep {rep}."); continue
        sim_file_path = os.path.join(sim_data_dir, sim_files[0])
        seed = sim_files[0].split('_seed')[-1].split('_with_nans.csv')[0] if '_seed' in sim_files[0] else "unknown"
        df_with_nans = pd.read_csv(sim_file_path, parse_dates=['Date'])
        
        df_imputed = df_with_nans.copy()
        for var in ALL_TARGET_COLS_FT:
            model, scaler = finetuned_models_dict.get(var), scalers_dict.get(var)
            if not model or not scaler: print(f"    Modelo/scaler no disponible para {var}."); continue
            
            univar_with_nans = df_with_nans[[var]].copy()
            nan_mask = univar_with_nans.isnull()
            filled_for_scaling = univar_with_nans.fillna(0)
            scaled_zeros = pd.DataFrame(scaler.transform(filled_for_scaling), columns=[var], index=univar_with_nans.index)
            
            imputed_univar = impute_univar_with_finetuned_model(model, scaled_zeros, nan_mask, scaler, SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT, device)
            if imputed_univar is not None: df_imputed[var] = imputed_univar[var]
        
        imputed_filename = f"{city_name_safe}_{scenario_name_for_dir}_rep{rep}_seed{seed}_imputed.csv"
        imputed_save_path = os.path.join(imputed_scenario_dir, imputed_filename)
        df_imputed.to_csv(imputed_save_path, index=False)
        print(f"    Archivo imputado guardado: {imputed_filename}")
        
        nan_mask_for_metrics = df_with_nans[ALL_TARGET_COLS_FT].isnull() & ~df_city_ground_truth_ft_reset[ALL_TARGET_COLS_FT].isnull()
        metrics = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_imputed, nan_mask_for_metrics, ALL_TARGET_COLS_FT)
        for var, scores in metrics.items():
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_for_dir, 'Repetition': rep, 'Seed': seed, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var, **scores})

# --- Guardar Métricas ---
if all_detailed_metrics_ft_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas...")
    sufijo = f"_{city_name_safe}"
    df_metrics.to_csv(os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}{sufijo}.csv'), index=False)
    print(df_metrics)
else: print("No se generaron métricas.")
print(f"\nProceso completado. Tiempo total: {(time.time() - script_start_time_ft) / 60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
if 'log_file' in locals() and not log_file.closed:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

UnboundLocalError: cannot access local variable 'start_idx' where it is not associated with a value